<a href="https://colab.research.google.com/github/almo-intellect/visu-predict/blob/COLAB_NOBASELINE_V14/VISU_Traffic_Transformer_(COLAB_NOBASELINE_V14_7_NO_FOLDS).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Nota da Propriedade Intelectual

Em conformidade com os direitos de propriedade intelectual e padrões de confidencialidade, esteja ciente de que os arquivos compartilhados com você por e-mail ou qualquer outro meio estão sujeitos à proteção sob as leis de propriedade intelectual. Esses arquivos podem conter informações proprietárias, segredos comerciais ou material protegido por direitos autorais de propriedade exclusiva da Almo Intellect.

Enfatizamos que esses arquivos são para sua referência e utilizados exclusivamente no contexto das tarefas ou responsabilidades que lhe foram atribuídas. Eles não devem ser distribuídos, transmitidos ou compartilhados com quaisquer outros indivíduos ou entidades sem autorização explícita da Almo Intellect.

A sua cooperação na manutenção da confidencialidade das informações contidas nestes ficheiros é crucial para preservar os nossos direitos de propriedade intelectual e manter a confidencialidade dos dados sensíveis.

Caso tenha alguma dúvida ou necessite de maiores esclarecimentos sobre o manuseio desses arquivos, sinta-se à vontade para contactar-nos.

Obrigado pela sua compreensão e estrita adesão a estas medidas de confidencialidade.

In compliance with intellectual property rights and confidentiality standards, please be aware that files shared with you via email or any other means are subject to protection under intellectual property laws. These files may contain proprietary information, trade secrets or copyrighted material exclusively owned by Almo Intellect.

We emphasize that these files are for your reference and used solely in the context of the tasks or responsibilities assigned to you. They must not be distributed, transmitted or shared with any other individuals or entities without explicit permission from Almo Intellect.

Your cooperation in maintaining the confidentiality of the information contained in these files is crucial to preserving our intellectual property rights and maintaining the confidentiality of sensitive data.

If you have any questions or require further clarification regarding the handling of these files, please feel free to contact us.

Thank you for your understanding and strict adherence to these confidentiality measures.


# VISU Traffic Transformer Model (config.yaml disabled)

## Imports

In [ ]:
from google.colab import drive
import os

if not os.path.exists('/content/drive'):
  drive.mount('/content/drive')
else:
  print("Google Drive is already mounted at /content/drive")


# for root, dirs, files in os.walk("/content/drive/"):  # Start from "My Drive"

#     for file in files:
#         print(os.path.join(root, file))
#     for dir in dirs:
#         print(os.path.join(root, dir))

In [ ]:
"""
Traffic Prediction with Transformers
Changelog:
- Structure with clear separation of concerns
- Type hints
- Mixed precision training for improved performance
- Gradient accumulation for effective larger batch sizes
"""

## @title VISU Traffic Transformer Model #TODO: Uncomment

from IPython import get_ipython
from IPython.display import display
# %%
# Install required libraries
# !pip install torch torchvision torchaudio pandas numpy scikit-learn matplotlib seaborn holidays statsmodels optuna torch_geometric pytz #pmdarima
!pip install torch pandas numpy scikit-learn matplotlib seaborn holidays statsmodels optuna torch_geometric #torchvision torchaudio  #pmdarima

# Install Azure packages - run this cell first
#!pip install azure-ai-ml azure-identity

import os
import pandas as pd
import numpy as np
import copy
import random
from typing import Dict, List, Tuple, Optional, Union, Any, Callable
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler
from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from torch.nn import TransformerEncoder, TransformerEncoderLayer
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import yaml
import math
import torch.optim as _optim
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.gridspec import GridSpec
import pytz
from datetime import datetime
from scipy import stats
import warnings
import gc
import requests

# Optional imports with error handling
try:
    import holidays
except ImportError:
    warnings.warn("holidays package not found, holiday features will be limited")
    holidays = None

# Try to import necessary mixed precision components
try:
    # import torch.cuda.amp
    from torch.cuda.amp import autocast, GradScaler
    AMP_AVAILABLE = True
    # Check if newer API with device_type is available
    try:
        # Try creating a GradScaler with device_type
        test_scaler = GradScaler(device_type='cuda')
        del test_scaler
        DEVICE_TYPE_SUPPORTED = True
    except TypeError:
        # Older PyTorch version without device_type support
        DEVICE_TYPE_SUPPORTED = False
except ImportError:
    warnings.warn("Mixed precision training not available (torch.cuda.amp not available)")
    AMP_AVAILABLE = False
    DEVICE_TYPE_SUPPORTED = False

# Benchmarking Imports
try:
    from statsmodels.tsa.arima.model import ARIMA
    from statsmodels.tsa.holtwinters import ExponentialSmoothing
    STATSMODELS_AVAILABLE = True
except ImportError:
    warnings.warn("statsmodels not available, some benchmarks will be disabled")
    STATSMODELS_AVAILABLE = False

# Optuna for hyperparameter tuning
try:
    import optuna
    OPTUNA_AVAILABLE = True
except ImportError:
    warnings.warn("optuna not available, hyperparameter optimization will be disabled")
    OPTUNA_AVAILABLE = False

# PyTorch Geometric Imports
try:
    from torch_geometric.nn import GCNConv, GATConv
    TORCH_GEOMETRIC_AVAILABLE = True
except ImportError:
    warnings.warn("PyTorch Geometric not available, GNN functionality will be limited")
    TORCH_GEOMETRIC_AVAILABLE = False

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

<ipython-input-12-5614eb7910e3>:68: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  test_scaler = GradScaler(device_type='cuda')


## GPU Memory & Performance Utilities

In [ ]:

# =============================================================================
# GPU Memory & Performance Utilities
# =============================================================================

def get_gpu_memory_info():
    """Get GPU memory usage information"""
    if not torch.cuda.is_available():
        return {"error": "CUDA not available"}

    try:
        # Try to use nvidia-smi via subprocess if needed
        # But for simplicity, we'll use PyTorch's built-in functions
        device_count = torch.cuda.device_count()
        gpu_info = {}

        for i in range(device_count):
            total_memory = torch.cuda.get_device_properties(i).total_memory
            reserved_memory = torch.cuda.memory_reserved(i)
            allocated_memory = torch.cuda.memory_allocated(i)
            free_memory = total_memory - reserved_memory

            gpu_info[f"gpu_{i}"] = {
                "total_memory_GB": total_memory / 1e9,
                "reserved_memory_GB": reserved_memory / 1e9,
                "allocated_memory_GB": allocated_memory / 1e9,
                "free_memory_GB": free_memory / 1e9,
                "utilization_pct": (allocated_memory / total_memory) * 100
            }

        return gpu_info

    except Exception as e:
        return {"error": str(e)}

def find_optimal_batch_size(
    model: nn.Module,
    sample_input: torch.Tensor,
    sample_target: torch.Tensor,
    max_batch_size: int = 2048,
    start_batch: int = 32,
    device: str = 'cuda'
) -> int:
    """
    Find the optimal batch size for the model and GPU memory

    Args:
        model: The model to test
        sample_input: A sample input tensor
        sample_target: A sample target tensor
        max_batch_size: Maximum batch size to test
        start_batch: Starting batch size
        device: Device to test on

    Returns:
        Optimal batch size
    """
    if device == 'cpu' or not torch.cuda.is_available():
        return 64  # Default for CPU

    model = model.to(device)
    optimal_batch_size = start_batch

    # Try to clear some memory first
    torch.cuda.empty_cache()
    gc.collect()

    print("Finding optimal batch size for GPU...")
    try:
        # Get a copy of the sample tensors on the correct device
        sample_input = sample_input.to(device)
        sample_target = sample_target.to(device)

        for batch_size in [2**i for i in range(int(np.log2(start_batch)), int(np.log2(max_batch_size))+1)]:
            try:
                # Try to process a batch of this size
                input_batch = sample_input.repeat(batch_size, 1, 1)
                target_batch = sample_target.repeat(batch_size, 1, 1)

                # Forward and backward pass
                if DEVICE_TYPE_SUPPORTED:
                    with autocast(device_type='cuda'):
                        output = model(input_batch)
                        loss = nn.MSELoss()(output, target_batch)
                else:
                    with autocast():
                        output = model(input_batch)
                        loss = nn.MSELoss()(output, target_batch)

                loss.backward()

                # If we got here without an OOM error, update optimal batch size
                optimal_batch_size = batch_size

                # Clean up to prevent memory accumulation
                del input_batch, target_batch, output, loss
                torch.cuda.empty_cache()

                print(f"  Successfully tested batch size: {batch_size}")

            except RuntimeError as e:
                if "out of memory" in str(e).lower():
                    print(f"  OOM at batch size: {batch_size}")
                    break
                else:
                    raise

    except Exception as e:
        print(f"Error while finding optimal batch size: {e}")
        return 64  # Fallback to default

    finally:
        # Clean up
        torch.cuda.empty_cache()
        gc.collect()

    # Return a slightly smaller batch size to be safe
    return max(int(optimal_batch_size * 0.8), start_batch)

# Simplified GPU monitoring - no longer using contextmanager to avoid import issues
def start_gpu_memory_monitor(config, interval=10):
    """Start GPU memory monitoring - returns a flag to control monitoring"""
    if not config.monitor_gpu_usage or not torch.cuda.is_available():
        return None

    # Import time here to ensure it's available
    import time

    # Flag to control monitoring
    stop_monitoring = [False]

    # Function that will run in a separate thread
    def monitor_memory_usage():
        print("Starting GPU memory monitoring...")
        start_time = time.time()

        while not stop_monitoring[0]:
            try:
                memory_info = get_gpu_memory_info()

                # Print memory usage for each GPU
                for gpu_id, gpu_data in memory_info.items():
                    if isinstance(gpu_data, dict) and 'error' not in gpu_data:
                        print(f"{gpu_id.upper()}: "
                              f"Used {gpu_data['allocated_memory_GB']:.2f}/{gpu_data['total_memory_GB']:.2f} GB "
                              f"({gpu_data['utilization_pct']:.1f}%)")

                # Also monitor CPU memory if psutil is available
                try:
                    import psutil
                    process = psutil.Process(os.getpid())
                    print(f"CPU Memory: {process.memory_info().rss / 1e9:.2f} GB")
                except ImportError:
                    pass

                time.sleep(interval)
            except Exception as e:
                print(f"Error in GPU monitoring: {e}")
                time.sleep(interval)

    try:
        # Start monitoring in a separate thread
        import threading
        monitor_thread = threading.Thread(target=monitor_memory_usage)
        monitor_thread.daemon = True  # Daemon thread will exit when main thread exits
        monitor_thread.start()

        return stop_monitoring
    except Exception as e:
        print(f"Failed to start GPU monitoring: {e}")
        return None

def stop_gpu_memory_monitor(stop_flag):
    """Stop GPU memory monitoring by setting the stop flag"""
    if stop_flag is not None:
        stop_flag[0] = True

def get_feature_dimensions(test_dataset):
    """Extract feature dimensions from a sample of the dataset"""
    sample_x, _ = test_dataset[0]

    # Extract dimensions of each feature group
    feature_dims = {}
    for feature_name, feature_tensor in sample_x.items():
        if feature_name != 'concatenated':  # Skip the concatenated tensor
            if isinstance(feature_tensor, torch.Tensor):
                # Handle different feature tensor shapes
                if len(feature_tensor.shape) == 3:  # [seq_len, num_sensors, feature_dim]
                    feature_dims[feature_name] = feature_tensor.shape[2]
                else:  # [seq_len, feature_dim]
                    feature_dims[feature_name] = feature_tensor.shape[1]

    # Get list of feature names from the dataset
    feature_names = list(feature_dims.keys())

    # If no dimensions found, use a default
    if not feature_dims:
        print("Warning: No feature dimensions could be extracted! Using default.")
        feature_dims = {'traffic': test_dataset.feature_groups['traffic']['dim']}

    return feature_dims, feature_names

## Config Module

In [ ]:
# =============================================================================
# Config Module
# =============================================================================

@dataclass
class TrainingConfig:
    """Configuration class for training parameters with transfer learning support"""
    base_output_dir: str
    dataset_name: str = 'METR-LA'  # Options: 'METR-LA', 'PEMS-BAY'
    batch_size: int = 16
    seq_length: int = 12
    pred_length: int = 12
    num_epochs: int = 1000
    patience: int = 12
    learning_rate: float = 0.0001
    hidden_dim: int = 256
    num_layers: int = 3
    num_heads: int = 16
    dropout: float = 0.05
    ff_dim_multiplier: int = 4
    activation: str = 'gelu'
    data_scaler_type: str = 'minmax'
    optimizer_type: str = 'adamw'
    loss_function: str = 'mae' # 'mse' 'huber' 'mae'
    use_time_features: bool = True
    use_holiday_feature: bool = False
    holiday_country_code: str = 'US'
    use_weather_feature: bool = True
    weather_feature_type: str = 'all_features'
    weather_data_file: str = None  # Will be set based on dataset_name
    gradient_clip: Optional[float] = 1.0
    scheduler_type: Optional[str] = 'cosine_warmup'
    scheduler_patience: int = 12
    scheduler_factor: float = 0.6
    step_scheduler_step_size: int = 10
    step_scheduler_gamma: float = 0.1
    use_lagged_features: bool = False
    num_lags: int = 1

    # Decoder Parameters
    decoder_type: str = 'linear' #'mlp' 'linear' or 'transformer'
    num_decoder_layers: int = 3  # Number of transformer decoder layers
    dim_feedforward: int = 256 # Decoder dim
    teacher_forcing_ratio: float = 0.20  # Ratio for teacher forcing during training

    # GNN parameters
    use_spatial_features: bool = True
    spatial_feature_dim: int = 256  # Must be equal to hidden_dim
    use_gnn_pre_transformer: bool = True
    gnn_max_pooling: bool = True
    gnn_type: str = 'gcn'  # GCN or GAT
    gat_heads: int = 8
    gat_concat: bool = True
    gnn_residual: bool = True
    gnn_layers: int = 3  # Number of GNN layers

    # Spatial features parameters
    coordinates_file: Optional[str] = None
    spatial_max_pooling: bool = True
    num_sensors: int = 207
    embedding_dim: int = 207
    use_spatial_bias: bool = False
    spatial_bias_type: str = 'additive'  # 'additive' or 'multiplicative'

    # Model architecture
    max_seq_length: int = 1000000  # For positional encoding
    feature_dims: Optional[Dict[str, int]] = 256  # For feature-wise attention

    # Attention visualization
    attention_visualization: bool = True
    attention_head_analysis: bool = True
    visualize_layer_progression: bool = True

    # Output and visualization
    save_predictions: bool = True
    num_sensors_to_plot: int = 25
    generate_report: bool = True

    # Data processing
    missing_value_strategy: str = 'mean'  # 'ffill_bfill', 'zero', 'mean', 'median'
    time_format: str = '%Y-%m-%d %H:%M:%S'

    # Training optimization
    use_quantile_regression: bool = False
    quantiles: List[float] = None
    optuna_trials: Optional[int] = 0
    warmup_epochs: int = 30
    use_mixed_precision: bool = True
    accumulation_steps: int = 16
    num_workers: int = 2
    pin_memory: bool = True
    find_optimal_batch_size: bool = True
    monitor_gpu_usage: bool = True

    # Transfer Learning Support Fields
    enable_transfer_learning: bool = False
    run_only_transfer_learning: bool = False
    source_model_path: Optional[str] = None
    target_dataset_name: Optional[str] = 'METR-MPT'
    target_data_path: Optional[str] = None
    freeze_encoder: bool = True
    freeze_layers: int = 1
    adapter_dim: int = 64
    transfer_learning_rate: float = 5e-5

    # Directories (to be set after initialization)
    input_dir: Optional[str] = None
    output_dir: Optional[str] = None
    model_dir: Optional[str] = None
    results_dir: Optional[str] = None

    def __post_init__(self):
        if self.quantiles is None:
            self.quantiles = [0.1, 0.5, 0.9]

        # Initialize feature_dims if None
        if self.feature_dims is None:
            self.feature_dims = {}

        # Set dataset-specific defaults
        self._set_dataset_specific_defaults()

        # Enhanced check for divisibility with better warning messages
        if self.num_heads > 0:
            if self.hidden_dim % self.num_heads != 0:
                original_hidden_dim = self.hidden_dim
                # Adjust hidden_dim to be divisible by num_heads
                self.hidden_dim = (self.hidden_dim // self.num_heads) * self.num_heads

                # If adjustment resulted in zero, set it to num_heads
                if self.hidden_dim == 0:
                    self.hidden_dim = self.num_heads

                warnings.warn(f"Adjusted hidden_dim from {original_hidden_dim} to {self.hidden_dim} to ensure divisibility by num_heads={self.num_heads}")

            # Also ensure hidden_dim is even for positional encoding
            if self.hidden_dim % 2 != 0:
                original_hidden_dim = self.hidden_dim
                self.hidden_dim += 1  # Make it even by adding 1
                warnings.warn(f"Adjusted hidden_dim from {original_hidden_dim} to {self.hidden_dim} to ensure it's even for positional encoding")

        # Check for mixed precision availability
        if self.use_mixed_precision and not AMP_AVAILABLE:
            self.use_mixed_precision = False
            warnings.warn("Mixed precision requested but not available, disabled")

        # Check for GNN availability
        if self.use_gnn_pre_transformer and not TORCH_GEOMETRIC_AVAILABLE:
            self.use_gnn_pre_transformer = False
            warnings.warn("GNN pre-transformer requested but PyTorch Geometric not available, disabled")

        # Adjust workers based on system capabilities
        import multiprocessing
        max_workers = multiprocessing.cpu_count()
        if self.num_workers > max_workers:
            warnings.warn(f"Reducing num_workers from {self.num_workers} to {max_workers} based on system capabilities")
            self.num_workers = max_workers

        # Validate weather feature type
        valid_weather_types = ['all_features', 'temperature', 'weather_condition_code',
                               'visibility', 'wind_speed', 'wind_direction_code',
                               'wind', 'humidity', 'dew_point', 'cloud_cover_code']
        if self.weather_feature_type not in valid_weather_types:
            warnings.warn(f"Invalid weather_feature_type: {self.weather_feature_type}, using 'all_features'")
            self.weather_feature_type = 'all_features'

        # Validate spatial bias type
        valid_spatial_bias_types = ['additive', 'multiplicative']
        if self.spatial_bias_type not in valid_spatial_bias_types:
            warnings.warn(f"Invalid spatial_bias_type: {self.spatial_bias_type}, using 'additive'")
            self.spatial_bias_type = 'additive'

        # Validate missing value strategy
        valid_missing_strategies = ['ffill_bfill', 'zero', 'mean', 'median', 'interpolate']
        if self.missing_value_strategy not in valid_missing_strategies:
            warnings.warn(f"Invalid missing_value_strategy: {self.missing_value_strategy}, using 'ffill_bfill'")
            self.missing_value_strategy = 'ffill_bfill'

        # Validate transfer learning settings if enabled
        if self.enable_transfer_learning:
            if self.target_dataset_name is None:
                warnings.warn("Transfer learning enabled but target_dataset_name not set. Using 'mozambique'")
                self.target_dataset_name = 'mozambique'

            if self.target_data_path is None:
                self.target_data_path = f"{self.target_dataset_name}_traffic.csv"
                warnings.warn(f"Transfer learning enabled but target_data_path not set. Using '{self.target_data_path}'")

    def _set_dataset_specific_defaults(self):
        """Set default values based on the selected dataset"""
        if self.dataset_name == 'PEMS-BAY':
            # PEMS-BAY has 325 sensors
            if not hasattr(self, 'num_features'):
                self.num_features = 325
        else:
            # METR-LA has 207 sensors
            if not hasattr(self, 'num_features'):
                self.num_features = 207

def ensure_compatible_dimensions(model_params, config):
    """
    Ensures that hidden_dim is divisible by num_heads and is even
    for compatibility with the MultiheadAttention module

    Args:
        model_params: Dictionary of model parameters
        config: Training configuration object

    Returns:
        Updated model_params and configc
    """
    # Check if hidden_dim is divisible by num_heads
    if model_params['hidden_dim'] % model_params['num_heads'] != 0:
        # Adjust to nearest value divisible by num_heads
        adjusted_hidden_dim = (model_params['hidden_dim'] // model_params['num_heads']) * model_params['num_heads']
        # Ensure it's not zero
        if adjusted_hidden_dim == 0:
            adjusted_hidden_dim = model_params['num_heads']

        print(f"Warning: Adjusting hidden_dim from {model_params['hidden_dim']} to {adjusted_hidden_dim} to ensure divisibility by num_heads={model_params['num_heads']}")
        model_params['hidden_dim'] = adjusted_hidden_dim
        config.hidden_dim = adjusted_hidden_dim

    # Also ensure hidden_dim is even for positional encoding
    if model_params['hidden_dim'] % 2 != 0:
        adjusted_hidden_dim = model_params['hidden_dim'] + 1
        print(f"Warning: Adjusting hidden_dim from {model_params['hidden_dim']} to {adjusted_hidden_dim} to ensure it's even for positional encoding")
        model_params['hidden_dim'] = adjusted_hidden_dim
        config.hidden_dim = adjusted_hidden_dim

    return model_params, config


def load_config(config_path: str = 'config.yaml') -> TrainingConfig:
    """Load configuration from YAML file and create TrainingConfig object"""
    try:
        with open(config_path, 'r') as f:
            config_dict = yaml.safe_load(f)
        return TrainingConfig(**config_dict)
    except (FileNotFoundError, yaml.YAMLError) as e:
        warnings.warn(f"Error loading config from {config_path}: {e}. Using default configuration.")
        return TrainingConfig(base_output_dir='/content/drive/Shareddrives/Almo-2002-R&D/1 - RD-Traffic-Prediction/Transformer_Versions/COLAB_NOBASELINE_V14-6')


def setup_directories(config: TrainingConfig) -> Tuple[str, str, str, str]:
    """Sets up input, output, model, and results directories with timestamped folders."""
    timestamp = get_maputo_timestamp()

    # Use local directories instead of Google Drive paths
    output_dir = os.path.join(config.base_output_dir, f"Transformers_Output_{timestamp}")
    input_dir = '/content/drive/Shareddrives/Almo-2002-R&D/1 - RD-Traffic-Prediction/Transformer_Versions/COLAB_NOBASELINE_V14-6/Transformers_Input'
    model_dir = os.path.join(output_dir, f"Models_{timestamp}")
    results_dir = os.path.join(output_dir, f"Results_{timestamp}")

    os.makedirs(input_dir, exist_ok=True)
    os.makedirs(output_dir, exist_ok=True)
    os.makedirs(model_dir, exist_ok=True)
    os.makedirs(results_dir, exist_ok=True)

    # Update config with directory paths
    config.input_dir = input_dir
    config.output_dir = output_dir
    config.model_dir = model_dir
    config.results_dir = results_dir

    # Handle coordinates file if configured
    if config.coordinates_file is None:
        # Set default coordinates file location based on dataset
        if config.dataset_name == 'PEMS-BAY':
            config.coordinates_file = os.path.join(input_dir, 'graph_sensor_locations_pems_bay.csv')
        elif config.dataset_name == 'METR-LA':
            config.coordinates_file = os.path.join(input_dir, 'graph_sensor_locations_metr_la.csv')
        else:
            print(f"Coordinates file not found")

    # Check for weather data file and set path based on dataset
    if config.dataset_name == 'PEMS-BAY':
        weather_src = config.weather_data_file or '/content/drive/Shareddrives/Almo-2002-R&D/1 - RD-Traffic-Prediction/Transformer_Versions/COLAB_NOBASELINE_V14-6/Transformers_Input/clean_weather_data_pems_bay.csv'
    elif config.dataset_name == 'METR-LA':
        weather_src = config.weather_data_file or '/content/drive/Shareddrives/Almo-2002-R&D/1 - RD-Traffic-Prediction/Transformer_Versions/COLAB_NOBASELINE_V14-6/Transformers_Input/clean_weather_data_metr_la.csv'
    else:
        print(f"Weather file not found")

    # Update config with correct weather file
    config.weather_data_file = weather_src

    # Create feature visualization directory if needed
    if config.attention_visualization or config.attention_head_analysis or config.visualize_layer_progression:
        vis_dir = os.path.join(results_dir, 'visualizations')
        os.makedirs(vis_dir, exist_ok=True)
        print(f"Created visualization directory at {vis_dir}")

    # If transfer learning is enabled, check for target dataset
    if config.enable_transfer_learning and config.target_data_path:
        target_src = config.target_data_path
        target_dest = os.path.join(input_dir, os.path.basename(config.target_data_path))

        if os.path.exists(target_src) and not os.path.exists(target_dest):
            try:
                import shutil
                shutil.copyfile(target_src, target_dest)
                print(f"Target dataset copied to {target_dest}")
                # Update path to point to the copied file
                config.target_data_path = target_dest
            except Exception as e:
                warnings.warn(f"Could not copy target dataset file: {e}")
        elif os.path.exists(target_dest):
            # If target file already exists in input_dir, update the path
            config.target_data_path = target_dest

    # Create reports directory if report generation is enabled
    if config.generate_report:
        reports_dir = os.path.join(results_dir, 'reports')
        os.makedirs(reports_dir, exist_ok=True)
        print(f"Created reports directory at {reports_dir}")

    # Create predictions directory if saving predictions is enabled
    if config.save_predictions:
        predictions_dir = os.path.join(results_dir, 'predictions')
        os.makedirs(predictions_dir, exist_ok=True)
        print(f"Created predictions directory at {predictions_dir}")

    return input_dir, output_dir, model_dir, results_dir

def get_adjacency_matrix_path(config: TrainingConfig, input_dir: str) -> Optional[str]:
    """
    Get the appropriate adjacency matrix path based on the dataset name

    Args:
        config: Training configuration object
        input_dir: Input directory path

    Returns:
        Path to adjacency matrix file or None if not found
    """
    # Get dataset name
    dataset_name = config.dataset_name

    # Initialize potential paths
    potential_paths = []

    # Dataset-specific paths
    if dataset_name == 'PEMS-BAY':
        adj_filename = 'adj_PEMS-BAY.pkl'
        potential_paths = [
            os.path.join(input_dir, adj_filename),
            os.path.join(input_dir, 'ADJACENCY_MATRIX_PEMS_BAY', adj_filename),
            os.path.join('/content/drive/Shareddrives/Almo-2002-R&D/1 - RD-Traffic-Prediction/ADJACENCY_MATRIX_PEMS_BAY', adj_filename),
            os.path.join('/content/drive/Shareddrives/Almo-2002-R&D/1 - RD-Traffic-Prediction', 'adj_PEMS-BAY.pkl')
        ]
    else:  # Default to METR-LA
        adj_filename = 'adj_METR-LA.pkl'
        potential_paths = [
            os.path.join(input_dir, adj_filename),
            os.path.join(input_dir, 'ADJACENCY_MATRIX_METR_LA', adj_filename),
            os.path.join('/content/drive/Shareddrives/Almo-2002-R&D/1 - RD-Traffic-Prediction/ADJACENCY_MATRIX_METR_LA', adj_filename),
            os.path.join('/content/drive/Shareddrives/Almo-2002-R&D/1 - RD-Traffic-Prediction', 'adj_METR-LA.pkl')
        ]

    # Add generic fallback paths
    potential_paths.extend([
        os.path.join(input_dir, f'adj_{dataset_name}.pkl'),
        os.path.join('.', f'adj_{dataset_name}.pkl'),
        os.path.join('./data', f'adj_{dataset_name}.pkl'),
    ])

    # Check all potential paths
    for path in potential_paths:
        if os.path.exists(path):
            print(f"Found adjacency matrix at: {path}")
            return path

    # If no file found, print warning and return None
    print(f"Warning: No adjacency matrix found for dataset {dataset_name}")
    return None

def get_maputo_timestamp() -> str:
    """Returns current timestamp in Maputo timezone with dataset name prefix."""
    maputo_tz = pytz.timezone('Africa/Maputo')
    if 'config' in globals() and hasattr(config, 'dataset_name'):
        dataset_prefix = f"{config.dataset_name}_"
    else:
        dataset_prefix = ""
    return f"{dataset_prefix}{datetime.now(maputo_tz).strftime('%Y%m%d_%H%M%S')}"


def apply_transfer_config(main_config: TrainingConfig, transfer_config) -> TrainingConfig:
    """
    Apply transfer learning configuration to the main config

    Args:
        main_config: Main TrainingConfig object
        transfer_config: TransferLearningConfig object

    Returns:
        Updated TrainingConfig with transfer learning settings
    """
    # Enable transfer learning
    main_config.enable_transfer_learning = True

    # Copy basic transfer settings
    main_config.source_model_path = transfer_config.pretrained_model_path
    main_config.target_dataset_name = transfer_config.target_dataset_name
    main_config.target_data_path = transfer_config.target_data_path
    main_config.freeze_encoder = transfer_config.freeze_encoder
    main_config.freeze_layers = transfer_config.freeze_layers

    # Apply adapter settings if enabled
    if transfer_config.use_adapters:
        main_config.adapter_dim = transfer_config.adapter_dim
    else:
        main_config.adapter_dim = 0

    # Use the transfer-specific learning rate
    main_config.transfer_learning_rate = transfer_config.learning_rate

    # Adjust training parameters for fine-tuning
    main_config.num_epochs = min(main_config.num_epochs, transfer_config.num_epochs)
    main_config.patience = min(main_config.patience, transfer_config.patience)
    main_config.batch_size = min(main_config.batch_size, transfer_config.batch_size)
    main_config.gradient_clip = transfer_config.gradient_clip

    print(f"Applied transfer learning configuration for {main_config.target_dataset_name} dataset")
    return main_config

## Transfer Learning Config Module

In [ ]:
# =============================================================================
# Transfer Learning Configuration Module
# =============================================================================

class TransferLearningConfig:
    """Configuration class for transfer learning parameters"""

    def __init__(self):
        # Source model (pre-trained) settings
        self.pretrained_model_path = '/content/drive/Shareddrives/Almo-2002-R&D/1 - RD-Traffic-Prediction/Transformer_Versions/COLAB_NOBASELINE_V14/Transformers_Output_20250421_111818/Models_20250421_111818/best_model_20250421_132601.pth'  # Path to pre-trained model
        self.source_dataset_name = 'METR-LA'                  # Name of source dataset

        # Target dataset settings
        self.target_dataset_name = 'mozambique'               # 'mozambique' or 'south_africa'
        self.target_data_path = '/content/drive/Shareddrives/Almo-2002-R&D/1 - RD-Traffic-Prediction/Transformer_Versions/COLAB_NOBASELINE_V14/Transformers_Input/METR-MPT-V1.csv'      # Path to target dataset
        self.test_split = 0.2                                 # Split ratio for test set

        # Transfer learning strategy
        self.freeze_encoder = True                            # Whether to freeze encoder layers
        self.freeze_layers = 1                                # Number of transformer layers to freeze
        self.adapter_dim = 64                                 # Dimension for adapter layers (0 to disable)
        self.use_adapters = True                              # Whether to use adapter layers

        # Fine-tuning hyperparameters
        self.learning_rate = 5e-5                             # Learning rate for fine-tuning (usually smaller than initial training)
        self.num_epochs = 50                                  # Maximum number of epochs for fine-tuning
        self.patience = 5                                     # Early stopping patience
        self.batch_size = 32                                  # Batch size for fine-tuning
        self.gradient_clip = 1.0                              # Gradient clipping value

        # Scheduler settings
        self.scheduler_type = 'plateau'                       # 'plateau', 'cosine', 'step', or None
        self.scheduler_patience = 3                           # Patience for ReduceLROnPlateau
        self.scheduler_factor = 0.5                           # Factor for ReduceLROnPlateau

        # Evaluation settings
        self.evaluate_on_source = True                        # Whether to evaluate on source dataset after fine-tuning
        self.save_comparison_report = True                    # Whether to save a comparison report

        # Visualization settings
        self.visualize_attention = True                       # Whether to visualize attention patterns
        self.visualize_training_curve = True                  # Whether to visualize training curve
        self.visualize_metrics_comparison = True              # Whether to visualize metrics comparison

        # Output settings
        self.save_fine_tuned_model = True                     # Whether to save the fine-tuned model
        self.fine_tuned_model_prefix = 'fine_tuned'           # Prefix for fine-tuned model filename

    def update(self, **kwargs):
        """Update config attributes from keyword arguments"""
        for key, value in kwargs.items():
            if hasattr(self, key):
                setattr(self, key, value)
            else:
                print(f"Warning: TransferLearningConfig has no attribute '{key}'")
        return self

    def print_summary(self):
        """Print a summary of the transfer learning configuration"""
        print("\n==== Transfer Learning Configuration ====")
        print(f"Target Dataset: {self.target_dataset_name}")
        print(f"Pre-trained Model: {self.pretrained_model_path}")
        print(f"Strategy: {'Partial fine-tuning' if self.freeze_encoder else 'Full fine-tuning'}")
        if self.freeze_encoder:
            print(f"  - Freezing {self.freeze_layers} transformer layers")
        print(f"  - Adapters: {'Enabled' if self.use_adapters else 'Disabled'}")
        if self.use_adapters:
            print(f"    - Adapter dimension: {self.adapter_dim}")
        print(f"Learning Rate: {self.learning_rate}")
        print(f"Batch Size: {self.batch_size}")
        print(f"Max Epochs: {self.num_epochs}")
        print("========================================\n")

    def to_dict(self):
        """Convert configuration to dictionary"""
        return {k: v for k, v in self.__dict__.items()}


# Create transfer learning configuration instance
transfer_config = TransferLearningConfig()

# Example of how to modify settings
# transfer_config.update(
#     target_dataset_name='south_africa',
#     target_data_path='south_africa_traffic.csv',
#     freeze_layers=2,
#     learning_rate=1e-5
# )

# Print configuration summary
transfer_config.print_summary()

# To apply these settings to main config, use in your workflow:
# config = apply_transfer_config(config, transfer_config)

## Logging Module

In [ ]:
# =============================================================================
# Logging Module
# =============================================================================

import sys
import os
from datetime import datetime
from dataclasses import asdict
from enum import Enum
import pytz

# Define the timestamp function
def generate_timestamp():
    """Returns current timestamp in Maputo timezone."""
    maputo_tz = pytz.timezone('Africa/Maputo')
    return datetime.now(maputo_tz).strftime("%Y%m%d_%H%M%S")

# Create timestamp and log filename
timestamp = generate_timestamp()
log_filename = f"training_log_{timestamp}.txt"

# Specify your output directory directly
base_output_dir = '/content/drive/Shareddrives/Almo-2002-R&D/1 - RD-Traffic-Prediction/Transformer_Versions/COLAB_NOBASELINE_V14-6/'
log_path = os.path.join(base_output_dir, log_filename)

# Make sure the directory exists
os.makedirs(base_output_dir, exist_ok=True)

def save_predictions_and_actuals(
    predictions: np.ndarray,
    actuals: np.ndarray,
    results_dir: str,
    filename: str = "predictions_and_actuals",
    fold: Optional[int] = None
) -> None:
    """
    Save model predictions and actual values to CSV files for later analysis.

    Args:
        predictions: numpy array of model predictions
        actuals: numpy array of actual values
        results_dir: Directory to save results
        filename: Directory to save results
        fold: Optional fold number for cross-validation experiments
    """
    # Create timestamp for unique filenames
    timestamp = generate_timestamp()

    # Create fold suffix if fold is provided
    fold_suffix = f'_fold_{fold}' if fold is not None else ''

    # Convert arrays to DataFrames
    df_predictions = pd.DataFrame(predictions.ravel(), columns=['predictions'])
    df_actuals = pd.DataFrame(actuals.ravel(), columns=['actuals'])

    # Combine predictions and actuals into one DataFrame
    df_combined = pd.concat([df_predictions, df_actuals], axis=1)

    # Define CSV path
    csv_path = os.path.join(results_dir, f'{filename}{fold_suffix}_{timestamp}.csv')

    # Save to CSV
    df_combined.to_csv(csv_path, index=False)

    print(f"Predictions and actuals saved to {csv_path}")


def save_experiment_results(
    config: TrainingConfig,
    model_params: Dict,
    metrics: Tuple[float, float, float, float],
    results_dir: str,
    fold: Optional[int] = None
) -> None:
    """
    Save experiment configuration, model parameters and metrics to a CSV file.
    Each attribute becomes its own column for better analysis.

    Args:
        config: Training configuration object
        model_params: Dictionary of model parameters
        metrics: Tuple of (mae, rmse, r2, mape)
        results_dir: Directory to save results
        fold: Optional fold number for cross-validation experiments
    """

    # Extract metrics
    mae, rmse, r2, mape = metrics

    # Create base dictionary with metrics and basic info
    row_data = {
        'timestamp': datetime.now().strftime('%Y-%m-%d_%H-%M-%S'),
        'fold': fold if fold is not None else 'NA',
        'mae': float(mae),
        'rmse': float(rmse),
        'r2': float(r2),
        'mape': float(mape),
    }

    # Add all config attributes with 'config_' prefix
    config_dict = asdict(config)
    for key, value in config_dict.items():
        # Handle Enum values
        if isinstance(value, Enum):
            value = value.value
        row_data[f'config_{key}'] = value

    # Add all model parameters with 'model_' prefix
    for key, value in model_params.items():
        row_data[f'model_{key}'] = value

    # Convert to DataFrame
    df_row = pd.DataFrame([row_data])

    # Define CSV path
    csv_path = os.path.join(results_dir, f'experiment_results{generate_timestamp()}.csv')

    # If file exists, append; if not, create new
    if os.path.exists(csv_path):
        df_row.to_csv(csv_path, mode='a', header=False, index=False)
    else:
        df_row.to_csv(csv_path, index=False)

    print(f"Experiment results appended to {csv_path}")

# Create a logger that writes to both console and file
class TeeLogger:
    def __init__(self, filename):
        self.terminal = sys.stdout
        self.log = open(filename, 'w')

    def write(self, message):
        self.terminal.write(message)
        self.log.write(message)
        self.log.flush()  # Ensure immediate writing

    def flush(self):
        self.terminal.flush()
        self.log.flush()

# Redirect stdout to our custom logger
sys.stdout = TeeLogger(log_path)

print(f"Log file created at: {log_path}")
print(f"All console output will now be saved to this file")

## Weather Features Module



In [ ]:
"""
Weather Integration Module for Traffic Transformer

This module provides functions to load, preprocess, and integrate weather data
with traffic data for use in the Traffic Transformer model.

Usage:
    from weather_integration import load_weather_data, create_weather_features, match_weather_to_traffic

    # Load weather data
    weather_df = load_weather_data('weather_data.csv')

    # Create weather features
    weather_features = create_weather_features(weather_df)

    # Match weather data to traffic timestamps
    matched_features = match_weather_to_traffic(weather_df, traffic_timestamps)
"""

# =============================================================================
# Weather Integration Module
# =============================================================================

import os
import pandas as pd
import numpy as np
from typing import Dict, List, Tuple, Optional, Union, Any
from sklearn.preprocessing import MinMaxScaler

# Dictionary mappings for categorical weather data
WEATHER_CONDITION_MAP = {
    'Fair': 0,
    'Partly Cloudy': 1,
    'Mostly Cloudy': 2,
    'Cloudy': 3,
    'Rain': 4,
    'Snow': 5,
    'Thunderstorm': 6,
    'Fog': 7
}

CLOUD_COVER_MAP = {
    'CLR': 0,  # Clear
    'FEW': 1,  # Few clouds
    'SCT': 2,  # Scattered clouds
    'BKN': 3,  # Broken clouds
    'OVC': 4   # Overcast
}

WIND_DIRECTION_MAP = {
    'CALM': 0,
    'N': 1, 'NNE': 2, 'NE': 3, 'ENE': 4,
    'E': 5, 'ESE': 6, 'SE': 7, 'SSE': 8,
    'S': 9, 'SSW': 10, 'SW': 11, 'WSW': 12,
    'W': 13, 'WNW': 14, 'NW': 15, 'NNW': 16,
    'VAR': 17  # Variable
}

class WeatherIntegration:
    """
    Weather data integration for traffic prediction models.
    Handles loading, preprocessing, and aligning weather data with traffic timestamps.
    """

    def __init__(self, weather_file_path: str = None):
        """
        Initialize the weather integration module.

        Args:
            weather_file_path: Path to weather CSV file (optional)
        """
        self.weather_df = None
        self.feature_arrays = None

        if weather_file_path is not None:
            self.load_weather_data(weather_file_path)

    def load_weather_data(self, filepath: str) -> pd.DataFrame:
        """
        Load and preprocess weather data from CSV file

        Args:
            filepath: Path to the weather data CSV file

        Returns:
            Preprocessed weather DataFrame with datetime index
        """
        print(f"Loading weather data from {filepath}")

        # Load the data
        weather_df = pd.read_csv(filepath)

        # Convert time column to datetime and set as index
        weather_df['datetime'] = pd.to_datetime(weather_df['datetime'])
        weather_df.set_index('datetime', inplace=True)

        # Handle missing values
        weather_df = weather_df.replace('', np.nan)

        # Convert categorical weather conditions to numerical values
        weather_df['weather_condition_code'] = weather_df['weather_condition'].map(
            lambda x: WEATHER_CONDITION_MAP.get(x, 0) if pd.notnull(x) else np.nan
        )

        # Convert cloud cover to numerical
        weather_df['cloud_cover_code'] = weather_df['cloud_cover'].map(
            lambda x: CLOUD_COVER_MAP.get(x, 0) if pd.notnull(x) else np.nan
        )

        # Handle wind direction
        weather_df['wind_direction_code'] = weather_df['wind_direction'].map(
            lambda x: WIND_DIRECTION_MAP.get(x, 0) if pd.notnull(x) else np.nan
        )

        # Fill missing values with appropriate method
        weather_df = weather_df.ffill().bfill()  # Forward fill then backward fill

        # Store the dataframe
        self.weather_df = weather_df

        # Create feature arrays
        self._create_feature_arrays()

        print(f"Weather data loaded: {len(weather_df)} records from {weather_df.index.min()} to {weather_df.index.max()}")
        return weather_df

    def _create_feature_arrays(self) -> None:
        """
        Create normalized feature arrays from the weather dataframe
        """
        if self.weather_df is None:
            raise ValueError("Weather data not loaded. Call load_weather_data first.")

        weather_df = self.weather_df

        # Select relevant weather features
        selected_features = [
            'temperature',
            'weather_condition_code',
            'visibility',
            'wind_speed',
            'wind_direction_code',
            'relative_humidity',
            'dew_point',
            'cloud_cover_code'
        ]

        # Ensure all selected features exist, with fallbacks
        for feature in selected_features:
            if feature not in weather_df.columns:
                if feature == 'weather_condition_code' and 'weather_condition' in weather_df.columns:
                    # Create from text field if codes haven't been created
                    weather_df['weather_condition_code'] = weather_df['weather_condition'].map(
                        lambda x: WEATHER_CONDITION_MAP.get(x, 0) if pd.notnull(x) else 0
                    )
                elif feature == 'cloud_cover_code' and 'cloud_cover' in weather_df.columns:
                    weather_df['cloud_cover_code'] = weather_df['cloud_cover'].map(
                        lambda x: CLOUD_COVER_MAP.get(x, 0) if pd.notnull(x) else 0
                    )
                elif feature == 'wind_direction_code' and 'wind_direction' in weather_df.columns:
                    weather_df['wind_direction_code'] = weather_df['wind_direction'].map(
                        lambda x: WIND_DIRECTION_MAP.get(x, 0) if pd.notnull(x) else 0
                    )
                else:
                    # Create dummy column with zeros
                    print(f"Warning: Feature {feature} not found in weather data. Using zeros.")
                    weather_df[feature] = 0

        # Create a subset with only selected features
        weather_features_df = weather_df[selected_features].copy()

        # Ensure all columns are numeric
        for col in weather_features_df.columns:
            weather_features_df[col] = pd.to_numeric(weather_features_df[col], errors='coerce')
            # Fill any NaNs with column mean or 0
            if weather_features_df[col].isna().any():
                if weather_features_df[col].count() > 0:
                    weather_features_df[col].fillna(weather_features_df[col].mean(), inplace=True)
                else:
                    weather_features_df[col].fillna(0, inplace=True)

        # Normalize numerical features between 0 and 1
        scaler = MinMaxScaler()
        numerical_features = ['temperature', 'visibility', 'wind_speed',
                              'relative_humidity', 'dew_point']

        try:
            weather_features_df[numerical_features] = scaler.fit_transform(
                weather_features_df[numerical_features]
            )
        except Exception as e:
            print(f"Warning: Error in normalizing numerical features: {str(e)}")
            print("Trying column-by-column normalization...")

            # Try normalization column by column
            for col in numerical_features:
                try:
                    min_val = weather_features_df[col].min()
                    max_val = weather_features_df[col].max()
                    if max_val > min_val:
                        weather_features_df[col] = (weather_features_df[col] - min_val) / (max_val - min_val)
                    else:
                        weather_features_df[col] = 0  # If all values are the same
                except Exception as e2:
                    print(f"Warning: Could not normalize {col}: {str(e2)}")
                    weather_features_df[col] = 0

        # Normalize categorical features to 0-1 range
        categorical_features = ['weather_condition_code', 'wind_direction_code', 'cloud_cover_code']
        max_vals = {
            'weather_condition_code': max(WEATHER_CONDITION_MAP.values()),
            'wind_direction_code': max(WIND_DIRECTION_MAP.values()),
            'cloud_cover_code': max(CLOUD_COVER_MAP.values())
        }

        for feat in categorical_features:
            if max_vals[feat] > 0:  # Avoid division by zero
                weather_features_df[feat] = weather_features_df[feat] / max_vals[feat]

        # Create feature arrays
        self.feature_arrays = {
            'temperature': weather_features_df['temperature'].values.reshape(-1, 1),
            'weather_condition': weather_features_df['weather_condition_code'].values.reshape(-1, 1),
            'visibility': weather_features_df['visibility'].values.reshape(-1, 1),
            'wind': np.column_stack([
                weather_features_df['wind_speed'].values,
                weather_features_df['wind_direction_code'].values
            ]),
            'humidity': weather_features_df['relative_humidity'].values.reshape(-1, 1),
            'dew_point': weather_features_df['dew_point'].values.reshape(-1, 1),
            'cloud_cover': weather_features_df['cloud_cover_code'].values.reshape(-1, 1),
            'all_features': np.column_stack([
                weather_features_df['temperature'].values,
                weather_features_df['weather_condition_code'].values,
                weather_features_df['visibility'].values,
                weather_features_df['wind_speed'].values,
                weather_features_df['wind_direction_code'].values,
                weather_features_df['relative_humidity'].values,
                weather_features_df['dew_point'].values,
                weather_features_df['cloud_cover_code'].values
            ])
        }

    def match_weather_to_traffic(
        self,
        traffic_timestamps: pd.DatetimeIndex,
        feature_name: str = 'all_features'
    ) -> np.ndarray:
        """
        Match weather data to traffic timestamps using closest time approach

        Args:
            traffic_timestamps: DatetimeIndex of traffic data timestamps
            feature_name: Which weather feature to use ('all_features' or specific feature)

        Returns:
            Numpy array of weather features matching traffic timestamps
        """
        if self.weather_df is None or self.feature_arrays is None:
            raise ValueError("Weather data not loaded. Call load_weather_data first.")

        if feature_name not in self.feature_arrays:
            raise ValueError(f"Feature '{feature_name}' not found. Available features: {list(self.feature_arrays.keys())}")

        # Get the requested feature
        selected_feature = self.feature_arrays[feature_name]

        # Match each traffic timestamp to nearest weather timestamp
        matched_indices = []
        weather_timestamps = self.weather_df.index

        for traffic_time in traffic_timestamps:
            # Find the closest weather timestamp
            closest_idx = weather_timestamps.get_indexer([traffic_time], method='nearest')[0]
            matched_indices.append(closest_idx)

        # Get the weather features at the matched indices
        matched_features = selected_feature[matched_indices]

        print(f"Weather features matched to {len(traffic_timestamps)} traffic timestamps")
        return matched_features

    def get_feature_dimension(self, feature_name: str = 'all_features') -> int:
        """
        Get the dimension (number of columns) for a specific feature type

        Args:
            feature_name: Name of the feature

        Returns:
            Number of columns in the feature
        """
        if self.feature_arrays is None:
            raise ValueError("Weather data not loaded. Call load_weather_data first.")

        if feature_name not in self.feature_arrays:
            raise ValueError(f"Feature '{feature_name}' not found. Available features: {list(self.feature_arrays.keys())}")

        return self.feature_arrays[feature_name].shape[1]

# Function to create a weather feature for the TrafficDataset class
def create_weather_feature_for_dataset(
    dataset,
    weather_file_path: str,
    feature_name: str = 'all_features'
) -> np.ndarray:
    """
    Create weather feature array for a traffic dataset

    Args:
        dataset: Instance of TrafficDataset class with timestamps attribute
        weather_file_path: Path to weather CSV file
        feature_name: Name of the weather feature to use

    Returns:
        Numpy array of weather features
    """
    if not hasattr(dataset, 'timestamps') or dataset.timestamps is None:
        raise ValueError("Dataset must have timestamps attribute")

    # Create weather integration object
    weather_integration = WeatherIntegration()

    # Load weather data
    weather_integration.load_weather_data(weather_file_path)

    # Match weather to traffic timestamps
    weather_features = weather_integration.match_weather_to_traffic(
        dataset.timestamps,
        feature_name=feature_name
    )

    return weather_features

## Spatial Features Module


In [ ]:
# =============================================================================
# Spatial Features Module
# =============================================================================

import os
import pickle
import numpy as np
import torch
import torch.nn as nn
import pandas as pd
from typing import Dict, List, Tuple, Optional, Union, Any
import matplotlib.pyplot as plt
import warnings
import requests
from sklearn.preprocessing import StandardScaler
import math


def load_adjacency_matrix(adjacency_matrix_path: str, fallback_size: int = 10) -> Tuple[np.ndarray, List[str], List[int]]:
    """
    Load adjacency matrix from pickle file with fallback and handle direct NumPy array loading.

    Args:
        adjacency_matrix_path: Path to the pickle file containing the adjacency matrix
        fallback_size: Size of fallback adjacency matrix if loading fails or file not found

    Returns:
        Tuple of (adjacency_matrix, sensor_ids, node_ids)
    """
    adj_matrix, sensor_ids, node_ids = None, None, None
    try:
        # Check if file exists
        if not os.path.exists(adjacency_matrix_path):
            raise FileNotFoundError(f"Adjacency matrix file not found at: {adjacency_matrix_path}")

        # Load pickle file
        with open(adjacency_matrix_path, 'rb') as f:
            try:
                graph_data = pickle.load(f, encoding='latin1')
            except Exception:
                # Fall back to default encoding if latin1 fails
                f.seek(0)  # Reset file pointer
                graph_data = pickle.load(f)

        # Validate graph data structure
        if isinstance(graph_data, list) and len(graph_data) >= 3:
            # Expected structure: [sensor_ids, node_ids, adj_matrix]
            sensor_ids = graph_data[0]
            node_ids = graph_data[1]
            adj_matrix = graph_data[2]

            # Sanity check the adjacency matrix dimensions
            if adj_matrix.shape[0] != adj_matrix.shape[1]:
                warnings.warn(f"Adjacency matrix is not square: {adj_matrix.shape}")
            if len(sensor_ids) != adj_matrix.shape[0]:
                 warnings.warn(f"Number of sensor IDs ({len(sensor_ids)}) doesn't match adjacency matrix dimension ({adj_matrix.shape[0]})")

            print(f"Successfully loaded adjacency matrix with shape {adj_matrix.shape} and {len(sensor_ids)} sensors from list structure.")

        elif isinstance(graph_data, np.ndarray):
            # Handle case where pickle file contains only the NumPy array
            warnings.warn(f"Loaded adjacency matrix directly as NumPy array from {adjacency_matrix_path}. Generating default sensor/node IDs.")
            adj_matrix = graph_data
            num_nodes = adj_matrix.shape[0]
            if num_nodes != adj_matrix.shape[1]:
                warnings.warn(f"Adjacency matrix is not square: {adj_matrix.shape}")

            # Generate default sensor IDs and node IDs
            sensor_ids = [f"sensor_{i}" for i in range(num_nodes)]
            node_ids = list(range(num_nodes))
            print(f"Successfully loaded adjacency matrix with shape {adj_matrix.shape} from NumPy array.")

        else:
            # Handle unexpected data structure
            raise ValueError(f"Unexpected structure in adjacency matrix file: {type(graph_data)}")

        return adj_matrix, sensor_ids, node_ids

    except Exception as e:
        # Generate a fallback adjacency matrix if loading fails
        warnings.warn(f"Error loading or processing adjacency matrix from {adjacency_matrix_path}: {str(e)}. Creating default identity matrix.")

        # Create fallback identity matrix (often safer than fully connected)
        adj_matrix = np.eye(fallback_size)
        sensor_ids = [f"sensor_{i}" for i in range(fallback_size)]
        node_ids = list(range(fallback_size))

        return adj_matrix, sensor_ids, node_ids

def normalize_adj(adj: np.ndarray) -> np.ndarray:
    """
    Symmetric normalization of the adjacency matrix for GCN.
    Formula: D^(-1/2) * (A + I) * D^(-1/2) where A is adjacency matrix,
    I is identity matrix, and D is diagonal degree matrix.

    Args:
        adj: The adjacency matrix with shape (num_nodes, num_nodes)

    Returns:
        Normalized adjacency matrix
    """
    # Add self-connections (A + I)
    adj = adj + np.eye(adj.shape[0])

    # Calculate degree matrix D
    d = np.array(adj.sum(1))

    # Calculate D^(-1/2)
    d_inv_sqrt = np.power(d, -0.5).flatten()

    # Handle any division by zero or infinity
    d_inv_sqrt[np.isinf(d_inv_sqrt)] = 0.

    # Create diagonal matrix
    d_mat_inv_sqrt = np.diag(d_inv_sqrt)

    # Calculate D^(-1/2) * (A + I) * D^(-1/2)
    normalized = d_mat_inv_sqrt @ adj @ d_mat_inv_sqrt

    return normalized


def create_distance_adj_matrix(
    coordinates: np.ndarray,
    threshold: float = 0.1,
    sigma: float = 0.1
) -> np.ndarray:
    """
    Create an adjacency matrix based on spatial distances between nodes.
    Uses Gaussian kernel: exp(-d²/σ²) where d is normalized distance.

    Args:
        coordinates: Array of node coordinates with shape (num_nodes, 2)
        threshold: Distance threshold for keeping edges (0-1)
        sigma: Parameter for Gaussian kernel

    Returns:
        Distance-based adjacency matrix
    """
    num_nodes = coordinates.shape[0]

    # Initialize distance matrix
    distances = np.zeros((num_nodes, num_nodes))

    # Compute pairwise Euclidean distances
    for i in range(num_nodes):
        for j in range(num_nodes):
            if i != j:
                # Calculate Euclidean distance between nodes
                dist = np.sqrt(np.sum((coordinates[i] - coordinates[j])**2))
                distances[i, j] = dist

    # Normalize distances to 0-1 range
    if distances.max() > 0:
        distances = distances / distances.max()

    # Apply Gaussian kernel
    adjacency = np.exp(- (distances**2) / (sigma**2))

    # Apply threshold
    adjacency[adjacency < threshold] = 0

    # Remove self-loops (will be added during normalization)
    np.fill_diagonal(adjacency, 0)

    return adjacency


class SpatialIntegration:
    """
    Handles loading, processing and transforming spatial features.
    Optimized to be initialized once and reused.
    """
    _instance = None  # Class variable to store singleton instance

    @classmethod
    def get_instance(cls, *args, **kwargs):
        """Get or create singleton instance of SpatialIntegration"""
        if cls._instance is None:
            cls._instance = cls(*args, **kwargs)
        return cls._instance

    def __init__(
        self,
        adjacency_matrix_path: Optional[str] = None,
        coordinates_path: Optional[str] = None,
        num_sensors: int = 325,
        spatial_dim: int = 325,
        embedding_dim: int = 325,
        device: str = 'cpu'
    ):
        """
        Initialize spatial feature integration.

        Args:
            adjacency_matrix_path: Path to adjacency matrix pickle file
            coordinates_path: Path to sensor coordinates file (CSV)
            num_sensors: Number of sensors/nodes in the graph (will be overridden if matrix loaded)
            spatial_dim: Dimension of spatial features per node
            embedding_dim: Dimension for node embeddings
            device: Computation device ('cpu' or 'cuda')
        """
        self.num_sensors = num_sensors
        self.spatial_dim = spatial_dim
        self.embedding_dim = embedding_dim
        self.device = device
        self.adjacency_matrix = None
        self.sensor_ids = None
        self.node_ids = None

        # Load adjacency matrix if path provided
        if adjacency_matrix_path and os.path.exists(adjacency_matrix_path):
             # Use the corrected load_adjacency_matrix function
            loaded_adj, loaded_sensor_ids, loaded_node_ids = load_adjacency_matrix(
                adjacency_matrix_path, fallback_size=num_sensors
            )
            self.adjacency_matrix = loaded_adj
            self.sensor_ids = loaded_sensor_ids
            self.node_ids = loaded_node_ids
            self.num_sensors = self.adjacency_matrix.shape[0] # Update num_sensors based on loaded matrix
        else:
            # Create a default adjacency matrix if no path or file not found
            warnings.warn(f"No valid adjacency matrix provided or found at {adjacency_matrix_path}. Creating default identity matrix with {num_sensors} nodes")
            self.adjacency_matrix = np.eye(num_sensors) # Use identity matrix as a safer default
            self.sensor_ids = [f"sensor_{i}" for i in range(num_sensors)]
            self.node_ids = list(range(num_sensors))

        # Load sensor coordinates if available
        self.coordinates = self._load_coordinates(coordinates_path)

        # Create normalized adjacency matrix for GCN
        self.normalized_adjacency = normalize_adj(self.adjacency_matrix)

        # Convert to PyTorch tensors
        self.adjacency_tensor = torch.tensor(self.adjacency_matrix, dtype=torch.float32).to(device)
        self.normalized_adjacency_tensor = torch.tensor(self.normalized_adjacency, dtype=torch.float32).to(device)

        # Create learned node embeddings
        self.node_embeddings = self._create_node_embeddings().detach()

        # Cache for projections
        self.projection = None

    def _load_coordinates(self, coordinates_path: Optional[str]) -> Optional[np.ndarray]:
        """
        Load sensor coordinates from file if available.

        Args:
            coordinates_path: Path to coordinates CSV file

        Returns:
            Numpy array of coordinates or None if not available
        """
        if not coordinates_path or not os.path.exists(coordinates_path):
            warnings.warn(f"Coordinates file not found at: {coordinates_path}")
            return None

        try:
            # Load coordinates CSV (expected format: sensor_id, latitude, longitude)
            df = pd.read_csv(coordinates_path)

            # Check required columns
            required_cols = ['sensor_id', 'latitude', 'longitude']
            if not all(col in df.columns for col in required_cols):
                alt_cols = ['id', 'lat', 'lon']  # Alternative column names
                if all(col in df.columns for col in alt_cols):
                    # Rename to expected format
                    df = df.rename(columns={
                        'id': 'sensor_id',
                        'lat': 'latitude',
                        'lon': 'longitude'
                    })
                else:
                    raise ValueError(f"Coordinates file must contain columns: {required_cols} or {alt_cols}")

            # Extract coordinates in correct order based on self.sensor_ids
            coordinates = np.zeros((self.num_sensors, 2))
            df_sensor_ids = df['sensor_id'].astype(str).tolist() # Ensure sensor IDs are strings

            for i, sensor_id in enumerate(self.sensor_ids):
                sensor_id_str = str(sensor_id) # Ensure comparison is string vs string
                if sensor_id_str in df_sensor_ids:
                    sensor_data = df[df['sensor_id'].astype(str) == sensor_id_str].iloc[0]
                    coordinates[i, 0] = sensor_data['latitude']
                    coordinates[i, 1] = sensor_data['longitude']
                else:
                    # If sensor not found, use fallback coordinates
                    warnings.warn(f"Sensor ID {sensor_id_str} not found in coordinates file. Using fallback coordinates.")
                    coordinates[i, 0] = i / self.num_sensors  # Normalized position
                    coordinates[i, 1] = i / self.num_sensors

            # Normalize coordinates
            scaler = StandardScaler()
            coordinates = scaler.fit_transform(coordinates)

            print(f"Loaded coordinates for {self.num_sensors} sensors")
            return coordinates

        except Exception as e:
            warnings.warn(f"Error loading coordinates: {str(e)}")
            return None

    def _create_node_embeddings(self) -> torch.Tensor:
        """
        Create learnable node embeddings or positional spatial features.

        Returns:
            Tensor of node embeddings with shape [num_sensors, embedding_dim]
        """
        # If we have real coordinates, create embeddings based on them
        if self.coordinates is not None:
            # Use a positional encoding similar to Transformer's approach
            # but applied to 2D spatial coordinates

            coordinate_embedding = np.zeros((self.num_sensors, self.embedding_dim))

            # Use the normalized coordinates to create positional embeddings
            for i in range(self.num_sensors):
                for j in range(0, self.embedding_dim, 4):
                    if j + 3 < self.embedding_dim:
                        # Latitude encoding
                        coordinate_embedding[i, j] = np.sin(self.coordinates[i, 0] * (1.0 / np.power(10000, j / self.embedding_dim)))
                        coordinate_embedding[i, j + 1] = np.cos(self.coordinates[i, 0] * (1.0 / np.power(10000, j / self.embedding_dim)))

                        # Longitude encoding
                        coordinate_embedding[i, j + 2] = np.sin(self.coordinates[i, 1] * (1.0 / np.power(10000, j / self.embedding_dim)))
                        coordinate_embedding[i, j + 3] = np.cos(self.coordinates[i, 1] * (1.0 / np.power(10000, j / self.embedding_dim)))

            # Convert to tensor
            return torch.tensor(coordinate_embedding, dtype=torch.float32).to(self.device)

        else:
            # If no coordinates available, use learnable embeddings
            # Initialize with Xavier normal to improve convergence
            embeddings = torch.empty(self.num_sensors, self.embedding_dim).to(self.device)
            nn.init.xavier_normal_(embeddings)
            return nn.Parameter(embeddings)

    def get_adjacency_matrix(self) -> torch.Tensor:
        """Get the adjacency matrix as a PyTorch tensor."""
        return self.adjacency_tensor

    def get_normalized_adjacency_matrix(self) -> torch.Tensor:
        """Get the normalized adjacency matrix for GCN as a PyTorch tensor."""
        return self.normalized_adjacency_tensor

    def get_node_embeddings(self) -> torch.Tensor:
        """Get node embeddings tensor."""
        return self.node_embeddings

    def get_spatial_features(self, batch_size: int) -> torch.Tensor:
        """
        Get spatial features tensor ready for model input.

        Args:
            batch_size: Batch size for the features

        Returns:
            Spatial features tensor with shape [batch_size, num_sensors, spatial_dim]
        """
        # Get embeddings [num_sensors, embedding_dim]
        embeddings = self.get_node_embeddings()

        # Project to desired spatial dimension if needed
        if self.embedding_dim != self.spatial_dim and self.projection is None:
            # Create projection if needed
            self.projection = nn.Linear(self.embedding_dim, self.spatial_dim).to(self.device)

        if self.embedding_dim != self.spatial_dim:
            # Use the projection
            spatial_features = self.projection(embeddings)
        else:
            spatial_features = embeddings

        # Repeat for batch size
        # Shape becomes [batch_size, num_sensors, spatial_dim]
        batched_features = spatial_features.unsqueeze(0).repeat(batch_size, 1, 1)

        return batched_features

    @staticmethod
    def create_spatial_integration_from_config(config, device='cpu'):
        """
        Create (or retrieve) a SpatialIntegration instance from configuration.
        Uses the singleton pattern to ensure only one instance exists.
        """
        # Get dataset name (default to METR-LA if not specified)
        dataset_name = getattr(config, 'dataset_name', 'METR-LA')

        # Get adjacency matrix path from config
        adjacency_path = None
        if hasattr(config, 'input_dir'):
             # Call the dedicated function to find the path
            adjacency_path = get_adjacency_matrix_path(config, config.input_dir)

        # Get coordinates file path
        coordinates_path = None
        if hasattr(config, 'coordinates_file') and config.coordinates_file:
            if os.path.exists(config.coordinates_file):
                coordinates_path = config.coordinates_file
        elif hasattr(config, 'input_dir'):
             # Try multiple potential names for coordinates file
            potential_coord_files = [
                f'graph_sensor_locations_{dataset_name}.csv',
                'graph_sensor_locations.csv'
            ]
            for fname in potential_coord_files:
                potential_path = os.path.join(config.input_dir, fname)
                if os.path.exists(potential_path):
                    coordinates_path = potential_path
                    # Update config for future reference
                    if hasattr(config, '__setattr__'):
                        config.coordinates_file = coordinates_path
                    print(f"Found coordinates file at: {coordinates_path}")
                    break
            if coordinates_path is None:
                print(f"Warning: Could not find coordinates file in {config.input_dir}")


        # Get spatial integration parameters from config
        if hasattr(config, 'dataset_name') and config.dataset_name == 'PEMS-BAY':
            num_sensors = 325  # Default for PEMS-BAY
        else:
            num_sensors = 207  # Default for METR-LA

        # Ensure spatial_feature_dim exists in config, else use default
        spatial_dim = getattr(config, 'spatial_feature_dim', num_sensors) # Default to num_sensors if not specified

        # Get embedding_dim from config or default
        embedding_dim = getattr(config, 'embedding_dim', num_sensors) # Default to num_sensors

        # Create or retrieve singleton instance
        return SpatialIntegration.get_instance(
            adjacency_matrix_path=adjacency_path,
            coordinates_path=coordinates_path,
            num_sensors=num_sensors,
            spatial_dim=spatial_dim,
            embedding_dim=embedding_dim,
            device=device
        )

## Data Module

In [ ]:
# =============================================================================
# Data Module
# =============================================================================

class TrafficDataset(Dataset):
    """Enhanced Traffic Dataset with feature-wise separation for transformer attention"""

    def __init__(
        self,
        data: np.ndarray,
        timestamps: Optional[pd.DatetimeIndex] = None,
        sequence_length: int = 12,
        prediction_window: int = 12,
        config: Optional[TrainingConfig] = None
    ):
        """
        Traffic Dataset with configurable features including lagged, spatial, and weather.
        """
        self.data = data
        self.seq_length = sequence_length
        self.pred_window = prediction_window
        self.timestamps = timestamps
        self.config = config or TrainingConfig(base_output_dir="./output")

        self.features_list = []

        # Store how many data points we have
        self.data_length = len(data)

        # Create feature dictionaries to store feature types
        self.feature_groups = {
            'traffic': {'data': self.data, 'dim': self.data.shape[1]},
        }
        self.features_list.extend(['traffic'])

        if self.config.use_time_features and timestamps is not None:
            self.create_time_features(timestamps)
            self.feature_groups['time'] = {'data': self.time_features, 'dim': self.time_features.shape[1]}
            self.features_list.extend(['time'])

        if self.config.use_holiday_feature and timestamps is not None:
            self.create_holiday_feature(timestamps)
            self.feature_groups['holiday'] = {'data': self.holiday_feature, 'dim': self.holiday_feature.shape[1]}
            self.features_list.extend(['holiday'])

        if self.config.use_weather_feature and timestamps is not None:
            self.create_weather_feature(timestamps) # Attempt to create the feature
            # Check if the feature was successfully created *before* trying to use it
            if hasattr(self, 'weather_feature') and self.weather_feature is not None:
                 # Only add to feature_groups if creation succeeded
                 self.feature_groups['weather'] = {'data': self.weather_feature, 'dim': self.weather_feature.shape[1]}
                 self.features_list.extend(['weather'])
                 print(f"Successfully added weather features with shape: {self.weather_feature.shape}") # Optional: Add success log
            else:
                 # This warning now correctly reflects the situation if creation failed
                 print("Warning: Weather feature requested but failed to create or load. Skipping weather features.")

        if self.config.use_lagged_features:
            self.create_lagged_features()
            self.feature_groups['lagged'] = {'data': self.lagged_features, 'dim': self.lagged_features.shape[1]}
            self.features_list.extend(['lagged'])

        if self.config.use_spatial_features:
            self.create_spatial_features()
            # Special handling for spatial features which are now 3D
            if hasattr(self, 'spatial_features') and len(self.spatial_features.shape) == 3:
                # Store the raw 3D features
                self.feature_groups['spatial'] = {
                    'data': self.spatial_features,
                    'dim': self.spatial_features.shape[2]  # Use the spatial_dim as dim
                }
            else:
                # Fallback for original 2D implementation
                self.feature_groups['spatial'] = {
                    'data': self.spatial_features,
                    'dim': self.spatial_features.shape[1]
                }
            self.features_list.extend(['spatial'])

        # Store feature dimensions for easier access
        self.feature_dims = {name: group['dim'] for name, group in self.feature_groups.items()}

        # For compatibility, also create concatenated version - this handles both 2D and 3D features
        self._prepare_concatenated_features()

        # Calculate total feature dimension for backward compatibility
        self.total_feature_dim = self.concatenated_features.shape[1]

        # Log initialization information
        print(f"TrafficDataset initialized with {len(self.features_list)} feature groups: {self.features_list}")
        print(f"Feature dimensions: {self.feature_dims}")
        print(f"Total concatenated dimension: {self.total_feature_dim}")

    def create_spatial_features(self) -> None:
        """
        Creates spatial features based on graph node embeddings with node-specific information preserved.
        Uses SpatialIntegration to get proper node embeddings representing spatial relationships.
        """
        # Create SpatialIntegration instance if not already created
        if not hasattr(self, '_spatial_integration'):
            self._spatial_integration = SpatialIntegration.create_spatial_integration_from_config(
                config=self.config,
                device='cpu'  # Initialize on CPU, will be moved to proper device during training
            )

        # Get node embeddings from the SpatialIntegration
        node_embeddings = self._spatial_integration.get_node_embeddings().cpu().numpy()

        # Get dimensions
        num_samples = len(self.data)
        num_sensors = self.data.shape[1]
        spatial_dim = self.config.spatial_feature_dim

        # Initialize spatial features array with proper dimensions
        # Shape: [num_samples, num_sensors, spatial_dim]
        self.spatial_features = np.zeros((num_samples, num_sensors, spatial_dim))

        # Get embedding dimension
        embedding_dim = node_embeddings.shape[1]

        # Project node embeddings to spatial_dim if needed
        if embedding_dim != spatial_dim:
            # Create a simple projection (could be a more complex learned transformation)
            if embedding_dim < spatial_dim:
                # Pad with zeros
                projected_embeddings = np.zeros((node_embeddings.shape[0], spatial_dim))
                projected_embeddings[:, :embedding_dim] = node_embeddings
            else:
                # Truncate
                projected_embeddings = node_embeddings[:, :spatial_dim]
        else:
            projected_embeddings = node_embeddings

        # Apply node-specific spatial features for each time step
        # This preserves the identity of each node across all samples
        for i in range(num_samples):
            for j in range(min(num_sensors, len(projected_embeddings))):
                # Assign the specific node embedding for this sensor
                self.spatial_features[i, j] = projected_embeddings[j]

        # Log success with dimension information
        print(f"Created spatial features with shape {self.spatial_features.shape}")
        print(f"Original node embeddings dimension: {embedding_dim}, Target spatial dimension: {spatial_dim}")
        print(f"Node-specific spatial information preserved across {num_sensors} sensors")

    def _prepare_concatenated_features(self):
        """Create concatenated features for backward compatibility, with special handling for 3D spatial features"""
        data_length = len(self.data)

        # Calculate the total feature dimension, handling spatial features specially
        total_feature_dim = 0
        for name, info in self.feature_groups.items():
            feature_data = info['data']
            feature_dim = info['dim']

            # Special handling for spatial features which are now 3D
            if name == 'spatial' and len(feature_data.shape) == 3:  # [num_samples, num_sensors, spatial_dim]
                # For concatenation, we'll use a flattened representation of the first node's features
                # or another aggregation method that produces a 2D result
                feature_dim = feature_data.shape[2]  # Use spatial_dim as feature_dim

            total_feature_dim += feature_dim

        # Initialize concatenated features array
        self.concatenated_features = np.zeros((data_length, total_feature_dim))

        # Populate concatenated features
        start_idx = 0
        for name, info in self.feature_groups.items():
            feature_data = info['data']
            feature_dim = info['dim']

            # Ensure feature data has correct length
            if len(feature_data) != data_length:
                print(f"Warning: {name} feature has length {len(feature_data)}, expected {data_length}")
                if len(feature_data) > data_length:
                    feature_data = feature_data[:data_length]
                else:
                    # Create padding with correct dimensions
                    if name == 'spatial' and len(feature_data.shape) == 3:
                        # For 3D spatial features
                        padding_shape = (data_length - len(feature_data), feature_data.shape[1], feature_data.shape[2])
                        padding = np.zeros(padding_shape)
                    else:
                        # For standard 2D features
                        padding = np.zeros((data_length - len(feature_data), feature_dim))

                    feature_data = np.concatenate([feature_data, padding], axis=0)

                # Update the feature data in the feature_groups
                self.feature_groups[name]['data'] = feature_data

            # Special handling for spatial features
            if name == 'spatial' and len(feature_data.shape) == 3:
                # Option 1: Use the mean across sensors for each spatial dimension
                mean_spatial = np.mean(feature_data, axis=1)  # [num_samples, spatial_dim]
                self.concatenated_features[:, start_idx:start_idx+feature_dim] = mean_spatial
            else:
                # Standard assignment for 2D features
                self.concatenated_features[:, start_idx:start_idx+feature_dim] = feature_data

            start_idx += feature_dim

    def __len__(self) -> int:
        return len(self.data) - self.seq_length - self.pred_window + 1

    def __getitem__(self, idx: int) -> Tuple[Dict[str, torch.Tensor], torch.Tensor]:
        """
        Get a sample with features grouped by type for feature-wise attention.
        Special handling for spatial features to maintain node-specific information.

        Returns:
            Tuple of (features dict, target tensor)
            - features dict has keys like 'traffic', 'weather', etc. with tensor values
            - target is the standard prediction target
        """
        # Create dictionary of feature group tensors
        x_grouped = {}

        for feature_name, feature_info in self.feature_groups.items():
            feature_data = feature_info['data']

            # Special handling for spatial features
            if feature_name == 'spatial' and hasattr(self, 'spatial_features'):
                # Spatial features should preserve node information
                # Extract the spatial features for this window
                x_feature = self.spatial_features[idx:idx+self.seq_length]

                # For spatial features, we want shape [seq_len, num_sensors, spatial_dim]
                x_grouped[feature_name] = torch.FloatTensor(x_feature)
            else:
                # Standard handling for other feature types
                x_feature = feature_data[idx:idx+self.seq_length]
                x_grouped[feature_name] = torch.FloatTensor(x_feature)

        # Target remains the same
        y = self.data[idx+self.seq_length:idx+self.seq_length+self.pred_window]

        # For backward compatibility, also provide concatenated version
        x_grouped['concatenated'] = torch.FloatTensor(
            self.concatenated_features[idx:idx+self.seq_length]
        )

        return x_grouped, torch.FloatTensor(y)

    def create_time_features(self, timestamps: pd.DatetimeIndex) -> None:
        """Create normalized time-based features with explicit format handling"""
        try:
            # First try with the specified format from config if available
            if hasattr(self.config, 'time_format') and self.config.time_format:
                times = pd.to_datetime(timestamps, format=self.config.time_format)
            else:
                # Try common formats
                for fmt in ['%Y-%m-%d %H:%M:%S', '%Y-%m-%d', '%m/%d/%Y %H:%M']:
                    try:
                        times = pd.to_datetime(timestamps, format=fmt)
                        break
                    except ValueError:
                        continue
                else:
                    # If all formats fail, fall back to automatic parsing
                    times = pd.to_datetime(timestamps)
        except Exception as e:
            # Last resort: use pandas default parser
            print(f"Warning: Error parsing timestamps: {e}. Using pandas default parser.")
            times = pd.to_datetime(timestamps)

        # Create the features as before
        self.time_features = np.stack([
            times.hour.values / 23.0,
            times.dayofweek.values / 6.0,
            (times.dayofyear // 7).values / 51.0,
            times.month.values / 11.0
        ], axis=1)

    def create_holiday_feature(self, timestamps: pd.DatetimeIndex) -> None:
        """Create holiday indicator feature"""
        if holidays is None:
            # Create dummy holiday feature if holidays package is not available
            self.holiday_feature = np.zeros((len(timestamps), 1))
            warnings.warn("holidays package not available, using dummy holiday feature")
            return

        dates = pd.to_datetime(timestamps).date
        country_code = self.config.holiday_country_code
        try:
            country_holidays = holidays.CountryHoliday(country_code, years=set(d.year for d in dates))
        except KeyError:
            warnings.warn(f"Country code '{country_code}' not recognized. Using US holidays instead.")
            country_holidays = holidays.CountryHoliday('US', years=set(d.year for d in dates))

        self.holiday_feature = np.array([(1 if d in country_holidays else 0) for d in dates], dtype=float).reshape(-1, 1)

    def create_weather_feature(self, timestamps: pd.DatetimeIndex) -> None:
        """Create weather feature using actual weather data"""
        try:
            # Path to weather data file
            weather_file = os.path.join(self.config.input_dir, os.path.basename(self.config.weather_data_file))
            print(f"Attempting to load weather data from: {weather_file}")

            # Check if file exists
            if not os.path.exists(weather_file):
                print(f"Weather file not found at: {weather_file}")
                # Try direct path
                if os.path.exists(self.config.weather_data_file):
                    weather_file = self.config.weather_data_file
                    print(f"Using direct path instead: {weather_file}")
                else:
                    raise FileNotFoundError(f"Weather file not found at either {weather_file} or {self.config.weather_data_file}")

            # Create weather integration object
            weather_integration = WeatherIntegration()

            # Load weather data
            weather_integration.load_weather_data(weather_file)

            # Match weather to traffic timestamps
            weather_features = weather_integration.match_weather_to_traffic(timestamps)
            print(f"Raw weather features shape: {weather_features.shape}")

            # Ensure correct reshaping if needed
            self.weather_feature = weather_features
            print(f"Final weather feature shape: {self.weather_feature.shape}")

        except Exception as e:
            # Fallback to simulated weather with matching dimensions
            print(f"Error loading weather data: {str(e)}.")

    def _load_weather_data(self, filepath: str) -> pd.DataFrame:
        """
        Load and preprocess weather data from CSV file

        Args:
            filepath: Path to the weather data CSV file

        Returns:
            Preprocessed weather DataFrame with datetime index
        """
        # Load the data
        weather_df = pd.read_csv(filepath)

        # Convert time column to datetime and set as index
        weather_df['datetime'] = pd.to_datetime(weather_df['datetime'])
        weather_df.set_index('datetime', inplace=True)

        # Handle missing values
        weather_df = weather_df.replace('', np.nan)

        # Convert categorical weather conditions to numerical values
        weather_df['weather_condition_code'] = weather_df['weather_condition'].map(
            lambda x: WEATHER_CONDITION_MAP.get(x, 0) if pd.notnull(x) else np.nan
        )

        # Convert cloud cover to numerical
        weather_df['cloud_cover_code'] = weather_df['cloud_cover'].map(
            lambda x: CLOUD_COVER_MAP.get(x, 0) if pd.notnull(x) else np.nan
        )

        # Handle wind direction
        weather_df['wind_direction_code'] = weather_df['wind_direction'].map(
            lambda x: WIND_DIRECTION_MAP.get(x, 0) if pd.notnull(x) else np.nan
        )

        # Fill missing values with appropriate method
        weather_df = weather_df.ffill().bfill()  # Forward fill then backward fill

        return weather_df

    def _create_weather_features(self, weather_df: pd.DataFrame) -> np.ndarray:
        """
        Create normalized weather features for the model

        Args:
            weather_df: Preprocessed weather DataFrame

        Returns:
            Array of normalized weather features
        """
        # Select relevant weather features
        selected_features = [
            'temperature',
            'weather_condition_code',
            'visibility',
            'wind_speed',
            'wind_direction_code',
            'relative_humidity',
            'dew_point',
            'cloud_cover_code'
        ]

        # Subset the dataframe to only include selected features
        weather_features_df = weather_df[selected_features].copy()

        # Normalize numerical features between 0 and 1
        scaler = MinMaxScaler()
        numerical_features = ['temperature', 'visibility', 'wind_speed',
                              'relative_humidity', 'dew_point']

        # Check if there are any non-numeric values
        for col in numerical_features:
            weather_features_df[col] = pd.to_numeric(weather_features_df[col], errors='coerce')
            # Fill any NaNs with column mean or 0
            if weather_features_df[col].isna().any():
                if weather_features_df[col].count() > 0:
                    weather_features_df[col].fillna(weather_features_df[col].mean(), inplace=True)
                else:
                    weather_features_df[col].fillna(0, inplace=True)

        weather_features_df[numerical_features] = scaler.fit_transform(
            weather_features_df[numerical_features]
        )

        # Categorical features are already normalized between 0 and N
        # Further normalize to 0-1 range for consistency
        categorical_features = ['weather_condition_code', 'wind_direction_code', 'cloud_cover_code']
        max_vals = {
            'weather_condition_code': 7,  # Based on weather_condition_map
            'wind_direction_code': 17,    # Based on wind_dir_map
            'cloud_cover_code': 4         # Based on cloud_cover_map
        }

        for feat in categorical_features:
            weather_features_df[feat] = weather_features_df[feat] / max_vals[feat]

        # Create a combined feature array
        all_features = np.column_stack([
            weather_features_df['temperature'].values,
            weather_features_df['weather_condition_code'].values,
            weather_features_df['visibility'].values,
            weather_features_df['wind_speed'].values,
            weather_features_df['wind_direction_code'].values,
            weather_features_df['relative_humidity'].values,
            weather_features_df['dew_point'].values,
            weather_features_df['cloud_cover_code'].values
        ])

        return all_features

    def _match_weather_to_traffic(
        self,
        weather_df: pd.DataFrame,
        traffic_timestamps: pd.DatetimeIndex
    ) -> np.ndarray:
        """
        Match weather data to traffic timestamps using closest time approach

        Args:
            weather_df: Weather DataFrame with datetime index
            traffic_timestamps: DatetimeIndex of traffic data timestamps

        Returns:
            Numpy array of weather features matching traffic timestamps
        """
        # Create features from weather data
        weather_features = self._create_weather_features(weather_df)

        # Match each traffic timestamp to nearest weather timestamp
        matched_indices = []
        weather_timestamps = weather_df.index

        for traffic_time in traffic_timestamps:
            # Find the closest weather timestamp
            closest_idx = weather_timestamps.get_indexer([traffic_time], method='nearest')[0]
            matched_indices.append(closest_idx)

        # Get the weather features at the matched indices
        matched_features = weather_features[matched_indices]

        # Reshape to have appropriate dimensions for the model
        return matched_features.reshape(-1, matched_features.shape[1])

    def create_lagged_features(self) -> None:
        """Creates lagged features from the sensor data itself."""
        num_lags = self.config.num_lags
        lagged_features = []

        for i in range(1, num_lags + 1):
            lagged_data = np.roll(self.data, shift=i, axis=0)
            lagged_data[:i] = np.nan  # Fill first 'i' rows with NaN
            lagged_features.append(lagged_data)

        self.lagged_features = np.concatenate(lagged_features, axis=1)
        # Handle NaN values with zero filling
        self.lagged_features = np.nan_to_num(self.lagged_features, nan=0.0)

    def get_adjacency_matrix(self) -> torch.Tensor:
        """
        Returns:
            Normalized adjacency matrix as a PyTorch tensor
        Raises:
            RuntimeError: If no valid adjacency matrix can be loaded
        """
        # Check if we've already created and cached the adjacency matrix
        if hasattr(self, '_cached_adjacency_matrix') and self._cached_adjacency_matrix is not None:
            return self._cached_adjacency_matrix

        # Get appropriate adjacency matrix path based on dataset name
        adj_path = get_adjacency_matrix_path(self.config, self.config.input_dir)

        if adj_path and os.path.exists(adj_path):
            try:
                # Use the original load and normalize functions
                print(f"Loading adjacency matrix from {adj_path}")
                with open(adj_path, 'rb') as f:
                    try:
                        graph_data = pickle.load(f, encoding='latin1')
                    except:
                        f.seek(0)  # Reset file pointer
                        graph_data = pickle.load(f)

                if isinstance(graph_data, list) and len(graph_data) >= 3:
                    adj_matrix = graph_data[2]
                    adj_norm = normalize_adj(adj_matrix)
                    adj_norm_tensor = torch.tensor(adj_norm, dtype=torch.float32).to(self.device if hasattr(self, 'device') else 'cpu')

                    # Cache for future use
                    self._cached_adjacency_matrix = adj_norm_tensor

                    print(f"Successfully loaded adjacency matrix with shape {adj_norm_tensor.shape}")
                    print(f"Matrix shape matches dataset: {adj_norm_tensor.shape[0] == self.data.shape[1]}")

                    return adj_norm_tensor
            except Exception as e:
                print(f"Failed to load adjacency matrix from {adj_path}: {e}")

        # If we get here, we couldn't load an adjacency matrix
        error_msg = (
            f"Could not load adjacency matrix for {self.config.dataset_name} from any of the standard locations. "
            "GNN pre-transformer requires a valid adjacency matrix. "
            f"Please place adj_{self.config.dataset_name}.pkl in the input directory or current working directory."
        )
        print(error_msg)

        raise RuntimeError(error_msg)

def prepare_data(
    df: pd.DataFrame,
    config: TrainingConfig
) -> Tuple[np.ndarray, pd.DatetimeIndex, Any, int]:
    """
    Prepare and preprocess data for training with enhanced missing value handling

    Args:
        df: Input dataframe with timestamp index
        config: Training configuration

    Returns:
        Tuple of:
        - Normalized data
        - Timestamps
        - Scaler object
        - Number of features
    """
    # Handle null values based on config strategy
    if config.missing_value_strategy == 'ffill_bfill':
        df.replace(0.0, np.nan, inplace=True)
        df.ffill(inplace=True)
        df.bfill(inplace=True)
    elif config.missing_value_strategy == 'zero':
        df.replace(np.nan, 0.0, inplace=True)
    elif config.missing_value_strategy == 'mean':
        df.replace(0.0, np.nan, inplace=True)
        df.fillna(df.mean(), inplace=True)
    elif config.missing_value_strategy == 'median':
        df.replace(0.0, np.nan, inplace=True)
        df.fillna(df.median(), inplace=True)
    elif config.missing_value_strategy == 'interpolate':
        df.replace(0.0, np.nan, inplace=True)
        df.interpolate(method='time', inplace=True)
        # Fill remaining NaNs at the beginning/end
        df.ffill(inplace=True)
        df.bfill(inplace=True)

    timestamps = df.index
    sensor_data = df.values
    num_features = sensor_data.shape[1]

    # Data Scaling
    if config.data_scaler_type == 'minmax':
        data_scaler = MinMaxScaler()
    elif config.data_scaler_type == 'standard':
        data_scaler = StandardScaler()
    elif config.data_scaler_type == 'robust':
        data_scaler = RobustScaler()
    else:
        warnings.warn(f"Invalid scaler type: {config.data_scaler_type}, using MinMaxScaler")
        data_scaler = MinMaxScaler()

    data_normalized = data_scaler.fit_transform(sensor_data)

    return data_normalized, timestamps, data_scaler, num_features

## Model Module

In [ ]:

# =============================================================================
# Model Module
# =============================================================================

class CustomTransformerEncoderLayer(nn.TransformerEncoderLayer):
    """Modified Transformer Encoder Layer to capture attention weights."""
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.attn_weights = None

    def _sa_block(self, x, attn_mask, key_padding_mask, is_causal=False):
        x, weights = self.self_attn(
            x, x, x,
            attn_mask=attn_mask,
            key_padding_mask=key_padding_mask,
            need_weights=True,
            is_causal=is_causal
        )
        self.attn_weights = weights.detach()
        return self.dropout1(x)

class SpatialBiasTransformerEncoderLayer(CustomTransformerEncoderLayer):
    """Modified Transformer Encoder Layer that incorporates spatial relationships as attention bias."""
    def __init__(
        self,
        d_model,
        nhead,
        dim_feedforward=256,
        dropout=0.1,
        activation="gelu",
        layer_norm_eps=1e-5,
        batch_first=False,
        use_spatial_bias=True,
        spatial_bias_type='additive',
        seq_length=12  # Use seq_length directly instead of config
    ):
        super().__init__(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation=activation,
            layer_norm_eps=layer_norm_eps,
            batch_first=batch_first
        )
        self.use_spatial_bias = use_spatial_bias
        self.spatial_bias_type = spatial_bias_type
        self.spatial_bias = None
        self.hidden_dim = d_model
        self.num_heads = nhead
        self.seq_length = seq_length  # Store sequence length directly
        self.spatial_bias_strength = 1.0  # Default value if no config

    def set_spatial_bias(self, bias):
        """Set spatial bias tensor for attention calculation"""
        self.spatial_bias = bias

    def _sa_block(self, x, attn_mask, key_padding_mask, is_causal=False):
        """Override self-attention block to incorporate spatial bias with proper reshaping for multi-head attention"""
        # Apply spatial bias if available
        if self.use_spatial_bias and self.spatial_bias is not None:
            # Verify shape compatibility with input
            if self.spatial_bias.size(1) == x.size(1) and self.spatial_bias.size(2) == x.size(1):
                # Get batch size and reshape bias for multi-head attention
                batch_size = self.spatial_bias.size(0)

                # Reshape spatial bias to [batch_size * num_heads, seq_len, seq_len]
                # PyTorch expects each head to have its own mask
                expanded_bias = self.spatial_bias.repeat_interleave(self.num_heads, dim=0)

                # Create attention mask if None
                if attn_mask is None:
                    attn_mask = expanded_bias
                else:
                    # Handle existing attention mask - must also expand it to match
                    if self.spatial_bias_type == 'additive':
                        attn_mask = attn_mask + expanded_bias
                    elif self.spatial_bias_type == 'multiplicative':
                        attn_mask = attn_mask * expanded_bias
            else:
                # Log warning about shape mismatch
                print(f"Warning: Spatial bias shape {self.spatial_bias.shape} doesn't match input shape {x.shape}. Skipping bias.")

        # Call original self-attention
        x, weights = self.self_attn(
            x, x, x,
            attn_mask=attn_mask,
            key_padding_mask=key_padding_mask,
            need_weights=True,
            is_causal=is_causal
        )
        self.attn_weights = weights.detach()
        return self.dropout1(x)

class TransformerDecoderLayer(nn.Module):
    """
    Transformer decoder layer with self-attention, cross-attention to encoder outputs,
    and feed-forward network.
    """
    def __init__(
        self,
        d_model: int,
        nhead: int,
        dim_feedforward: int = 256,
        dropout: float = 0.1,
        activation: str = "gelu",
        batch_first: bool = True,
        device=None,
        dtype=None
    ):
        super(TransformerDecoderLayer, self).__init__()

        # Self-attention mechanism
        self.self_attn = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=nhead,
            dropout=dropout,
            batch_first=batch_first,
            device=device,
            dtype=dtype
        )

        # Cross-attention to encoder outputs
        self.cross_attn = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=nhead,
            dropout=dropout,
            batch_first=batch_first,
            device=device,
            dtype=dtype
        )

        # Feed-forward network
        self.linear1 = nn.Linear(d_model, dim_feedforward, device=device, dtype=dtype)
        self.dropout = nn.Dropout(dropout)
        self.linear2 = nn.Linear(dim_feedforward, d_model, device=device, dtype=dtype)

        # Layer normalization
        self.norm1 = nn.LayerNorm(d_model, device=device, dtype=dtype)
        self.norm2 = nn.LayerNorm(d_model, device=device, dtype=dtype)
        self.norm3 = nn.LayerNorm(d_model, device=device, dtype=dtype)

        # Dropout
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)

        # Activation function
        if activation == "gelu":
            self.activation = nn.GELU()
        elif activation == "relu":
            self.activation = nn.ReLU()
        else:
            raise ValueError(f"Unsupported activation: {activation}")

        # To store attention weights for visualization
        self.self_attn_weights = None
        self.cross_attn_weights = None

    def forward(
        self,
        tgt: torch.Tensor,
        memory: torch.Tensor,
        tgt_mask: Optional[torch.Tensor] = None,
        memory_mask: Optional[torch.Tensor] = None,
        tgt_key_padding_mask: Optional[torch.Tensor] = None,
        memory_key_padding_mask: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        """
        Forward pass for decoder layer

        Args:
            tgt: Target sequence (decoder input)
            memory: Encoder output
            tgt_mask: Mask for target sequence (typically used for causal attention)
            memory_mask: Mask for memory sequence
            tgt_key_padding_mask: Padding mask for target sequence
            memory_key_padding_mask: Padding mask for memory sequence

        Returns:
            Processed tensor after self-attention, cross-attention and feed-forward
        """
        # Self-attention block
        tgt2, self_attn_weights = self.self_attn(
            query=tgt,
            key=tgt,
            value=tgt,
            attn_mask=tgt_mask,
            key_padding_mask=tgt_key_padding_mask,
            need_weights=True
        )
        # Store self-attention weights
        self.self_attn_weights = self_attn_weights

        # Add & Norm (first residual connection)
        tgt = tgt + self.dropout1(tgt2)
        tgt = self.norm1(tgt)

        # Cross-attention block (attending to encoder outputs)
        tgt2, cross_attn_weights = self.cross_attn(
            query=tgt,
            key=memory,
            value=memory,
            attn_mask=memory_mask,
            key_padding_mask=memory_key_padding_mask,
            need_weights=True
        )
        # Store cross-attention weights
        self.cross_attn_weights = cross_attn_weights

        # Add & Norm (second residual connection)
        tgt = tgt + self.dropout2(tgt2)
        tgt = self.norm2(tgt)

        # Feed-forward block
        tgt2 = self.linear2(self.dropout(self.activation(self.linear1(tgt))))

        # Add & Norm (third residual connection)
        tgt = tgt + self.dropout3(tgt2)
        tgt = self.norm3(tgt)

        return tgt

class TransformerDecoder(nn.Module):
    """
    Full transformer decoder consisting of multiple decoder layers
    """
    def __init__(
        self,
        decoder_layer,
        num_layers: int,
        norm=None,
        return_intermediate: bool = False
    ):
        super(TransformerDecoder, self).__init__()

        # Create a ModuleList of decoder layers
        self.layers = nn.ModuleList([
            # Deep copy each layer to avoid shared parameters
            copy.deepcopy(decoder_layer) for _ in range(num_layers)
        ])

        # Normalization layer after the entire stack
        self.norm = norm
        self.num_layers = num_layers

        # Whether to return outputs from all decoder layers
        self.return_intermediate = return_intermediate

        # Store attention weights
        self.self_attn_weights = []
        self.cross_attn_weights = []

    def forward(
        self,
        tgt: torch.Tensor,
        memory: torch.Tensor,
        tgt_mask: Optional[torch.Tensor] = None,
        memory_mask: Optional[torch.Tensor] = None,
        tgt_key_padding_mask: Optional[torch.Tensor] = None,
        memory_key_padding_mask: Optional[torch.Tensor] = None,
    ) -> Union[torch.Tensor, Tuple[torch.Tensor, List[torch.Tensor]]]:
        """
        Forward pass through all decoder layers

        Args:
            tgt: Target sequence (decoder input)
            memory: Encoder output
            tgt_mask: Mask for target sequence (typically used for causal attention)
            memory_mask: Mask for memory sequence
            tgt_key_padding_mask: Padding mask for target sequence
            memory_key_padding_mask: Padding mask for memory sequence

        Returns:
            Processed tensor or tuple of (output tensor, intermediate outputs)
        """
        output = tgt
        self.self_attn_weights = []
        self.cross_attn_weights = []

        # If we want to return intermediate decoder outputs
        intermediate = [] if self.return_intermediate else None

        # Process through each decoder layer
        for layer in self.layers:
            output = layer(
                output,
                memory,
                tgt_mask=tgt_mask,
                memory_mask=memory_mask,
                tgt_key_padding_mask=tgt_key_padding_mask,
                memory_key_padding_mask=memory_key_padding_mask
            )

            # Store attention weights from this layer
            if hasattr(layer, 'self_attn_weights'):
                self.self_attn_weights.append(layer.self_attn_weights)

            if hasattr(layer, 'cross_attn_weights'):
                self.cross_attn_weights.append(layer.cross_attn_weights)

            # Collect intermediate outputs if requested
            if self.return_intermediate:
                if self.norm is not None:
                    intermediate.append(self.norm(output))
                else:
                    intermediate.append(output)

        # Apply final normalization if provided
        if self.norm is not None:
            output = self.norm(output)

        # Return intermediate outputs if requested
        if self.return_intermediate:
            return output, torch.stack(intermediate)

        return output

class PositionalEncoding(nn.Module):
    """
    Positional Encoding module with enhanced dimension checking and error handling.
    Ensures d_model is even and properly applies positional encoding to the input.
    """
    def __init__(self, d_model: int, dropout: float = 0.1, max_len: int = 50000):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)

        # Ensure d_model is even
        if d_model % 2 != 0:
            raise ValueError(f"d_model must be even for positional encoding, got {d_model}")

        # Explicitly initialize buffer with proper shape
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()

        # Compute div_term with precise checks
        div_term_indices = torch.arange(0, d_model, 2).float()
        try:
            div_term = torch.exp(div_term_indices * (-math.log(10000.0) / d_model))
        except Exception as e:
            raise ValueError(f"Error calculating div_term: {e}, d_model={d_model}")

        # Do additional validation
        if len(div_term) * 2 > d_model:
            raise ValueError(f"div_term length ({len(div_term)}) too large for d_model={d_model}")

        # Apply sin and cos with explicit shape checking
        try:
            pe[:, 0::2] = torch.sin(position * div_term)
            pe[:, 1::2] = torch.cos(position * div_term)
        except Exception as e:
            raise RuntimeError(f"Error in positional encoding calculation: {e}\n"
                              f"Shapes - position: {position.shape}, div_term: {div_term.shape}, "
                              f"pe: {pe.shape}, d_model: {d_model}")

        # Register as buffer (not a parameter)
        pe = pe.unsqueeze(0).transpose(0, 1)  # Shape: [max_len, 1, d_model]
        self.register_buffer('pe', pe)

        # For debugging
        self.d_model = d_model

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: Input tensor of shape [seq_len, batch_size, d_model]

        Returns:
            Tensor with positional encoding added
        """
        # Validate input shape
        if x.size(-1) != self.d_model:
            raise ValueError(f"Input feature dimension {x.size(-1)} doesn't match "
                           f"positional encoding dimension {self.d_model}")

        # Add positional encoding with proper shape handling
        try:
            max_len = min(x.size(0), self.pe.size(0))
            x = x + self.pe[:max_len, :]
        except Exception as e:
            raise RuntimeError(f"Error adding positional encoding: {e}\n"
                              f"Shapes - x: {x.shape}, pe: {self.pe.shape}, "
                              f"x size(0): {x.size(0)}, pe size(0): {self.pe.size(0)}")

        return self.dropout(x)


class GCNEncoder(nn.Module):
    """Graph Neural Network Encoder supporting both GCN and GAT with proper configuration"""
    def __init__(
        self,
        input_dim: int,
        hidden_dim: int,
        num_layers: int = 3,
        dropout: float = 0.1,
        gnn_type: str = 'gcn',
        gat_heads: int = 8,
        gat_concat: bool = True,
        residual: bool = True
    ):
        super(GCNEncoder, self).__init__()

        if not TORCH_GEOMETRIC_AVAILABLE:
            raise ImportError("PyTorch Geometric is required for GNN functionality")

        self.layers = nn.ModuleList()
        self.gnn_type = gnn_type
        self.dropout = nn.Dropout(dropout)

        # Calculate dimensions based on GAT heads if using GAT
        if gnn_type == 'gcn':
            # For GCN, dimensions are straightforward
            layer_input_dim = input_dim
            layer_output_dim = hidden_dim

            # Create GCN layers
            for i in range(num_layers):
                self.layers.append(GCNConv(layer_input_dim, layer_output_dim))
                layer_input_dim = layer_output_dim

        elif gnn_type == 'gat':
            # For first GAT layer
            if gat_concat:
                # If concatenating heads, output dim is hidden_dim * heads for first layer
                first_layer_out_dim = hidden_dim // gat_heads  # Adjust to maintain desired hidden_dim after concat
            else:
                first_layer_out_dim = hidden_dim

            # First layer
            self.layers.append(GATConv(
                in_channels=input_dim,
                out_channels=first_layer_out_dim,
                heads=gat_heads,
                concat=gat_concat,
                dropout=dropout,
                add_self_loops=True
            ))

            # For middle layers
            middle_layer_in_dim = hidden_dim if gat_concat else first_layer_out_dim

            # Middle layers (if any)
            for i in range(1, num_layers - 1):
                if gat_concat:
                    # Keep consistent hidden dimension by adjusting out_channels
                    middle_layer_out_dim = hidden_dim // gat_heads
                else:
                    middle_layer_out_dim = hidden_dim

                self.layers.append(GATConv(
                    in_channels=middle_layer_in_dim,
                    out_channels=middle_layer_out_dim,
                    heads=gat_heads,
                    concat=gat_concat,
                    dropout=dropout,
                    add_self_loops=True
                ))

                middle_layer_in_dim = hidden_dim if gat_concat else middle_layer_out_dim

            # Last layer - often use 1 head for the final layer to get exact output dimension
            if num_layers > 1:
                final_heads = 1
                self.layers.append(GATConv(
                    in_channels=middle_layer_in_dim,
                    out_channels=hidden_dim,
                    heads=final_heads,
                    concat=False,  # No concat for last layer
                    dropout=dropout,
                    add_self_loops=True
                ))
        else:
            raise ValueError(f"Invalid GNN type: {gnn_type}. Choose 'gcn' or 'gat'.")

        # Whether to use residual connections
        self.use_residual = residual

        # Layer normalization after each layer
        self.layer_norms = nn.ModuleList([
            nn.LayerNorm(hidden_dim) for _ in range(num_layers)
        ])

    def _dense_to_sparse(self, adj_matrix):
        """
        Convert dense adjacency matrix to sparse edge_index format

        Args:
            adj_matrix: Dense adjacency matrix tensor of shape [num_nodes, num_nodes]

        Returns:
            edge_index: Sparse adjacency in COO format with shape [2, num_edges]
        """
        # Get indices where values are non-zero (connections exist)
        indices = torch.nonzero(adj_matrix, as_tuple=True)

        # Stack to create edge_index format [2, num_edges]
        edge_index = torch.stack(indices)

        return edge_index

    def forward(self, x: torch.Tensor, adjacency_matrix: torch.Tensor) -> torch.Tensor:
        """
        Forward pass through GNN layers

        Args:
            x: Node features tensor [num_nodes, input_dim]
            adjacency_matrix: Adjacency matrix [num_nodes, num_nodes]

        Returns:
            Updated node features [num_nodes, hidden_dim]
        """
        # Convert dense adjacency matrix to edge_index format
        edge_index = self._dense_to_sparse(adjacency_matrix)

        # Process through GNN layers
        for i, (layer, norm) in enumerate(zip(self.layers, self.layer_norms)):
            # Store input for potential residual connection
            residual = x if x.size(-1) == layer.out_channels else None

            # Apply GNN layer
            x_new = layer(x, edge_index)

            # Apply activation except for the final layer
            if i < len(self.layers) - 1:
                x_new = F.gelu(x_new)

            # Apply dropout
            x_new = self.dropout(x_new)

            # Apply residual connection if dimensions match
            if self.use_residual and residual is not None:
                x_new = x_new + residual

            # Apply layer normalization
            x_new = norm(x_new)

            # Update x
            x = x_new

        return x

    def reset_parameters(self):
        """Reset parameters of all layers"""
        for layer in self.layers:
            layer.reset_parameters()
        for norm in self.layer_norms:
            norm.reset_parameters()

class FeatureAttention(nn.Module):
    """
    Feature-wise attention module that processes each feature group separately
    and combines them with learnable weights. Now optimized to be the primary
    processing mechanism for the transformer model.
    """
    def __init__(
        self,
        feature_dims: Dict[str, int],
        hidden_dim: int,
        num_heads: int,
        dropout: float = 0.1,
        max_seq_length: int = 50000,
        spatial_max_pooling: bool = True
    ):
        super(FeatureAttention, self).__init__()

        self.feature_dims = feature_dims
        self.hidden_dim = hidden_dim
        self.num_heads = num_heads
        self.spatial_max_pooling = spatial_max_pooling

        # Feature keys excluding 'concatenated'
        self.feature_keys = [k for k in feature_dims.keys() if k != 'concatenated']

        print(f"Initializing FeatureAttention with feature keys: {self.feature_keys}")
        print(f"Feature dimensions: {feature_dims}")

        # Create embedding layers for each feature group
        self.embeddings = nn.ModuleDict()
        for name, dim in feature_dims.items():
            print(f"Creating embedding layer for {name} with input dim {dim}, output dim {hidden_dim}")
            self.embeddings[name] = nn.Linear(dim, hidden_dim)

        # Add positional encoding for each feature group
        self.pos_encoders = nn.ModuleDict({
            name: PositionalEncoding(hidden_dim, dropout, max_seq_length)
            for name in self.feature_keys
        })

        # Feature-specific transformer blocks with separate attention
        self.feature_transformers = nn.ModuleDict({
            name: nn.TransformerEncoderLayer(
                d_model=hidden_dim,
                nhead=num_heads,
                dim_feedforward=hidden_dim * 4,
                dropout=dropout,
                batch_first=True,
                activation='gelu'
            )
            for name in self.feature_keys
        })

        # Context-adaptive feature weights
        self.context_encoder = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, len(self.feature_keys))
        )

        # Feature gating mechanism
        self.feature_gates = nn.ModuleDict({
            name: nn.Sequential(
                nn.Linear(hidden_dim * 2, hidden_dim),
                nn.LayerNorm(hidden_dim),
                nn.GELU(),
                nn.Linear(hidden_dim, hidden_dim),
                nn.Sigmoid()
            )
            for name in self.feature_keys
        })

        # Cross-attention for fusion
        self.cross_attention = nn.MultiheadAttention(
            embed_dim=hidden_dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )

        # First-level fusion to combine feature groups
        self.first_fusion = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout)
        )

        # Second-level fusion for final output
        self.fusion_layer = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout)
        )

        # Store attention weights and feature importances
        self.attention_weights = None
        self.feature_importances = {}
        self.pairwise_weights = {}
        self.gate_values = {}

    def forward(self, feature_dict: Dict[str, torch.Tensor]) -> torch.Tensor:
        """
        Process feature groups with dedicated attention and enhanced handling for spatial and GNN-processed features.
        Includes comprehensive error handling and shape verification.

        Args:
            feature_dict: Dictionary of feature tensors with shape [batch, seq, feature_dim]
                        For spatial features: [batch, seq, num_sensors, spatial_dim]

        Returns:
            Combined representation with shape [batch, seq, hidden_dim]
        """
        # Check which features are actually available in the input
        available_features = [k for k in self.feature_keys if k in feature_dict]

        if len(available_features) == 0:
            # Fall back to concatenated if no individual features are available
            if 'concatenated' in feature_dict:
                batch_size = feature_dict['concatenated'].size(0)
                seq_len = feature_dict['concatenated'].size(1)

                # Process the concatenated features through a single linear layer
                if 'traffic' in self.embeddings:
                    embedded = self.embeddings['traffic'](feature_dict['concatenated'])
                else:
                    # Create a temporary embedding layer if needed
                    temp_embedding = nn.Linear(
                        feature_dict['concatenated'].shape[-1],
                        self.hidden_dim
                    ).to(feature_dict['concatenated'].device)
                    embedded = temp_embedding(feature_dict['concatenated'])

                return embedded
            else:
                # Return dummy output if no features available at all
                batch_size = next(iter(feature_dict.values())).size(0)
                seq_len = next(iter(feature_dict.values())).size(1)
                return torch.zeros(batch_size, seq_len, self.hidden_dim,
                                device=next(iter(feature_dict.values())).device)

        # Debug: Print shapes of input features
        print("Input feature shapes:")
        for name in available_features:
            shape_info = f"{feature_dict[name].shape}" if name in feature_dict else "not available"
            print(f"  {name}: {shape_info}")

        # Process each available feature group independently
        feature_outputs = {}
        for name in available_features:
            if name not in feature_dict:
                continue

            feature_tensor = feature_dict[name]

            # Get the expected input dimension for this feature
            expected_dim = self.feature_dims.get(name, None)
            if expected_dim is None:
                print(f"Warning: Feature {name} has no dimension in feature_dims dictionary. Skipping.")
                continue

            # Debug shape info
            print(f"Processing feature {name} with shape {feature_tensor.shape}, expected dim {expected_dim}")

            # Shape handling and verification
            if name == 'traffic':
                # Special handling for traffic tensor to avoid flattening issues
                if len(feature_tensor.shape) == 3:  # [batch, seq, num_sensors]
                    batch_size, seq_len, num_sensors = feature_tensor.shape

                    # Check if dimensions match
                    if num_sensors != expected_dim:
                        print(f"Warning: Traffic feature has {num_sensors} sensors but expected {expected_dim}")
                        # Reshape to expected sensors if possible
                        if num_sensors > expected_dim:
                            print(f"Truncating traffic feature from {num_sensors} to {expected_dim}")
                            feature_tensor = feature_tensor[:, :, :expected_dim]
                        else:
                            # Pad with zeros
                            padding = torch.zeros(batch_size, seq_len, expected_dim - num_sensors,
                                              device=feature_tensor.device)
                            feature_tensor = torch.cat([feature_tensor, padding], dim=2)
                            print(f"Padded traffic feature from {num_sensors} to {expected_dim}")

                elif len(feature_tensor.shape) > 3:
                    # Handle case where traffic tensor might have additional dimensions
                    print(f"Warning: Traffic feature has {len(feature_tensor.shape)} dimensions, reshaping")
                    # Reshape to [batch, seq, num_sensors]
                    batch_size = feature_tensor.size(0)
                    seq_len = feature_tensor.size(1)
                    feature_tensor = feature_tensor.contiguous().view(batch_size, seq_len, -1)

                    # If reshaped tensor is too large, truncate
                    if feature_tensor.size(2) > expected_dim:
                        print(f"Warning: Feature {name} expected dimension {expected_dim} but got {feature_tensor.size(2)}")
                        feature_tensor = feature_tensor[:, :, :expected_dim]

            elif len(feature_tensor.shape) == 3:  # Standard [batch, seq, feature_dim]
                batch_size, seq_len, feat_dim = feature_tensor.shape

                # Check if dimensions match
                if feat_dim != expected_dim and name != 'spatial':
                    print(f"Warning: Feature {name} expected dimension {expected_dim} but got {feat_dim}")
                    # Reshape if possible
                    if feat_dim > expected_dim:
                        print(f"Truncating feature {name} from {feat_dim} to {expected_dim}")
                        feature_tensor = feature_tensor[:, :, :expected_dim]
                    elif feat_dim < expected_dim:
                        # Pad with zeros
                        padding = torch.zeros(batch_size, seq_len, expected_dim - feat_dim,
                                            device=feature_tensor.device)
                        feature_tensor = torch.cat([feature_tensor, padding], dim=2)
                        print(f"Padded feature {name} from {feat_dim} to {expected_dim}")

            # Special handling for spatial features with max pooling option
            if name == 'spatial' and len(feature_tensor.shape) == 4:  # [batch, seq, num_sensors, spatial_dim]
                batch_size, seq_len, num_sensors, spatial_dim = feature_tensor.shape
                print(f"Processing spatial feature with shape: [batch={batch_size}, seq={seq_len}, sensors={num_sensors}, dim={spatial_dim}]")

                if self.spatial_max_pooling:
                    # Apply max pooling across sensor dimension
                    # This transforms from [batch, seq, num_sensors, spatial_dim] to [batch, seq, spatial_dim]
                    try:
                        pooled_spatial = torch.max(feature_tensor, dim=2).values
                        print(f"Applied max pooling to spatial features: new shape {pooled_spatial.shape}")

                        # Continue with standard processing for the pooled features
                        embedded = self.embeddings[name](pooled_spatial)

                        # Apply positional encoding
                        embedded = embedded.transpose(0, 1)  # [seq, batch, dim]
                        embedded = self.pos_encoders[name](embedded)
                        embedded = embedded.transpose(0, 1)  # Back to [batch, seq, dim]

                        # Process with transformer
                        transformed = self.feature_transformers[name](embedded)
                        feature_outputs[name] = transformed
                        print(f"Transformed {name} shape after max pooling: {transformed.shape}")

                    except Exception as e:
                        print(f"Error processing spatial feature with max pooling: {e}")
                        continue
                else:
                    # Original approach: Process each sensor separately and then average
                    sensor_embeddings = []
                    for s in range(num_sensors):
                        try:
                            # Extract features for this sensor: [batch, seq, spatial_dim]
                            sensor_features = feature_tensor[:, :, s, :]

                            # Check if dimensions match
                            if sensor_features.shape[2] != expected_dim:
                                print(f"Warning: Spatial feature expected dimension {expected_dim} but got {sensor_features.shape[2]}")
                                # Reshape if possible
                                if sensor_features.shape[2] > expected_dim:
                                    sensor_features = sensor_features[:, :, :expected_dim]
                                else:
                                    # Pad with zeros
                                    padding = torch.zeros(batch_size, seq_len, expected_dim - sensor_features.shape[2],
                                                        device=sensor_features.device)
                                    sensor_features = torch.cat([sensor_features, padding], dim=2)

                            # Embed to common dimension
                            embedded = self.embeddings[name](sensor_features)

                            # Apply positional encoding
                            embedded = embedded.transpose(0, 1)  # [seq, batch, dim]
                            embedded = self.pos_encoders[name](embedded)
                            embedded = embedded.transpose(0, 1)  # Back to [batch, seq, dim]

                            # Process with transformer
                            transformed = self.feature_transformers[name](embedded)
                            sensor_embeddings.append(transformed)
                        except Exception as e:
                            print(f"Error processing sensor {s} for spatial feature: {e}")
                            print(f"Sensor features shape: {sensor_features.shape if 'sensor_features' in locals() else 'undefined'}")
                            print(f"Expected input dim: {expected_dim}, Embedding weight shape: {self.embeddings[name].weight.shape}")
                            continue

                    if sensor_embeddings:
                        # Instead of stack+mean which can cause memory issues, use incremental averaging
                        feature_outputs[name] = sum(sensor_embeddings) / len(sensor_embeddings)
                        print(f"Successfully processed spatial feature with {len(sensor_embeddings)} sensors (per-sensor approach)")
                    else:
                        print(f"Warning: No valid sensor embeddings for spatial feature")
                        continue
            else:
                # Standard processing for regular features
                try:
                    print(f"Embedding feature {name} with shape {feature_tensor.shape}")
                    # Embed feature to common dimension
                    embedded = self.embeddings[name](feature_tensor)
                    print(f"Embedded shape: {embedded.shape}")

                    # Apply positional encoding
                    embedded = embedded.transpose(0, 1)  # [seq, batch, dim]
                    embedded = self.pos_encoders[name](embedded)
                    embedded = embedded.transpose(0, 1)  # Back to [batch, seq, dim]

                    # Process with transformer
                    transformed = self.feature_transformers[name](embedded)
                    feature_outputs[name] = transformed
                    print(f"Transformed {name} shape: {transformed.shape}")
                except Exception as e:
                    print(f"Error processing feature {name}: {e}")
                    print(f"Feature tensor shape: {feature_tensor.shape}")
                    if name in self.embeddings:
                        print(f"Expected input dim: {expected_dim}, Embedding weight shape: {self.embeddings[name].weight.shape}")
                    continue

        # If no features were successfully processed, return zeros
        if not feature_outputs:
            batch_size = next(iter(feature_dict.values())).size(0)
            seq_len = next(iter(feature_dict.values())).size(1)
            return torch.zeros(batch_size, seq_len, self.hidden_dim,
                            device=next(iter(feature_dict.values())).device)

        # Calculate context-based feature weights
        if 'traffic' in feature_outputs:
            context_features = feature_outputs['traffic']
        else:
            # Average all available features
            context_features = torch.stack(list(feature_outputs.values())).mean(dim=0)

        # Get the list of actual feature keys we have outputs for
        available_feature_keys = list(feature_outputs.keys())
        num_available_features = len(available_feature_keys)

        # Generate global context vector
        global_context = context_features.mean(dim=1, keepdim=True)

        # Predict dynamic feature weights based on context
        raw_weights = self.context_encoder(global_context).squeeze(1)

        # IMPORTANT: Ensure weights match available features
        if raw_weights.shape[1] != num_available_features:
            print(f"Adjusting weight dimensions from {raw_weights.shape[1]} to {num_available_features}")
            # Create new correctly sized weights
            raw_weights = raw_weights[:, :num_available_features]
            if raw_weights.shape[1] < num_available_features:
                # If we have too few weights, pad with average values
                padding = torch.ones(raw_weights.shape[0], num_available_features - raw_weights.shape[1],
                                    device=raw_weights.device) / num_available_features
                raw_weights = torch.cat([raw_weights, padding], dim=1)

        # Apply softmax to get normalized weights
        normalized_weights = torch.softmax(raw_weights, dim=-1)

        # Store feature importances (average across batch)
        self.feature_importances = {}
        for i, name in enumerate(feature_outputs.keys()):
            if i < normalized_weights.shape[1]:  # Safety check
                self.feature_importances[name] = normalized_weights[:, i].mean().item()

        # Apply adaptive feature gating
        gated_features = {}
        for i, (name, feature) in enumerate(feature_outputs.items()):
            # Combine feature with global context for gating
            gate_input = torch.cat([
                feature,
                global_context.expand(-1, feature.size(1), -1)
            ], dim=2)

            # Calculate gate values
            if name in self.feature_gates:
                gate = self.feature_gates[name](gate_input)

                # Apply gate to feature
                gated_features[name] = feature * gate

                # Store gate values
                self.gate_values[name] = gate.mean().item()
            else:
                # If no gate available, use feature as is
                gated_features[name] = feature

        # Weighted combination of features
        combined_features = None
        feature_names = list(gated_features.keys())

        print(f"Combining {len(feature_names)} features with normalized weights shape: {normalized_weights.shape}")

        for i, name in enumerate(feature_names):
            # Extract batch-specific weights for this feature
            # Ensure index is within bounds
            if i < normalized_weights.shape[1]:
                feature_weight = normalized_weights[:, i].view(-1, 1, 1)
            else:
                # Use equal weight if index out of bounds
                feature_weight = torch.ones(normalized_weights.shape[0], 1, 1,
                                          device=normalized_weights.device) / len(feature_names)

            # Apply weight to feature
            weighted_feature = feature_weight * gated_features[name]

            # Combine features
            if combined_features is None:
                combined_features = weighted_feature
            else:
                combined_features = combined_features + weighted_feature

        # Apply first-level fusion
        first_level = self.first_fusion(combined_features)

        # Apply cross-attention if we have at least 2 feature groups
        if len(feature_outputs) >= 2:
            # Use traffic as query if available, otherwise use combined features
            traffic_features = feature_outputs.get('traffic', first_level)

            # Apply cross-attention
            try:
                attn_output, attn_weights = self.cross_attention(
                    traffic_features, first_level, first_level
                )

                # Store attention weights for visualization
                self.attention_weights = attn_weights

                # Enhanced fusion with both outputs
                fusion_input = torch.cat([first_level, attn_output], dim=2)
                combined_features = self.fusion_layer(fusion_input)
                print(f"Cross-attention fusion successful, final shape: {combined_features.shape}")
            except Exception as e:
                print(f"Error in cross-attention: {e}")
                # Fall back to first-level fusion
                combined_features = first_level
                print(f"Using first-level fusion fallback, shape: {first_level.shape}")
        else:
            # Only one feature group, use first level fusion
            combined_features = first_level
            print(f"Only one feature group available, using first-level fusion, shape: {first_level.shape}")

        return combined_features

class TrafficTransformer(nn.Module):
    """
    Enhanced Traffic Transformer Model with encoder-decoder architecture and
    autoregressive generation capability for multi-step prediction
    """
    def __init__(
        self,
        input_dim: int,
        hidden_dim: int,
        num_layers: int,
        num_heads: int,
        num_features: int,
        dropout: float = 0.1,
        ff_dim_multiplier: int = 4,
        activation: str = 'gelu',
        decoder_type: str = 'transformer',
        use_gnn_pre_transformer: bool = True,
        spatial_feature_dim: int = 256,
        gnn_type: str = 'gcn',
        pred_len: int = 12,
        feature_dims: Optional[Dict[str, int]] = None,
        use_spatial_bias: bool = True,
        spatial_bias_type: str = 'additive',
        max_seq_length: int = 50000,
        gnn_residual: bool = True,
        gnn_layers: int = 3,
        gat_heads: int = 16,
        gat_concat: bool = True,
        num_decoder_layers: int = 3,
        teacher_forcing_ratio: float = 0.50,
        seq_length: int = 12,
        gnn_max_pooling: bool = True,
        spatial_max_pooling: bool = True,
        config = None
    ):
        """Initialize Traffic Transformer with encoder-decoder architecture"""
        super(TrafficTransformer, self).__init__()

        # Store parameters needed for spatial bias
        self.seq_length = seq_length

        # Validate dimensions
        if hidden_dim % num_heads != 0:
            raise ValueError(f"Hidden dimension {hidden_dim} must be divisible by number of attention heads {num_heads}")

        if hidden_dim % 2 != 0:
            raise ValueError(f"Hidden dimension {hidden_dim} must be even for positional encoding, got {hidden_dim}")

        # Store instance attributes
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.num_heads = num_heads
        self.num_features = num_features
        self.pred_len = pred_len
        self.use_gnn_pre_transformer = use_gnn_pre_transformer
        self.input_dim = input_dim
        self.feature_dims = feature_dims or {'traffic': input_dim}
        self.use_spatial_bias = use_spatial_bias
        self.spatial_bias_type = spatial_bias_type
        self.decoder_type = decoder_type
        self.teacher_forcing_ratio = teacher_forcing_ratio
        self.num_decoder_layers = num_decoder_layers
        self.gnn_max_pooling = gnn_max_pooling
        self.spatial_max_pooling = spatial_max_pooling

        # For backward compatibility (embedding for legacy tensor input)
        self.embedding = nn.Linear(input_dim, hidden_dim)

        # Feature-wise attention for processing input features
        self.feature_attention = FeatureAttention(
            feature_dims=self.feature_dims,
            hidden_dim=hidden_dim,
            num_heads=num_heads,
            dropout=dropout,
            max_seq_length=max_seq_length,
            spatial_max_pooling=spatial_max_pooling
        )

        # Attention weights storage
        self.attention_weights = None

        # Create GNN encoder if needed
        if use_gnn_pre_transformer:
            if not TORCH_GEOMETRIC_AVAILABLE:
                raise ImportError("PyTorch Geometric is required for GNN pre-transformer")

            # Change input_dim to be the per-node feature dimension
            # For traffic data, typically 1 value per node (the traffic speed/flow)
            node_feature_dim = 1

            self.gnn_encoder = GCNEncoder(
                input_dim=node_feature_dim,  # Changed from input_dim to node_feature_dim
                hidden_dim=hidden_dim,
                num_layers=gnn_layers,
                dropout=dropout,
                gnn_type=gnn_type,
                gat_heads=gat_heads,
                gat_concat=gat_concat,
                residual=gnn_residual
            )

        # Positional encoding is needed for both encoder and decoder
        self.pos_encoder = PositionalEncoding(hidden_dim, dropout)

        # Spatial bias components for encoder self-attention
        if use_spatial_bias and spatial_feature_dim > 0:
            # Compute attention head dimension
            head_dim = hidden_dim // num_heads

            # Projections for spatial bias computation
            self.spatial_query_proj = nn.Linear(spatial_feature_dim, head_dim)
            self.spatial_key_proj = nn.Linear(spatial_feature_dim, head_dim)

            # Layer to compute bias from projected features
            self.spatial_bias_layer = nn.Sequential(
                nn.Linear(head_dim, 1),
                nn.Sigmoid()  # Scale to 0-1 for bias calculation
            )

        # Encoder layers with spatial bias
        self.encoder = nn.ModuleList([
            SpatialBiasTransformerEncoderLayer(
                d_model=hidden_dim,
                nhead=num_heads,
                dim_feedforward=hidden_dim * ff_dim_multiplier,
                dropout=dropout,
                activation=activation,
                batch_first=True,
                use_spatial_bias=use_spatial_bias,
                spatial_bias_type=spatial_bias_type,
                # Use seq_length from parameters rather than config
                seq_length=seq_length
            )
            for _ in range(num_layers)
        ])

        # This allows existing visualization and analysis functions to work
        self.transformer = self.encoder

        # Create decoder based on specified type
        if decoder_type == 'transformer':
            # Create a standard transformer decoder layer
            decoder_layer = TransformerDecoderLayer(
                d_model=hidden_dim,
                nhead=num_heads,
                dim_feedforward=hidden_dim * ff_dim_multiplier,
                dropout=dropout,
                activation=activation,
                batch_first=True
            )

            # Create the full decoder with multiple layers
            self.transformer_decoder = TransformerDecoder(
                decoder_layer=decoder_layer,
                num_layers=num_decoder_layers,
                norm=nn.LayerNorm(hidden_dim)
            )

            # Output projection for transformer decoder
            self.decoder_proj = nn.Linear(hidden_dim, num_features)

        elif decoder_type == 'mlp':
            # Simple MLP decoder (original approach)
            self.decoder = nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim * 2),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(hidden_dim * 2, hidden_dim),
                nn.GELU(),
                nn.Linear(hidden_dim, num_features * pred_len)
            )

        elif decoder_type == 'linear':
            # Simple linear decoder - directly maps from hidden states to output
            self.decoder = nn.Linear(hidden_dim, num_features * pred_len)

        else:
            raise ValueError(f"Invalid decoder type: {decoder_type}. Choose 'transformer', 'mlp', or 'linear'.")

        # For autoregressive generation, we need an embedding for decoder input
        self.target_embedding = nn.Linear(num_features, hidden_dim)

        # Additional components for autoregressive decoding
        self.start_token = nn.Parameter(torch.zeros(1, 1, hidden_dim))
        self.register_buffer('causal_mask', self._generate_square_subsequent_mask(pred_len))

    def _generate_square_subsequent_mask(self, sz: int) -> torch.Tensor:
        """
        Generate a causal mask for the decoder self-attention

        Args:
            sz: Size of the square mask

        Returns:
            A causal attention mask (lower triangular)
        """
        mask = torch.triu(torch.ones(sz, sz), diagonal=1)
        mask = mask.masked_fill(mask == 1, float('-inf'))
        return mask

    def compute_spatial_bias(self, spatial_features: torch.Tensor) -> torch.Tensor:
        """
        Compute spatial bias for attention using the spatial features.

        Args:
            spatial_features: Spatial features tensor [batch_size, seq_length, num_sensors, spatial_dim]

        Returns:
            Attention bias tensor with shape [batch_size, num_heads, seq_length, seq_length]
        """
        # Get dimensions from input
        batch_size = spatial_features.size(0)
        seq_length = spatial_features.size(1)

        # For static spatial relationships, we can use the first timestep
        # Shape: [batch_size, num_sensors, spatial_dim]
        static_spatial_features = spatial_features[:, 0]

        # Project spatial features to query and key space
        # Shape: [batch_size, num_sensors, head_dim]
        spatial_queries = self.spatial_query_proj(static_spatial_features)
        spatial_keys = self.spatial_key_proj(static_spatial_features)

        # Compute raw attention scores between sensors
        # Shape: [batch_size, num_sensors, num_sensors]
        # (scaled dot-product attention)
        head_dim = self.hidden_dim // self.num_heads
        scale = 1.0 / math.sqrt(head_dim)
        raw_spatial_attn = torch.bmm(spatial_queries, spatial_keys.transpose(1, 2)) * scale

        # Apply the bias layer to get attention modulation values
        # Shape: [batch_size, num_sensors, num_sensors]
        if hasattr(self, 'spatial_bias_layer'):
            # Element-wise processing of attention scores
            spatial_bias_flat = raw_spatial_attn.view(batch_size, -1, 1)
            spatial_bias_weights = self.spatial_bias_layer(spatial_bias_flat).view(batch_size, num_sensors, num_sensors)

            # Apply appropriate scaling based on bias type
            if self.spatial_bias_type == 'additive':
                # Scale to appropriate range for additive bias
                spatial_bias = (spatial_bias_weights * 2 - 1) * scale
            else:  # multiplicative
                # Keep values positive for multiplicative bias
                spatial_bias = spatial_bias_weights
        else:
            # Fallback if spatial_bias_layer is not defined
            spatial_bias = raw_spatial_attn

        # Repeat for all sequence positions and heads
        # First expand for sequence positions (assuming static spatial relationships)
        # Shape: [batch_size, seq_length, seq_length, num_heads]
        expanded_bias = spatial_bias.unsqueeze(1).expand(batch_size, seq_length, seq_length, self.num_heads)

        # Reshape to format expected by transformer attention
        # [batch_size, num_heads, seq_length, seq_length]
        return expanded_bias.permute(0, 3, 1, 2)

    def encode(
        self,
        src: Union[torch.Tensor, Dict[str, torch.Tensor]],
        adjacency_matrix: Optional[torch.Tensor] = None
    ) -> torch.Tensor:
        """
        Encode the input sequence using the encoder with improved GNN processing

        Args:
            src: Input tensor [batch, seq, features] or dictionary of feature tensors
            adjacency_matrix: Optional adjacency matrix for GNN

        Returns:
            Encoder output tensor [batch, seq, hidden_dim]
        """
        # Handle different input formats
        if isinstance(src, torch.Tensor):
            # Legacy input format - just a tensor
            src_dict = {'concatenated': src}
            src = self.embedding(src)

            # For legacy tensor input, apply positional encoding
            src = self.pos_encoder(src)

            use_feature_attention = False
            spatial_features = None
        else:
            # Dictionary of feature tensors
            src_dict = src.copy()  # Create a copy to avoid modifying the input

            # Extract spatial features if available
            spatial_features = src_dict.get('spatial', None)

            # Handle 3D spatial features
            if spatial_features is not None and len(spatial_features.shape) == 4:
                # Format: [batch, seq_len, num_sensors, spatial_dim]
                spatial_features_3d = spatial_features

                # Remove spatial from the feature dictionary if we're using it only for bias
                if 'spatial' in src_dict and self.use_spatial_bias:
                    src_dict.pop('spatial')

            # Apply GNN if enabled - IMPROVED GNN PROCESSING HERE
            if self.use_gnn_pre_transformer and adjacency_matrix is not None and 'traffic' in src_dict:
                batch_size, seq_len, num_sensors = src_dict['traffic'].shape

                # Create GNN-enhanced traffic features
                gnn_enhanced_traffic = []

                # Process each time step separately
                for t in range(seq_len):
                    # Extract batch of sensor values for this time step: [batch, num_sensors]
                    time_features = src_dict['traffic'][:, t, :]

                    # Process each batch item individually or use PyG's batching
                    batch_enhanced_features = []

                    for b in range(batch_size):
                        # Extract single batch item's features: [num_sensors]
                        # Reshape to [num_sensors, 1] to represent 1 feature per node
                        node_features = time_features[b].view(num_sensors, 1)

                        # Process through GNN
                        enhanced_nodes = self.gnn_encoder(node_features, adjacency_matrix)

                        # Add to batch results
                        batch_enhanced_features.append(enhanced_nodes)

                    # Stack batch results: [batch, num_sensors, hidden_dim]
                    enhanced_time_features = torch.stack(batch_enhanced_features)

                    # Add to sequence
                    gnn_enhanced_traffic.append(enhanced_time_features)

                # Stack across time dimension: [batch, seq, num_sensors, hidden_dim]
                gnn_enhanced = torch.stack(gnn_enhanced_traffic, dim=1)

                # Add max pooling option here:
                if hasattr(self, 'gnn_max_pooling') and self.gnn_max_pooling:
                    # Apply max pooling across sensor dimension
                    # This transforms from [batch, seq, num_sensors, hidden_dim] to [batch, seq, hidden_dim]
                    gnn_aggregated = torch.max(gnn_enhanced, dim=2).values
                    # Set this as the GNN traffic feature
                    src_dict['gnn_traffic'] = gnn_aggregated
                else:
                    # Original behavior: flatten the tensor
                    src_dict['gnn_traffic'] = gnn_enhanced.view(batch_size, seq_len, -1)

            # Apply feature attention
            src = self.feature_attention(src_dict)
            use_feature_attention = True

            # Get attention weights
            if hasattr(self.feature_attention, 'attention_weights'):
                self.attention_weights = self.feature_attention.attention_weights

            # Store feature importances
            if hasattr(self.feature_attention, 'feature_importances'):
                self.feature_importances = self.feature_attention.feature_importances

        # Compute spatial bias if enabled
        spatial_bias = None
        if self.use_spatial_bias and hasattr(self, 'spatial_query_proj'):
            try:
                # Use the 3D spatial features if available
                if 'spatial_features_3d' in locals() and spatial_features_3d is not None:
                    spatial_bias = self.compute_spatial_bias(spatial_features_3d)
                elif spatial_features is not None:
                    spatial_bias = self.compute_spatial_bias(spatial_features)
            except Exception as e:
                print(f"Warning: Error computing spatial bias: {e}")

        # Apply encoder transformer layers
        memory = src
        for i, layer in enumerate(self.encoder):
            # Set spatial bias if available
            if spatial_bias is not None and hasattr(layer, 'set_spatial_bias'):
                layer.set_spatial_bias(spatial_bias)

            # Apply layer
            memory = layer(memory)

            # Capture attention weights from last layer
            if i == len(self.encoder) - 1 and hasattr(layer, 'attn_weights'):
                if self.attention_weights is None and layer.attn_weights is not None:
                    self.attention_weights = layer.attn_weights

        return memory

    def decode_single_step(
        self,
        memory: torch.Tensor,
        decoder_input: torch.Tensor,
        decoder_mask: Optional[torch.Tensor] = None
    ) -> torch.Tensor:
        """
        Decode a single step using the transformer decoder

        Args:
            memory: Encoder output [batch, seq, hidden_dim]
            decoder_input: Decoder input [batch, tgt_len, hidden_dim]
            decoder_mask: Optional mask for decoder self-attention

        Returns:
            Output prediction for a single step [batch, num_features]
        """
        if self.decoder_type == 'transformer':
            # Use transformer decoder
            decoder_output = self.transformer_decoder(
                tgt=decoder_input,
                memory=memory,
                tgt_mask=decoder_mask
            )

            # Take last position output and project to feature dimension
            pred = self.decoder_proj(decoder_output[:, -1])
            return pred
        else:
            # Use MLP decoder (original approach)
            # No autoregressive decoding in this case
            return None

    def generate(
        self,
        memory: torch.Tensor,
        steps: int = None,
        temperature: float = 1.0,
        initial_input: Optional[torch.Tensor] = None
    ) -> torch.Tensor:
        """
        Generate a sequence autoregressively using the decoder

        Args:
            memory: Encoder output [batch, seq, hidden_dim]
            steps: Number of steps to generate (defaults to self.pred_len)
            temperature: Sampling temperature (1.0 = greedy)
            initial_input: Optional initial decoder input

        Returns:
            Generated sequence [batch, steps, num_features]
        """
        if steps is None:
            steps = self.pred_len

        if self.decoder_type != 'transformer':
            # For non-transformer decoder, use standard forward pass
            outputs = self.forward({'memory': memory})
            return outputs

        # Get batch size
        batch_size = memory.size(0)
        device = memory.device

        # Initialize with start token or provided initial input
        if initial_input is not None:
            # Use provided initial input
            if initial_input.size(-1) != self.hidden_dim:
                # Embed if not already embedded
                decoder_input = self.target_embedding(initial_input)
            else:
                decoder_input = initial_input
        else:
            # Use start token
            decoder_input = self.start_token.expand(batch_size, 1, -1)

        # List to store predictions
        predictions = []

        # Generate sequence step by step
        for i in range(steps):
            # Get appropriate decoder mask for the current sequence length
            tgt_len = decoder_input.size(1)

            if tgt_len > 1:
                # Use a custom mask for this length
                decoder_mask = self._generate_square_subsequent_mask(tgt_len).to(device)
            else:
                # No mask needed for a single token
                decoder_mask = None

            # Process through decoder to get next token
            decoder_output = self.transformer_decoder(
                tgt=decoder_input,
                memory=memory,
                tgt_mask=decoder_mask
            )

            # Take the output from the last position
            last_token = decoder_output[:, -1:]  # Keep the time dimension

            # Project to feature space
            next_features = self.decoder_proj(last_token)

            # Apply temperature if not 1.0
            if temperature != 1.0:
                # This is not really applicable to regression, but kept for API consistency
                next_features = next_features / temperature

            # Store prediction
            predictions.append(next_features)

            # Embed the new token
            next_token_embedded = self.target_embedding(next_features)

            # Add to decoder input for next iteration
            decoder_input = torch.cat([decoder_input, next_token_embedded], dim=1)

        # Concatenate predictions along the time dimension
        if len(predictions) > 0:
            # Shape becomes [batch, steps, num_features]
            return torch.cat(predictions, dim=1)
        else:
            # Return empty tensor with correct shape if no predictions
            return torch.zeros(batch_size, 0, self.num_features, device=device)

    def forward(
        self,
        src: Dict[str, torch.Tensor],
        target: Optional[torch.Tensor] = None,
        adjacency_matrix: Optional[torch.Tensor] = None
    ) -> torch.Tensor:
        """
        Forward pass focused exclusively on feature-wise attention

        Args:
            src: Dictionary of feature tensors with shape [batch, seq, feature_dim]
                For spatial features: [batch, seq, num_sensors, spatial_dim]
            target: Optional target values for teacher forcing
            adjacency_matrix: Optional adjacency matrix for GNN

        Returns:
            Output tensor [batch, pred_len, num_features]
        """
        # Process input through feature-wise attention
        memory = self.encode(src, adjacency_matrix)

        # Different approaches based on decoder type
        if self.decoder_type == 'transformer':
            # Use autoregressive transformer decoder
            batch_size = memory.size(0)
            device = memory.device

            if self.training and target is not None and random.random() < self.teacher_forcing_ratio:
                # Teacher forcing mode (during training)
                # Shift target to create decoder input (remove last step, prepend start token)
                decoder_input = target[:, :-1, :]  # Remove last step

                # Prepend start token
                start_tokens = self.start_token.expand(batch_size, 1, -1)
                if decoder_input.size(1) > 0:
                    # Combine start token with shifted target
                    decoder_input = torch.cat([
                        start_tokens,
                        self.target_embedding(decoder_input)
                    ], dim=1)
                else:
                    # Use only start token if target is just one step
                    decoder_input = start_tokens

                # Create causal mask for decoder self-attention
                tgt_len = decoder_input.size(1)
                decoder_mask = self._generate_square_subsequent_mask(tgt_len).to(device)

                # Process through transformer decoder
                decoder_output = self.transformer_decoder(
                    tgt=decoder_input,
                    memory=memory,
                    tgt_mask=decoder_mask
                )

                # Project decoder output to feature space
                predictions = self.decoder_proj(decoder_output)

                # For training return all steps
                return predictions
            else:
                # Autoregressive generation mode (evaluation or no teacher forcing)
                predictions = self.generate(
                    memory=memory,
                    steps=self.pred_len,
                    temperature=1.0
                )

                return predictions
        else:
            # Use original MLP or linear decoder approach
            # Apply decoder to last encoder output
            output = self.decoder(memory[:, -1, :])

            # Reshape output to [batch, pred_len, num_features]
            return output.view(-1, self.pred_len, self.num_features)

    def freeze_layers(self, freeze_encoder: bool = True, num_layers: int = 1):
        """Freeze specified parts of the model for transfer learning"""
        if freeze_encoder:
            # Freeze embedding layer
            for param in self.embedding.parameters():
                param.requires_grad = False

            # Freeze positional encoding
            if hasattr(self.pos_encoder, 'pe'):
                self.pos_encoder.pe.requires_grad = False

            # Freeze specified transformer layers
            for i, layer in enumerate(self.encoder):
                if i < num_layers:
                    for param in layer.parameters():
                        param.requires_grad = False

            print(f"Froze embedding layer and {num_layers} transformer layers")
        else:
            print("No parameter freezing applied - full fine-tuning")

        # Report number of trainable parameters
        trainable_params = sum(p.numel() for p in self.parameters() if p.requires_grad)
        total_params = sum(p.numel() for p in self.parameters())
        print(f"Trainable parameters: {trainable_params:,} of {total_params:,} total ({trainable_params/total_params:.1%})")


class PyTorchLSTMForecaster(nn.Module):
    """Improved LSTM-based model for time series forecasting"""
    def __init__(
        self,
        input_size: int,
        hidden_size: int,
        output_size: int,
        num_layers: int,
        seq_length: int,  # Added sequence length parameter
        pred_length: int,  # Added prediction length parameter
        dropout: float = 0.1,
        epochs: int = 100,
        batch_size: int = 32,
        learning_rate: float = 0.001,
        device: str = 'cpu'
    ):
        super(PyTorchLSTMForecaster, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.seq_length = seq_length
        self.pred_length = pred_length
        self.input_size = input_size
        self.output_size = output_size

        # LSTM with dropout
        self.lstm = nn.LSTM(
            input_size,
            hidden_size,
            num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )

        # Decoder to map from hidden state to output
        self.decoder = nn.Linear(hidden_size, output_size * pred_length)

        # Training parameters
        self.epochs = epochs
        self.batch_size = batch_size
        self.learning_rate = learning_rate
        self.device = device
        self.history = {'loss': [], 'val_loss': []}  # Store training history

    def forward(self, x):
        """
        Forward pass through LSTM model

        Args:
            x: Input tensor of shape [batch_size, seq_length, input_size]

        Returns:
            Output tensor of shape [batch_size, pred_length, output_size]
        """
        # Initialize hidden state and cell state
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)

        # Forward propagate LSTM
        out, _ = self.lstm(x, (h0, c0))

        # Decode the hidden state of the last time step
        out = self.decoder(out[:, -1, :])

        # Reshape to [batch_size, pred_length, output_size]
        out = out.view(-1, self.pred_length, self.output_size)

        return out

    def fit(self, X_train, y_train, X_val=None, y_val=None):
        """
        Train the LSTM model on input data with proper sequence handling

        Args:
            X_train: Training input of shape [num_samples, seq_length, input_size]
            y_train: Training target of shape [num_samples, pred_length, output_size]
            X_val: Optional validation input
            y_val: Optional validation target
        """
        self.to(self.device)
        optimizer = optim.Adam(self.parameters(), lr=self.learning_rate)
        criterion = nn.MSELoss()

        # Convert numpy arrays to tensors if needed
        if not isinstance(X_train, torch.Tensor):
            X_train = torch.tensor(X_train, dtype=torch.float32)
        if not isinstance(y_train, torch.Tensor):
            y_train = torch.tensor(y_train, dtype=torch.float32)

        # Move to device
        X_train = X_train.to(self.device)
        y_train = y_train.to(self.device)

        # Prepare validation data if provided
        if X_val is not None and y_val is not None:
            if not isinstance(X_val, torch.Tensor):
                X_val = torch.tensor(X_val, dtype=torch.float32)
            if not isinstance(y_val, torch.Tensor):
                y_val = torch.tensor(y_val, dtype=torch.float32)

            X_val = X_val.to(self.device)
            y_val = y_val.to(self.device)

        # Create data loaders
        train_dataset = TensorDataset(X_train, y_train)
        train_loader = DataLoader(
            train_dataset,
            batch_size=self.batch_size,
            shuffle=True
        )

        # Early stopping
        best_loss = float('inf')
        patience = 25
        no_improve = 0

        for epoch in range(self.epochs):
            self.train()  # Set model to training mode
            total_loss = 0

            for X_batch, y_batch in train_loader:
                # Forward pass
                outputs = self(X_batch)
                loss = criterion(outputs, y_batch)

                # Backward and optimize
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

                total_loss += loss.item()

            avg_train_loss = total_loss / len(train_loader)
            self.history['loss'].append(avg_train_loss)

            # Validation if data provided
            if X_val is not None and y_val is not None:
                self.eval()  # Set model to evaluation mode
                with torch.no_grad():
                    val_outputs = self(X_val)
                    val_loss = criterion(val_outputs, y_val)

                val_loss = val_loss.item()
                self.history['val_loss'].append(val_loss)

                # Early stopping check
                if val_loss < best_loss:
                    best_loss = val_loss
                    no_improve = 0
                else:
                    no_improve += 1

                if no_improve >= patience:
                    print(f'Early stopping at epoch {epoch+1}')
                    break

                print(f'Epoch [{epoch+1}/{self.epochs}], Train Loss: {avg_train_loss:.4f}, Val Loss: {val_loss:.4f}')
            else:
                print(f'Epoch [{epoch+1}/{self.epochs}], Loss: {avg_train_loss:.4f}')

    def predict(self, X):
        """
        Generate predictions with the trained model

        Args:
            X: Input data of shape [num_samples, seq_length, input_size]

        Returns:
            Predictions of shape [num_samples, pred_length, output_size]
        """
        self.eval()  # Set model to evaluation mode

        # Convert to tensor if needed
        if not isinstance(X, torch.Tensor):
            X = torch.tensor(X, dtype=torch.float32)

        # Move to device
        X = X.to(self.device)

        with torch.no_grad():
            predictions = self(X)

        return predictions.cpu().numpy()

# Cosine learning rate scheduler with warmup
class CosineWarmupLR(_optim.lr_scheduler.LRScheduler):
    """Cosine annealing with warmup learning rate scheduler"""
    def __init__(
        self,
        optimizer: torch.optim.Optimizer,
        warmup_epochs: int,
        total_epochs: int,
        base_lr: float,
        warmup_lr: float = 0.0,
        last_epoch: int = -1
    ):
        self.warmup_epochs = warmup_epochs
        self.total_epochs = total_epochs
        self.base_lr = base_lr
        self.warmup_lr = warmup_lr
        super().__init__(optimizer, last_epoch)

    def get_lr(self) -> List[float]:
        if self.last_epoch < self.warmup_epochs:
            # Linear warmup phase
            alpha = self.last_epoch / self.warmup_epochs
            return [self.warmup_lr + (self.base_lr - self.warmup_lr) * alpha] * len(self.optimizer.param_groups)
        else:
            # Cosine annealing phase
            progress = float(self.last_epoch - self.warmup_epochs) / float(max(1, self.total_epochs - self.warmup_epochs))
            return [max(0.0, self.base_lr * 0.5 * (1.0 + math.cos(math.pi * progress)))] * len(self.optimizer.param_groups)

## Training Module

In [ ]:

# =============================================================================
# Training Module
# =============================================================================

def train_model(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    scheduler: Optional[torch.optim.lr_scheduler._LRScheduler],
    criterion: Callable,
    config: TrainingConfig,
    data_scaler: Any,
    device: str = 'cpu',
    adjacency_matrix: Optional[torch.Tensor] = None
) -> Tuple[nn.Module, List[float], List[float]]:
    """
    Train the traffic forecasting model with support for transformer decoder and teacher forcing
    """
    num_epochs = config.num_epochs
    patience = config.patience
    best_loss = float('inf')
    no_improve = 0
    train_losses = []
    val_losses = []
    model_dir = getattr(config, 'model_dir', None)
    use_quantile_regression = config.use_quantile_regression
    loss_function_type = config.loss_function
    accumulation_steps = config.accumulation_steps

    # Check if the model uses transformer decoder
    uses_transformer_decoder = (
        hasattr(model, 'decoder_type') and
        getattr(model, 'decoder_type', None) == 'transformer'
    )

    # Setup for mixed precision training
    use_amp = config.use_mixed_precision and AMP_AVAILABLE and device != 'cpu'
    if use_amp:
        if DEVICE_TYPE_SUPPORTED:
            scaler = GradScaler(device_type='cuda')
        else:
            # Older PyTorch versions don't support device_type
            scaler = GradScaler()
    else:
        scaler = None

    # Log memory usage initially
    if torch.cuda.is_available() and device == 'cuda':
        print(f"Initial GPU memory allocated: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")
        print(f"Initial GPU memory reserved: {torch.cuda.memory_reserved(0) / 1e9:.2f} GB")

    for epoch in range(num_epochs):
        model.train()
        train_loss = 0
        optimizer.zero_grad()  # Zero gradients at the start of each epoch

        for batch_idx, batch_data in enumerate(train_loader):
            data, target = batch_data

            # Handle dictionary data format
            if isinstance(data, dict):
                data = {k: v.to(device) for k, v in data.items()}
            else:
                data = data.to(device)
            target = target.to(device)

            # Mixed precision forward pass
            if use_amp:
                if DEVICE_TYPE_SUPPORTED:
                    with autocast(device_type='cuda'):
                        # Forward pass with target for teacher forcing if model supports it
                        if uses_transformer_decoder:
                            if config.use_spatial_features and config.use_gnn_pre_transformer:
                                output = model(data, target, adjacency_matrix.to(device))
                            else:
                                output = model(data, target)
                        else:
                            # Standard forward pass (no target)
                            if config.use_spatial_features and config.use_gnn_pre_transformer:
                                output = model(data, adjacency_matrix=adjacency_matrix.to(device))
                            else:
                                output = model(data)

                        if use_quantile_regression:
                            quantiles = config.quantiles
                            loss = quantile_loss(output, target, quantiles) / accumulation_steps
                        elif loss_function_type == 'hybrid':
                            loss = hybrid_loss(output, target) / accumulation_steps
                        else:
                            loss = criterion(output, target) / accumulation_steps
                else:
                    # Older PyTorch versions
                    with autocast():
                        # Forward pass with target for teacher forcing if model supports it
                        if uses_transformer_decoder:
                            if config.use_spatial_features and config.use_gnn_pre_transformer:
                                output = model(data, target, adjacency_matrix.to(device))
                            else:
                                output = model(data, target)
                        else:
                            # Standard forward pass (no target)
                            if config.use_spatial_features and config.use_gnn_pre_transformer:
                                output = model(data, adjacency_matrix=adjacency_matrix.to(device))
                            else:
                                output = model(data)

                        if use_quantile_regression:
                            quantiles = config.quantiles
                            loss = quantile_loss(output, target, quantiles) / accumulation_steps
                        elif loss_function_type == 'hybrid':
                            loss = hybrid_loss(output, target) / accumulation_steps
                        else:
                            loss = criterion(output, target) / accumulation_steps

                # Mixed precision backward pass
                scaler.scale(loss).backward()

                # Gradient accumulation - only step every accumulation_steps
                if (batch_idx + 1) % accumulation_steps == 0 or (batch_idx + 1) == len(train_loader):
                    if config.gradient_clip is not None:
                        scaler.unscale_(optimizer)
                        nn.utils.clip_grad_norm_(model.parameters(), config.gradient_clip)

                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad()
            else:
                # Standard precision training
                # Forward pass with target for teacher forcing if model supports it
                if uses_transformer_decoder:
                    if config.use_spatial_features and config.use_gnn_pre_transformer:
                        output = model(data, target, adjacency_matrix.to(device))
                    else:
                        output = model(data, target)
                else:
                    # Standard forward pass (no target)
                    if config.use_spatial_features and config.use_gnn_pre_transformer:
                        output = model(data, adjacency_matrix=adjacency_matrix.to(device))
                    else:
                        output = model(data)

                if use_quantile_regression:
                    quantiles = config.quantiles
                    loss = quantile_loss(output, target, quantiles) / accumulation_steps
                elif loss_function_type == 'hybrid':
                    loss = hybrid_loss(output, target) / accumulation_steps
                else:
                    loss = criterion(output, target) / accumulation_steps

                # Standard backward pass
                loss.backward()

                # Gradient accumulation - only step every accumulation_steps
                if (batch_idx + 1) % accumulation_steps == 0 or (batch_idx + 1) == len(train_loader):
                    if config.gradient_clip is not None:
                        nn.utils.clip_grad_norm_(model.parameters(), config.gradient_clip)

                    optimizer.step()
                    optimizer.zero_grad()

            # Track loss (use full loss for logging)
            train_loss += loss.item() * accumulation_steps

        avg_train_loss = train_loss / len(train_loader)
        train_losses.append(avg_train_loss)

        # Validation
        avg_val_loss, _ = evaluate_model(
            model=model,
            dataloader=val_loader,
            criterion=criterion,
            device=device,
            config=config,
            adjacency_matrix=adjacency_matrix,
            data_scaler=data_scaler
        )
        val_losses.append(avg_val_loss)

        # Early stopping check
        if avg_val_loss < best_loss:
            best_loss = avg_val_loss
            no_improve = 0

            # Save best model if model_dir is available
            if model_dir is not None:
                try:
                    timestamp = get_maputo_timestamp()
                    model_path = os.path.join(model_dir, f'best_model_{timestamp}.pth')
                    torch.save({
                        'epoch': epoch,
                        'model_state_dict': model.state_dict(),
                        'optimizer_state_dict': optimizer.state_dict(),
                        'loss': best_loss,
                        'config': {k: v for k, v in vars(config).items() if not k.startswith('_')}
                    }, model_path)
                except Exception as e:
                    print(f"Warning: Could not save model: {str(e)}")
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f'Early stopping at epoch {epoch+1}')
                break

        # Update learning rate
        if config.scheduler_type == 'cosine_warmup':
            scheduler.step()
        elif scheduler and config.scheduler_type == 'plateau':
            scheduler.step(avg_val_loss)
        elif scheduler:  # Other scheduler types
            scheduler.step()

        print(f'Epoch {epoch+1}/{num_epochs} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}')

        # If model has feature importances, print them
        if hasattr(model, 'feature_importances') and model.feature_importances:
            print("Feature Importances:")
            for feature, importance in model.feature_importances.items():
                print(f"  {feature}: {importance:.4f}")

    return model, train_losses, val_losses

def evaluate_model(
    model: nn.Module,
    dataloader: DataLoader,
    criterion: Callable,
    data_scaler: Any,
    device: str = 'cpu',
    config: Optional[TrainingConfig] = None,
    adjacency_matrix: Optional[torch.Tensor] = None
) -> Tuple[float, Tuple[float, float, float, float]]:
    model.eval()
    total_loss = 0
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for batch_data in dataloader:
            # Handle different data formats
            if isinstance(batch_data, (tuple, list)) and len(batch_data) == 2:
                data, target = batch_data
                if isinstance(data, dict):
                    data = {k: v.to(device) for k, v in data.items()}
                else:
                    data = data.to(device)
                target = target.to(device)
            else:
                print(f"Unexpected batch_data format: {type(batch_data)}")
                continue

            # Run model forward pass
            output = model(data)
            loss = criterion(output, target)

            total_loss += loss.item()
            all_preds.append(output.cpu().numpy())
            all_targets.append(target.cpu().numpy())

    # After we're done, check if we have any predictions
    if not all_preds:
        print("No predictions were made! Check the error messages above.")
        # Return dummy values
        return 999.0, (999.0, 999.0, 0.0, 999.0)

    predictions = np.concatenate(all_preds)
    actuals = np.concatenate(all_targets)

    # Reshape for inverse transformation
    num_samples, pred_window, num_features = predictions.shape
    predictions_2d = predictions.reshape(-1, num_features)
    actuals_2d = actuals.reshape(-1, num_features)

    # Inverse transform
    predictions_inv = data_scaler.inverse_transform(predictions_2d)
    actuals_inv = data_scaler.inverse_transform(actuals_2d)

    # Calculate metrics
    mae = mean_absolute_error(actuals_inv.ravel(), predictions_inv.ravel())
    rmse = np.sqrt(mean_squared_error(actuals_inv.ravel(), predictions_inv.ravel()))
    r2 = r2_score(actuals_inv.ravel(), predictions_inv.ravel())
    mape = robust_mape(actuals_inv.ravel(), predictions_inv.ravel())

    print(f'MAE: {mae:.2f}, RMSE: {rmse:.2f}, R²: {r2:.2f}, MAPE: {mape:.2f}%')

    return total_loss / len(dataloader), (mae, rmse, r2, mape)

def predict(
    model: nn.Module,
    dataloader: DataLoader,
    scaler: Any,
    device: str = 'cpu',
    adjacency_matrix: Optional[torch.Tensor] = None,
    config: Optional[TrainingConfig] = None
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Make predictions using the trained model with support for autoregressive generation
    """
    model.eval()
    all_preds = []
    all_targets = []

    # Use AMP for prediction if available on GPU
    use_amp = hasattr(torch.cuda, 'amp') and device != 'cpu'

    with torch.no_grad():
        for batch_data in dataloader:
            data, target = batch_data

            # Handle dictionary data format
            if isinstance(data, dict):
                data = {k: v.to(device) for k, v in data.items()}
            else:
                data = data.to(device)
            target = target.to(device)

            if use_amp:
                if DEVICE_TYPE_SUPPORTED:
                    with autocast(device_type='cuda'):
                        # For prediction, don't use teacher forcing
                        if hasattr(dataloader.dataset, 'config') and dataloader.dataset.config.use_spatial_features and dataloader.dataset.config.use_gnn_pre_transformer:
                            output = model(data, adjacency_matrix=adjacency_matrix.to(device))
                        else:
                            output = model(data)
                else:
                    # Older PyTorch versions
                    with autocast():
                        # For prediction, don't use teacher forcing
                        if hasattr(dataloader.dataset, 'config') and dataloader.dataset.config.use_spatial_features and dataloader.dataset.config.use_gnn_pre_transformer:
                            output = model(data, adjacency_matrix=adjacency_matrix.to(device))
                        else:
                            output = model(data)
            else:
                # For prediction, don't use teacher forcing
                if hasattr(dataloader.dataset, 'config') and dataloader.dataset.config.use_spatial_features and dataloader.dataset.config.use_gnn_pre_transformer:
                    output = model(data, adjacency_matrix=adjacency_matrix.to(device))
                else:
                    output = model(data)

            all_preds.append(output.cpu().numpy())
            all_targets.append(target.cpu().numpy())

    predictions = np.concatenate(all_preds)
    actuals = np.concatenate(all_targets)

    # Reshape for inverse transformation
    num_samples, pred_window, num_features = predictions.shape
    predictions_2d = predictions.reshape(-1, num_features)
    actuals_2d = actuals.reshape(-1, num_features)

    # Inverse transform
    predictions_inv = scaler.inverse_transform(predictions_2d)
    actuals_inv = scaler.inverse_transform(actuals_2d)

    return predictions_inv, actuals_inv

def evaluate_baseline_model(predictions: np.ndarray, actuals: np.ndarray) -> Tuple[float, float, float, float]:
    """
    Evaluate baseline model predictions against actuals

    Args:
        predictions: Predicted values
        actuals: Actual values

    Returns:
        Tuple of (MAE, RMSE, R², MAPE)
    """
    mae = mean_absolute_error(actuals.ravel(), predictions.ravel())
    rmse = np.sqrt(mean_squared_error(actuals.ravel(), predictions.ravel()))
    r2 = r2_score(actuals.ravel(), predictions.ravel())
    mape = robust_mape(actuals.ravel(), predictions.ravel())
    return mae, rmse, r2, mape

def train_baseline_models(
    train_data: np.ndarray,
    test_data: np.ndarray,
    config: TrainingConfig,
    device: str,
    timestamps_train: Optional[pd.DatetimeIndex] = None,
    timestamps_test: Optional[pd.DatetimeIndex] = None
) -> Dict[str, List[Tuple[float, float, float, float]]]:
    """Return empty baseline metrics - ARIMA, Exp Smoothing and LSTM baselines removed"""
    print("Baseline models (ARIMA, ExpSmoothing, LSTM) have been disabled")

    # Return empty dictionary of metrics
    return {'naive_forecast': []}

# --- Loss Functions ---
def quantile_loss(output: torch.Tensor, target: torch.Tensor, quantiles: List[float]) -> torch.Tensor:
    """
    Quantile Loss function for prediction intervals

    Args:
        output: Model output with shape [batch, pred_len, num_quantiles]
        target: Target values
        quantiles: List of quantiles

    Returns:
        Quantile loss value
    """
    losses = []
    for i, q in enumerate(quantiles):
        errors = target - output[:, :, i]
        losses.append(torch.max((q-1) * errors, q * errors).mean())
    loss = torch.sum(torch.stack(losses))
    return loss


def hybrid_loss(output: torch.Tensor, target: torch.Tensor, alpha: float = 0.5) -> torch.Tensor:
    """
    Hybrid Loss function: Weighted combination of MSE and MAE

    Args:
        output: Model output
        target: Target values
        alpha: Weight for MSE component (1-alpha for MAE)

    Returns:
        Hybrid loss value
    """
    mse_loss = nn.MSELoss()(output, target)
    mae_loss = nn.L1Loss()(output, target)
    loss = alpha * mse_loss + (1 - alpha) * mae_loss
    return loss


def robust_mape(y_true: np.ndarray, y_pred: np.ndarray, epsilon: float = 1e-8) -> float:
    """
    Robust MAPE to handle division by zero and near-zero values

    Args:
        y_true: True values
        y_pred: Predicted values
        epsilon: Small value to prevent division by zero

    Returns:
        MAPE value as percentage
    """
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    mask = y_true != 0
    if not np.any(mask):
        return np.nan  # Or 0, or another appropriate value if all y_true are zero
    y_true_masked = y_true[mask]
    y_pred_masked = y_pred[mask]
    return np.mean(np.abs((y_true_masked - y_pred_masked) / (y_true_masked + epsilon))) * 100

##Transfer Learning Training Module

In [ ]:
# =============================================================================
# Transfer Learning Training Module
# =============================================================================

def train_transfer_model(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    source_loader: Optional[DataLoader] = None,
    optimizer: Optional[torch.optim.Optimizer] = None,
    scheduler: Optional[torch.optim.lr_scheduler._LRScheduler] = None,
    criterion: Optional[Callable] = None,
    config: TrainingConfig = None,
    data_scaler: Optional[Union[MinMaxScaler, StandardScaler, RobustScaler]] = None,
    source_scaler: Optional[Union[MinMaxScaler, StandardScaler, RobustScaler]] = None,
    device: str = 'cpu',
    adjacency_matrix: Optional[torch.Tensor] = None
) -> Tuple[nn.Module, Dict[str, List[float]], Dict[str, Dict[str, float]]]:
    """
    Train a model using transfer learning from a pre-trained model.

    Args:
        model: Pre-trained model to fine-tune
        train_loader: DataLoader with training data from target dataset
        val_loader: DataLoader with validation data from target dataset
        source_loader: Optional DataLoader with test data from source dataset (for comparison)
        optimizer: Optional optimizer (will be created if None)
        scheduler: Optional learning rate scheduler (will be created if None)
        criterion: Optional loss function (will be created if None)
        config: Training configuration
        data_scaler: Scaler used for the target dataset
        source_scaler: Scaler used for the source dataset (required if source_loader is provided)
        device: Device to train on ('cpu' or 'cuda')
        adjacency_matrix: Optional adjacency matrix for GNN

    Returns:
        Tuple of (fine-tuned model, training history, evaluation metrics)
    """
    if config is None:
        raise ValueError("Configuration object is required for transfer learning")

    # Setup directories for saving results
    output_dir = config.output_dir
    results_dir = config.results_dir
    model_dir = config.model_dir

    # Get training parameters specific to transfer learning
    num_epochs = min(30, config.num_epochs)  # Typically need fewer epochs for fine-tuning
    patience = min(5, config.patience)       # Shorter patience for transfer learning
    learning_rate = config.transfer_learning_rate if hasattr(config, 'transfer_learning_rate') else 5e-5
    target_dataset_name = config.target_dataset_name if hasattr(config, 'target_dataset_name') else "target"

    # Print transfer learning setup
    print(f"\n=== Transfer Learning Setup ({target_dataset_name}) ===")
    print(f"Learning Rate: {learning_rate}")
    print(f"Max Epochs: {num_epochs}")
    print(f"Patience: {patience}")
    print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

    # Create optimizer if not provided (only optimize parameters that require gradients)
    if optimizer is None:
        optimizer = optim.AdamW(
            [p for p in model.parameters() if p.requires_grad],
            lr=learning_rate,
            weight_decay=0.01  # Slightly stronger regularization for fine-tuning
        )

    # Create scheduler if not provided
    if scheduler is None:
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', patience=patience//2, factor=0.5, verbose=True
        )

    # Create criterion if not provided
    if criterion is None:
        if config.loss_function == 'mse':
            criterion = nn.MSELoss()
        elif config.loss_function == 'mae':
            criterion = nn.L1Loss()
        elif config.loss_function == 'huber':
            criterion = nn.SmoothL1Loss()
        else:
            criterion = nn.MSELoss()

    # Initialize tracking variables
    best_loss = float('inf')
    no_improve = 0
    history = {
        'train_loss': [],
        'val_loss': []
    }
    metrics = {
        'source': None,
        'target': None
    }

    # Setup for mixed precision training
    use_amp = config.use_mixed_precision and AMP_AVAILABLE and device != 'cpu'
    if use_amp:
        if DEVICE_TYPE_SUPPORTED:
            scaler = GradScaler(device_type='cuda')
        else:
            scaler = GradScaler()
    else:
        scaler = None

    try:
        # Fine-tuning loop
        for epoch in range(num_epochs):
            # Training phase
            model.train()
            train_loss = 0
            optimizer.zero_grad()  # Zero gradients at the start of each epoch

            for batch_idx, (data, target) in enumerate(train_loader):
                data, target = data.to(device), target.to(device)

                # Mixed precision forward pass
                if use_amp:
                    with autocast(device_type='cuda' if DEVICE_TYPE_SUPPORTED else None):
                        if config.use_spatial_features and config.use_gnn_pre_transformer and adjacency_matrix is not None:
                            output = model(data, adjacency_matrix.to(device))
                        else:
                            output = model(data)

                        loss = criterion(output, target) / config.accumulation_steps

                    # Mixed precision backward pass
                    scaler.scale(loss).backward()

                    # Gradient accumulation - only step every accumulation_steps
                    if (batch_idx + 1) % config.accumulation_steps == 0 or (batch_idx + 1) == len(train_loader):
                        if config.gradient_clip is not None:
                            scaler.unscale_(optimizer)
                            nn.utils.clip_grad_norm_(model.parameters(), config.gradient_clip)

                        scaler.step(optimizer)
                        scaler.update()
                        optimizer.zero_grad()
                else:
                    # Standard precision training
                    if config.use_spatial_features and config.use_gnn_pre_transformer and adjacency_matrix is not None:
                        output = model(data, adjacency_matrix.to(device))
                    else:
                        output = model(data)

                    loss = criterion(output, target) / config.accumulation_steps

                    # Standard backward pass
                    loss.backward()

                    # Gradient accumulation - only step every accumulation_steps
                    if (batch_idx + 1) % config.accumulation_steps == 0 or (batch_idx + 1) == len(train_loader):
                        if config.gradient_clip is not None:
                            nn.utils.clip_grad_norm_(model.parameters(), config.gradient_clip)

                        optimizer.step()
                        optimizer.zero_grad()

                # Track loss (multiply by accumulation_steps to get actual loss)
                train_loss += loss.item() * config.accumulation_steps

            avg_train_loss = train_loss / len(train_loader)
            history['train_loss'].append(avg_train_loss)

            # Validation phase
            model.eval()
            val_loss = 0

            with torch.no_grad():
                for data, target in val_loader:
                    data, target = data.to(device), target.to(device)

                    if use_amp:
                        with autocast(device_type='cuda' if DEVICE_TYPE_SUPPORTED else None):
                            if config.use_spatial_features and config.use_gnn_pre_transformer and adjacency_matrix is not None:
                                output = model(data, adjacency_matrix.to(device))
                            else:
                                output = model(data)
                            loss = criterion(output, target)
                    else:
                        if config.use_spatial_features and config.use_gnn_pre_transformer and adjacency_matrix is not None:
                            output = model(data, adjacency_matrix.to(device))
                        else:
                            output = model(data)
                        loss = criterion(output, target)

                    val_loss += loss.item()

            avg_val_loss = val_loss / len(val_loader)
            history['val_loss'].append(avg_val_loss)

            # Update learning rate scheduler
            if scheduler is not None:
                if isinstance(scheduler, optim.lr_scheduler.ReduceLROnPlateau):
                    scheduler.step(avg_val_loss)
                else:
                    scheduler.step()

            # Print progress
            print(f'Epoch {epoch+1}/{num_epochs} | Train Loss: {avg_train_loss:.6f} | Val Loss: {avg_val_loss:.6f}')

            # Early stopping and model saving
            if avg_val_loss < best_loss:
                best_loss = avg_val_loss
                no_improve = 0

                # Save best model
                if model_dir:
                    try:
                        timestamp = get_maputo_timestamp()
                        model_path = os.path.join(model_dir, f'fine_tuned_{target_dataset_name}_{timestamp}.pth')
                        torch.save({
                            'epoch': epoch,
                            'model_state_dict': model.state_dict(),
                            'optimizer_state_dict': optimizer.state_dict(),
                            'loss': best_loss,
                            'target_dataset': target_dataset_name,
                            'config': {k: v for k, v in vars(config).items() if not k.startswith('_')}
                        }, model_path)
                        print(f"Saved fine-tuned model: {model_path}")
                    except Exception as e:
                        print(f"Warning: Could not save model: {str(e)}")
            else:
                no_improve += 1
                if no_improve >= patience:
                    print(f'Early stopping at epoch {epoch+1}')
                    break

        # Plot training history
        if results_dir:
            try:
                plt.figure(figsize=(10, 6))
                plt.plot(history['train_loss'], label='Training Loss')
                plt.plot(history['val_loss'], label='Validation Loss')
                plt.title(f'Transfer Learning to {target_dataset_name.title()} Dataset')
                plt.xlabel('Epochs')
                plt.ylabel('Loss')
                plt.legend()
                plt.grid(True, alpha=0.3)

                # Add details about freezing
                if hasattr(config, 'freeze_encoder') and hasattr(config, 'freeze_layers'):
                    plt.figtext(0.02, 0.02,
                               f"Frozen layers: {config.freeze_layers if config.freeze_encoder else 'None'}\n"
                               f"Adapters: {'Yes' if hasattr(config, 'adapter_dim') and config.adapter_dim > 0 else 'No'}",
                               fontsize=10)

                plt.tight_layout()
                plt.savefig(os.path.join(results_dir, f'transfer_learning_curve_{target_dataset_name}.png'), dpi=300)
                plt.close()
                print(f"Saved training curve to {os.path.join(results_dir, f'transfer_learning_curve_{target_dataset_name}.png')}")
            except Exception as e:
                print(f"Warning: Could not plot training history: {str(e)}")

        # Evaluate on target dataset
        print(f"\n=== Evaluating Transfer Learning on {target_dataset_name} ===")
        target_metrics = evaluate_model(
            model=model,
            dataloader=val_loader,
            criterion=criterion,
            data_scaler=data_scaler,
            device=device,
            config=config,
            adjacency_matrix=adjacency_matrix
        )[1]  # Get only the metrics tuple
        metrics['target'] = {
            'mae': target_metrics[0],
            'rmse': target_metrics[1],
            'r2': target_metrics[2],
            'mape': target_metrics[3]
        }

        # Evaluate on source dataset if provided
        if source_loader is not None and source_scaler is not None:
            print(f"\n=== Evaluating Transfer Learning on Source Dataset (METR-LA) ===")
            source_metrics = evaluate_model(
                model=model,
                dataloader=source_loader,
                criterion=criterion,
                data_scaler=source_scaler,
                device=device,
                config=config,
                adjacency_matrix=adjacency_matrix
            )[1]  # Get only the metrics tuple
            metrics['source'] = {
                'mae': source_metrics[0],
                'rmse': source_metrics[1],
                'r2': source_metrics[2],
                'mape': source_metrics[3]
            }

            # Create comparative visualization if both metrics are available
            if metrics['source'] and metrics['target'] and results_dir:
                try:
                    plt.figure(figsize=(12, 8))
                    metric_names = ['mae', 'rmse', 'r2', 'mape']
                    source_values = [metrics['source'][m] for m in metric_names]
                    target_values = [metrics['target'][m] for m in metric_names]

                    # Special handling for R² (higher is better)
                    r2_idx = metric_names.index('r2')
                    inverted_source_r2 = 1 - source_values[r2_idx]
                    inverted_target_r2 = 1 - target_values[r2_idx]
                    source_values[r2_idx] = inverted_source_r2
                    target_values[r2_idx] = inverted_target_r2
                    metric_names[r2_idx] = 'r2 (inverted)'

                    x = range(len(metric_names))
                    width = 0.35

                    plt.bar([i - width/2 for i in x], source_values, width, label='METR-LA (Source)')
                    plt.bar([i + width/2 for i in x], target_values, width, label=f'{target_dataset_name.title()} (Target)')

                    plt.xlabel('Metrics')
                    plt.ylabel('Value (Lower is Better)')
                    plt.title('Transfer Learning Performance Comparison')
                    plt.xticks(x, metric_names)
                    plt.legend()
                    plt.grid(True, alpha=0.3)

                    # Add text showing improvement/degradation percentages
                    for i, (source, target) in enumerate(zip(source_values, target_values)):
                        if metric_names[i] != 'r2 (inverted)':
                            change_pct = (target - source) / source * 100
                            color = 'green' if change_pct < 0 else 'red'
                            plt.annotate(
                                f"{change_pct:.1f}%",
                                xy=(i, max(source, target) * 1.05),
                                ha='center',
                                color=color
                            )

                    plt.tight_layout()
                    plt.savefig(os.path.join(results_dir, f'transfer_comparison_{target_dataset_name}.png'), dpi=300)
                    plt.close()
                    print(f"Saved transfer comparison to {os.path.join(results_dir, f'transfer_comparison_{target_dataset_name}.png')}")
                except Exception as e:
                    print(f"Warning: Could not create comparison visualization: {str(e)}")

        return model, history, metrics

    except Exception as e:
        print(f"\n=== Error during transfer learning: {str(e)} ===")
        import traceback
        traceback.print_exc()

        # Return the model in its current state along with partial history/metrics
        return model, history, metrics

def evaluate_model(
    model: nn.Module,
    dataloader: DataLoader,
    criterion: Callable,
    data_scaler: MinMaxScaler | StandardScaler | RobustScaler,
    device: str = 'cpu',
    config: Optional[TrainingConfig] = None,
    adjacency_matrix: Optional[torch.Tensor] = None
) -> Tuple[float, Tuple[float, float, float, float]]:
    """
    Evaluate the model and calculate metrics

    Args:
        model: Model to evaluate
        dataloader: DataLoader with evaluation data
        criterion: Loss function
        device: Device to evaluate on
        config: Training configuration
        adjacency_matrix: Optional adjacency matrix for GNN

    Returns:
        Tuple of (average loss, (MAE, RMSE, R², MAPE))
    """
    model.eval()
    total_loss = 0
    all_preds = []
    all_targets = []
    use_quantile_regression = config.use_quantile_regression if config else False
    loss_function_type = config.loss_function if config else 'mse'

    # Use mixed precision for evaluation if enabled
    use_amp = config and config.use_mixed_precision and AMP_AVAILABLE and device != 'cpu'

    with torch.no_grad():
        for data, target in dataloader:
            data, target = data.to(device), target.to(device)

            if use_amp:
                if DEVICE_TYPE_SUPPORTED:
                    with autocast(device_type='cuda'):
                        if config and config.use_spatial_features and config.use_gnn_pre_transformer:
                            output = model(data, adjacency_matrix.to(device))
                        else:
                            output = model(data)

                        if use_quantile_regression:
                            quantiles = config.quantiles
                            loss = quantile_loss(output, target, quantiles)
                        elif loss_function_type == 'hybrid':
                            loss = hybrid_loss(output, target)
                        else:
                            loss = criterion(output, target)
                else:
                    # Older PyTorch versions
                    with autocast():
                        if config and config.use_spatial_features and config.use_gnn_pre_transformer:
                            output = model(data, adjacency_matrix.to(device))
                        else:
                            output = model(data)

                        if use_quantile_regression:
                            quantiles = config.quantiles
                            loss = quantile_loss(output, target, quantiles)
                        elif loss_function_type == 'hybrid':
                            loss = hybrid_loss(output, target)
                        else:
                            loss = criterion(output, target)
            else:
                if config and config.use_spatial_features and config.use_gnn_pre_transformer:
                    output = model(data, adjacency_matrix.to(device))
                else:
                    output = model(data)

                if use_quantile_regression:
                    quantiles = config.quantiles
                    loss = quantile_loss(output, target, quantiles)
                elif loss_function_type == 'hybrid':
                    loss = hybrid_loss(output, target)
                else:
                    loss = criterion(output, target)

            total_loss += loss.item()
            all_preds.append(output.cpu().numpy())
            all_targets.append(target.cpu().numpy())

    predictions = np.concatenate(all_preds)
    actuals = np.concatenate(all_targets)

    # save_predictions_and_actuals(
    #     predictions=predictions,
    #     actuals=actuals,
    #     results_dir=config.results_dir,
    #     filename="scaled_predictions"
    # )

    # Reshape for inverse transformation
    num_samples, pred_window, num_features = predictions.shape
    predictions_2d = predictions.reshape(-1, num_features)
    actuals_2d = actuals.reshape(-1, num_features)

    # Inverse transform
    predictions_inv = data_scaler.inverse_transform(predictions_2d)
    actuals_inv = data_scaler.inverse_transform(actuals_2d)

    # Calculate metrics
    mae = mean_absolute_error(actuals_inv.ravel(), predictions_inv.ravel())
    rmse = np.sqrt(mean_squared_error(actuals_inv.ravel(), predictions_inv.ravel()))
    r2 = r2_score(actuals_inv.ravel(), predictions_inv.ravel())
    mape = robust_mape(actuals_inv.ravel(), predictions_inv.ravel())

    print(f'MAE: {mae:.2f}, RMSE: {rmse:.2f}, R²: {r2:.2f}, MAPE: {mape:.2f}%')

    return total_loss / len(dataloader), (mae, rmse, r2, mape)


def predict(
    model: nn.Module,
    dataloader: DataLoader,
    scaler: Any,
    device: str = 'cpu',
    adjacency_matrix: Optional[torch.Tensor] = None,
    config: Optional[TrainingConfig] = None
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Make predictions using the trained model and inverse transform the results

    Args:
        model: Trained model
        dataloader: DataLoader with test data
        scaler: Scaler used for normalization
        device: Device to use for prediction
        adjacency_matrix: Optional adjacency matrix for GNN

    Returns:
        Tuple of (predictions, actuals) in original scale
    """
    model.eval()
    all_preds = []
    all_targets = []

    # Use AMP for prediction if available on GPU
    use_amp = hasattr(torch.cuda, 'amp') and device != 'cpu'

    with torch.no_grad():
        for data, target in dataloader:
            data, target = data.to(device), target.to(device)

            if use_amp:
                if DEVICE_TYPE_SUPPORTED:
                    with autocast(device_type='cuda'):
                        if hasattr(dataloader.dataset, 'config') and dataloader.dataset.config.use_spatial_features and dataloader.dataset.config.use_gnn_pre_transformer:
                            output = model(data, adjacency_matrix.to(device))
                        else:
                            output = model(data)
                else:
                    # Older PyTorch versions
                    with autocast():
                        if hasattr(dataloader.dataset, 'config') and dataloader.dataset.config.use_spatial_features and dataloader.dataset.config.use_gnn_pre_transformer:
                            output = model(data, adjacency_matrix.to(device))
                        else:
                            output = model(data)
            else:
                if hasattr(dataloader.dataset, 'config') and dataloader.dataset.config.use_spatial_features and dataloader.dataset.config.use_gnn_pre_transformer:
                    output = model(data, adjacency_matrix.to(device))
                else:
                    output = model(data)

            all_preds.append(output.cpu().numpy())
            all_targets.append(target.cpu().numpy())

    predictions = np.concatenate(all_preds)
    actuals = np.concatenate(all_targets)

    # Reshape for inverse transformation
    num_samples, pred_window, num_features = predictions.shape
    predictions_2d = predictions.reshape(-1, num_features)
    actuals_2d = actuals.reshape(-1, num_features)

    # Inverse transform
    predictions_inv = scaler.inverse_transform(predictions_2d)
    actuals_inv = scaler.inverse_transform(actuals_2d)

    # save_predictions_and_actuals(
    #     predictions=predictions_inv,
    #     actuals=actuals_inv,
    #     results_dir=config.output_dir,
    #     filename="original_scale_predictions"
    # )

    return predictions_inv, actuals_inv


def evaluate_baseline_model(predictions: np.ndarray, actuals: np.ndarray) -> Tuple[float, float, float, float]:
    """
    Evaluate baseline model predictions against actuals

    Args:
        predictions: Predicted values
        actuals: Actual values

    Returns:
        Tuple of (MAE, RMSE, R², MAPE)
    """
    mae = mean_absolute_error(actuals.ravel(), predictions.ravel())
    rmse = np.sqrt(mean_squared_error(actuals.ravel(), predictions.ravel()))
    r2 = r2_score(actuals.ravel(), predictions.ravel())
    mape = robust_mape(actuals.ravel(), predictions.ravel())
    return mae, rmse, r2, mape

def train_baseline_models(
    train_data: np.ndarray,
    test_data: np.ndarray,
    config: TrainingConfig,
    device: str,
    timestamps_train: Optional[pd.DatetimeIndex] = None,
    timestamps_test: Optional[pd.DatetimeIndex] = None
) -> Dict[str, List[Tuple[float, float, float, float]]]:
    """Return empty baseline metrics - ARIMA, Exp Smoothing and LSTM baselines removed"""
    print("Baseline models (ARIMA, ExpSmoothing, LSTM) have been disabled")

    # Return empty dictionary of metrics
    return {'naive_forecast': []}

# --- Loss Functions ---
def quantile_loss(output: torch.Tensor, target: torch.Tensor, quantiles: List[float]) -> torch.Tensor:
    """
    Quantile Loss function for prediction intervals

    Args:
        output: Model output with shape [batch, pred_len, num_quantiles]
        target: Target values
        quantiles: List of quantiles

    Returns:
        Quantile loss value
    """
    losses = []
    for i, q in enumerate(quantiles):
        errors = target - output[:, :, i]
        losses.append(torch.max((q-1) * errors, q * errors).mean())
    loss = torch.sum(torch.stack(losses))
    return loss


def hybrid_loss(output: torch.Tensor, target: torch.Tensor, alpha: float = 0.5) -> torch.Tensor:
    """
    Hybrid Loss function: Weighted combination of MSE and MAE

    Args:
        output: Model output
        target: Target values
        alpha: Weight for MSE component (1-alpha for MAE)

    Returns:
        Hybrid loss value
    """
    mse_loss = nn.MSELoss()(output, target)
    mae_loss = nn.L1Loss()(output, target)
    loss = alpha * mse_loss + (1 - alpha) * mae_loss
    return loss


def robust_mape(y_true: np.ndarray, y_pred: np.ndarray, epsilon: float = 1e-8) -> float:
    """
    Robust MAPE to handle division by zero and near-zero values

    Args:
        y_true: True values
        y_pred: Predicted values
        epsilon: Small value to prevent division by zero

    Returns:
        MAPE value as percentage
    """
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    mask = y_true != 0
    if not np.any(mask):
        return np.nan  # Or 0, or another appropriate value if all y_true are zero
    y_true_masked = y_true[mask]
    y_pred_masked = y_pred[mask]
    return np.mean(np.abs((y_true_masked - y_pred_masked) / (y_true_masked + epsilon))) * 100


## Visualization Module

In [ ]:
# =============================================================================
# Visualization Module
# =============================================================================

def plot_attention_weights(
    model: nn.Module,
    seq_length: int,
    results_dir: str,
    config: Optional[TrainingConfig] = None,
    show_feature_groups: bool = True,
    include_pairwise: bool = True  # New parameter to show pairwise attention
) -> None:
    """
    Enhanced visualization of attention weights with additional insights into feature interactions.

    Args:
        model: The model containing attention weights
        seq_length: Input sequence length
        results_dir: Directory to save the visualization
        config: Training configuration object
        show_feature_groups: Whether to show feature group boundaries and labels
        include_pairwise: Whether to include pairwise attention visualizations
    """
    # Create a directory for visualizations if it doesn't exist
    vis_dir = os.path.join(results_dir, 'attention_vis')
    os.makedirs(vis_dir, exist_ok=True)

    # Main attention visualization (similar to original)
    plt.figure(figsize=(12, 10))

    # Use 4/5 of the figure for the attention plot
    plt.subplot(5, 1, (1, 4))

    if not hasattr(model, 'attention_weights') or model.attention_weights is None:
        plt.text(0.5, 0.5, "No attention weights captured", ha='center', va='center', fontsize=14)
        plt.title('Attention Weights - Not Available')
    else:
        try:
            # Get attention weights and ensure proper shape
            weights = model.attention_weights

            # Convert to numpy if it's a tensor
            if isinstance(weights, torch.Tensor):
                weights = weights.cpu().detach().numpy()

            # Handle different possible shapes
            if len(weights.shape) == 3:  # [batch, seq, seq]
                weights = weights[0] if weights.shape[0] == 1 else np.mean(weights, axis=0)
            elif len(weights.shape) > 3:  # [batch, heads, seq, seq]
                weights = np.mean(weights, axis=(0, 1))  # Average across batches and heads
            elif len(weights.shape) == 2 and weights.shape[1] == 1:  # [seq, 1] shape
                weights = np.tile(weights, (1, seq_length))

            # Create the base heatmap
            ax = sns.heatmap(weights, cmap='viridis',
                    xticklabels=range(1, weights.shape[1] + 1),
                    yticklabels=range(1, weights.shape[0] + 1))

            plt.title('Attention Weights - Last Layer', fontsize=16)
            plt.xlabel('Key Positions')
            plt.ylabel('Query Positions')

            # Add feature group boundaries and labels if requested
            if show_feature_groups and config:
                # Define feature groups based on config
                feature_groups = []

                # Traffic data features (always present)
                num_sensors = getattr(config, 'num_features', weights.shape[1])
                feature_groups.append(("Traffic Data", num_sensors))

                # Time features
                if hasattr(config, 'use_time_features') and config.use_time_features:
                    time_features = 4  # hour, dayofweek, weekofyear, month
                    feature_groups.append(("Time Features", time_features))

                # Holiday feature
                if hasattr(config, 'use_holiday_feature') and config.use_holiday_feature:
                    feature_groups.append(("Holiday", 1))

                # Weather features
                if hasattr(config, 'use_weather_feature') and config.use_weather_feature:
                    if config.weather_feature_type == 'all_features':
                        weather_features = 8  # All weather features
                    elif config.weather_feature_type == 'temperature':
                        weather_features = 1
                    elif config.weather_feature_type == 'wind':
                        weather_features = 2  # wind_speed, wind_direction
                    else:
                        weather_features = 1  # Default for other single weather features
                    feature_groups.append(("Weather", weather_features))

                # Lagged features
                if hasattr(config, 'use_lagged_features') and config.use_lagged_features:
                    lagged_features = getattr(config, 'num_lags', 24)
                    feature_groups.append(("Lagged Data", lagged_features))

                # Spatial features
                if hasattr(config, 'use_spatial_features') and config.use_spatial_features:
                    spatial_features = getattr(config, 'spatial_feature_dim', 16)
                    feature_groups.append(("Spatial", spatial_features))

                # Add group boundaries and labels
                colors = ['red', 'blue', 'green', 'purple', 'orange', 'cyan']

                # Add vertical lines and labels (for keys)
                current_pos = 0
                color_idx = 0
                for group_name, group_size in feature_groups:
                    if current_pos + group_size <= weights.shape[1]:  # Check boundaries
                        # Add vertical line
                        plt.axvline(x=current_pos, color=colors[color_idx], linestyle='-', linewidth=1.5)

                        # Add group label above the heatmap
                        plt.text(current_pos + group_size/2, -0.5, group_name,
                                horizontalalignment='center', size='small',
                                color=colors[color_idx], weight='bold')

                        current_pos += group_size
                        color_idx = (color_idx + 1) % len(colors)

                # Add horizontal lines and labels (for queries)
                current_pos = 0
                color_idx = 0
                for group_name, group_size in feature_groups:
                    if current_pos + group_size <= weights.shape[0]:  # Check boundaries
                        # Add horizontal line
                        plt.axhline(y=current_pos, color=colors[color_idx], linestyle='-', linewidth=1.5)

                        # Add group label to the left of the heatmap
                        plt.text(-0.5, current_pos + group_size/2, group_name,
                                verticalalignment='center', horizontalalignment='right',
                                size='small', color=colors[color_idx], weight='bold')

                        current_pos += group_size
                        color_idx = (color_idx + 1) % len(colors)
        except Exception as e:
            # Provide error information in the plot
            plt.clf()
            plt.text(0.5, 0.5, f"Error plotting attention weights: {str(e)}\n"
                               f"Shape: {getattr(model.attention_weights, 'shape', 'unknown')}",
                     ha='center', va='center', wrap=True)
            plt.title('Attention Visualization Error')

    # Add model details at the bottom 1/5 of the figure
    model_details = ""
    if config:
        model_details = (
            f"Model: Transformer | Layers: {config.num_layers} | Heads: {config.num_heads} | "
            f"Hidden Dim: {config.hidden_dim} | Seq Length: {seq_length} | "
            f"Prediction Length: {config.pred_length} | Dropout: {config.dropout} | "
            f"Generated: {get_maputo_timestamp()}"
        )

    plt.subplot(5, 1, 5)
    plt.axis('off')
    plt.text(0.01, 0.5, model_details, wrap=True, fontsize=9)

    # Add explanation of attention if showing feature groups
    if show_feature_groups:
        plt.figtext(0.5, 0.01,
                    "Colored lines indicate feature group boundaries.\n"
                    "Brighter colors in the heatmap show stronger attention weights between features.",
                    ha='center', fontsize=9, bbox=dict(facecolor='lightyellow', alpha=0.5))

    # Save with timestamp
    timestamp = get_maputo_timestamp()
    plt.tight_layout()
    plt.savefig(os.path.join(vis_dir, f'attention_heatmap_{timestamp}.png'), dpi=300)
    plt.close()

    # NEW: Feature Importance Visualization
    if hasattr(model, 'feature_attention') and hasattr(model.feature_attention, 'feature_importances'):
        feature_importances = model.feature_attention.feature_importances
        if feature_importances:
            plt.figure(figsize=(10, 6))

            feature_names = list(feature_importances.keys())
            importance_values = list(feature_importances.values())

            # Create bar chart
            bars = plt.bar(feature_names, importance_values,
                          color=plt.cm.viridis(np.linspace(0, 1, len(feature_names))))

            # Add values on bars
            for bar, value in zip(bars, importance_values):
                plt.text(bar.get_x() + bar.get_width()/2,
                        bar.get_height() + 0.01,
                        f'{value:.4f}',
                        ha='center', va='bottom')

            plt.title('Feature Group Importance Weights', fontsize=16)
            plt.ylabel('Importance Weight')
            plt.ylim(0, max(importance_values) * 1.2)  # Add space for text
            plt.grid(axis='y', alpha=0.3)

            # Save visualization
            plt.tight_layout()
            plt.savefig(os.path.join(vis_dir, f'feature_importance_{timestamp}.png'), dpi=300)
            plt.close()

    # NEW: Pairwise Attention Visualization
    if include_pairwise and hasattr(model, 'feature_attention') and hasattr(model.feature_attention, 'pairwise_weights'):
        pairwise_weights = model.feature_attention.pairwise_weights
        if pairwise_weights:
            # Create a grid of pairwise attention visualizations
            num_pairs = len(pairwise_weights)
            if num_pairs > 0:
                # Determine grid size
                grid_size = int(np.ceil(np.sqrt(num_pairs)))

                plt.figure(figsize=(15, 15))

                # Plot each pairwise attention
                for i, (pair_name, weights) in enumerate(pairwise_weights.items()):
                    plt.subplot(grid_size, grid_size, i+1)

                    # Convert tensor to numpy if needed
                    if isinstance(weights, torch.Tensor):
                        weights = weights.cpu().detach().numpy()

                    # Average across batch/heads if needed
                    if len(weights.shape) == 4:  # [batch, heads, seq, seq]
                        weights = weights.mean(axis=(0, 1))
                    elif len(weights.shape) == 3:  # [batch, seq, seq]
                        weights = weights.mean(axis=0)

                    # Create heatmap
                    sns.heatmap(weights, cmap='viridis')
                    plt.title(pair_name, fontsize=10)
                    plt.xticks([])
                    plt.yticks([])

                plt.tight_layout()
                plt.savefig(os.path.join(vis_dir, f'pairwise_attention_{timestamp}.png'), dpi=300)
                plt.close()

    # NEW: Feature Gate Visualization
    if hasattr(model, 'feature_attention') and hasattr(model.feature_attention, 'gate_values'):
        gate_values = model.feature_attention.gate_values
        if gate_values:
            plt.figure(figsize=(10, 6))

            gate_names = list(gate_values.keys())
            gate_values_list = list(gate_values.values())

            # Create bar chart
            bars = plt.bar(gate_names, gate_values_list,
                          color=plt.cm.plasma(np.linspace(0, 1, len(gate_names))))

            # Add values on bars
            for bar, value in zip(bars, gate_values_list):
                plt.text(bar.get_x() + bar.get_width()/2,
                        bar.get_height() + 0.01,
                        f'{value:.4f}',
                        ha='center', va='bottom')

            plt.title('Feature Gate Activation Values', fontsize=16)
            plt.ylabel('Average Gate Value (0-1)')
            plt.ylim(0, max(max(gate_values_list) * 1.2, 0.1))  # Add space for text
            plt.grid(axis='y', alpha=0.3)

            # Save visualization
            plt.tight_layout()
            plt.savefig(os.path.join(vis_dir, f'feature_gates_{timestamp}.png'), dpi=300)
            plt.close()

    print(f"Attention visualizations saved to {vis_dir}")

def visualize_decoder_attention(
    model: nn.Module,
    test_loader: DataLoader,
    results_dir: str,
    config: Optional[TrainingConfig] = None,
    max_samples: int = 3,
    adjacency_matrix: Optional[torch.Tensor] = None
) -> None:
    """
    Visualize attention patterns in the transformer decoder

    Args:
        model: The model with transformer decoder
        test_loader: DataLoader with test data
        results_dir: Directory to save visualizations
        config: Training configuration
        max_samples: Maximum number of samples to visualize
        adjacency_matrix: Optional adjacency matrix for GNN
    """
    # Check if model has transformer decoder
    if not (hasattr(model, 'decoder_type') and
            getattr(model, 'decoder_type', None) == 'transformer' and
            hasattr(model, 'transformer_decoder')):
        print("Model does not have transformer decoder. Skipping decoder attention visualization.")
        return

    # Create directory for decoder attention visualizations
    decoder_vis_dir = os.path.join(results_dir, 'decoder_attention')
    os.makedirs(decoder_vis_dir, exist_ok=True)

    # Set model to evaluation mode
    model.eval()
    device = next(model.parameters()).device

    # Get a few samples for visualization
    sample_count = 0
    timestamp = get_maputo_timestamp()

    with torch.no_grad():
        for data, target in test_loader:
            if sample_count >= max_samples:
                break

            # Process batch
            if isinstance(data, dict):
                data = {k: v.to(device) for k, v in data.items()}
            else:
                data = data.to(device)
            target = target.to(device)

            # Get encoder outputs
            memory = model.encode(data, adjacency_matrix)

            # Generate autoregressive predictions
            outputs = model.generate(
                memory=memory,
                steps=getattr(config, 'pred_length', model.pred_len),
                temperature=1.0
            )

            # Get attention weights from decoder
            self_attn_weights = model.transformer_decoder.self_attn_weights
            cross_attn_weights = model.transformer_decoder.cross_attn_weights

            if not self_attn_weights or not cross_attn_weights:
                print("No decoder attention weights captured. Skipping visualization.")
                continue

            # Create visualization for this sample
            sample_count += 1

            # Plot self-attention weights
            plt.figure(figsize=(15, 5 * len(self_attn_weights)))
            for i, attn_weights in enumerate(self_attn_weights):
                # Convert to numpy if tensor
                if isinstance(attn_weights, torch.Tensor):
                    attn_weights = attn_weights.cpu().numpy()

                # Average across batch and heads if needed
                if len(attn_weights.shape) > 2:
                    if len(attn_weights.shape) == 4:  # [batch, heads, seq, seq]
                        attn_weights = attn_weights.mean(axis=(0, 1))
                    elif len(attn_weights.shape) == 3:  # [batch, seq, seq]
                        attn_weights = attn_weights.mean(axis=0)

                plt.subplot(len(self_attn_weights), 1, i+1)
                sns.heatmap(attn_weights, cmap='viridis')
                plt.title(f'Layer {i+1} Self-Attention')
                plt.xlabel('Key Position')
                plt.ylabel('Query Position')

            plt.tight_layout()
            plt.savefig(os.path.join(decoder_vis_dir, f'decoder_self_attn_sample{sample_count}_{timestamp}.png'), dpi=300)
            plt.close()

            # Plot cross-attention weights
            plt.figure(figsize=(15, 5 * len(cross_attn_weights)))
            for i, attn_weights in enumerate(cross_attn_weights):
                # Convert to numpy if tensor
                if isinstance(attn_weights, torch.Tensor):
                    attn_weights = attn_weights.cpu().numpy()

                # Average across batch and heads if needed
                if len(attn_weights.shape) > 2:
                    if len(attn_weights.shape) == 4:  # [batch, heads, seq, seq]
                        attn_weights = attn_weights.mean(axis=(0, 1))
                    elif len(attn_weights.shape) == 3:  # [batch, seq, seq]
                        attn_weights = attn_weights.mean(axis=0)

                plt.subplot(len(cross_attn_weights), 1, i+1)
                sns.heatmap(attn_weights, cmap='viridis')
                plt.title(f'Layer {i+1} Cross-Attention (Decoder → Encoder)')
                plt.xlabel('Encoder Position')
                plt.ylabel('Decoder Position')

            plt.tight_layout()
            plt.savefig(os.path.join(decoder_vis_dir, f'decoder_cross_attn_sample{sample_count}_{timestamp}.png'), dpi=300)
            plt.close()

            # Visualize predictions vs actuals for this sample
            pred_window = outputs.shape[1]

            plt.figure(figsize=(12, 6))
            # Plot for a random sensor
            num_sensors = outputs.shape[2]
            random_sensor = random.randint(0, num_sensors-1)

            # Get predictions and ground truth
            pred = outputs[0, :, random_sensor].cpu().numpy()
            true = target[0, :pred_window, random_sensor].cpu().numpy()

            # Inverse transform if scaler is provided
            if data_scaler is not None:
                # Create arrays with the proper shape for the scaler
                pred_array = np.zeros((1, num_sensors))
                true_array = np.zeros((1, num_sensors))

                # Set the values for our selected sensor
                pred_array[0, random_sensor] = pred[0]
                true_array[0, random_sensor] = true[0]

                # Inverse transform
                pred_orig = data_scaler.inverse_transform(pred_array)[0, random_sensor]
                true_orig = data_scaler.inverse_transform(true_array)[0, random_sensor]
            else:
                pred_orig = pred
                true_orig = true

            # Plot
            plt.plot(range(pred_window), pred, 'r-', label='Predicted')
            plt.plot(range(min(pred_window, len(true))), true[:pred_window], 'b-', label='Actual')
            plt.title(f'Generative Predictions for Sensor {random_sensor}')
            plt.xlabel('Time Step')
            plt.ylabel('Value')
            plt.legend()
            plt.grid(True, alpha=0.3)

            plt.tight_layout()
            plt.savefig(os.path.join(decoder_vis_dir, f'decoder_predictions_sample{sample_count}_sensor{random_sensor}_{timestamp}.png'), dpi=300)
            plt.close()

    if sample_count > 0:
        print(f"Decoder attention visualizations saved to {decoder_vis_dir}")
    else:
        print("No decoder attention visualizations were generated")

def plot_layer_progression(
    model: nn.Module,
    input_batch: torch.Tensor,
    results_dir: str,
    config: Optional[TrainingConfig] = None,
    show_feature_groups: bool = True,
    adjacency_matrix: Optional[torch.Tensor] = None  # Add adjacency_matrix parameter
) -> None:
    """
    Visualize how attention patterns evolve across transformer layers.

    Args:
        model: The trained transformer model
        input_batch: A batch of input data [batch_size, seq_len, feature_dim]
        results_dir: Directory to save the visualization
        config: Training configuration (for feature group labeling)
        show_feature_groups: Whether to show feature group boundaries
        adjacency_matrix: Adjacency matrix for GNN (required if use_gnn_pre_transformer=True)
    """
    device = next(model.parameters()).device

    # Ensure model is in eval mode
    model.eval()

    # Check if we need adjacency matrix but don't have it
    if (hasattr(config, 'use_gnn_pre_transformer') and config.use_gnn_pre_transformer and
        adjacency_matrix is None):
        print("Warning: Model requires adjacency matrix but none provided. Attempting to create or load one.")
        try:
            # Try to get adjacency matrix from dataset or another source
            if hasattr(config, 'input_dir'):
                adj_path = os.path.join(config.input_dir, 'adj_METR-LA.pkl')
                if os.path.exists(adj_path):
                    # Load adjacency matrix using appropriate function
                    with open(adj_path, 'rb') as f:
                        try:
                            adj_data = pickle.load(f, encoding='latin1')
                        except:
                            # Fall back to default encoding if latin1 fails
                            f.seek(0)  # Reset file pointer
                            adj_data = pickle.load(f)

                        if isinstance(adj_data, list) and len(adj_data) >= 3:
                            adj_matrix = adj_data[2]
                            adj_matrix = normalize_adj(adj_matrix)  # Assuming normalize_adj is defined
                            adjacency_matrix = torch.tensor(adj_matrix, dtype=torch.float32).to(device)
                            print("Loaded adjacency matrix from file.")
        except Exception as e:
            print(f"Failed to load adjacency matrix: {e}")
            print("Cannot generate layer-wise visualization without adjacency matrix.")
            return

        # If still no adjacency matrix, we can't proceed
        if adjacency_matrix is None:
            print("Could not load or create adjacency matrix. Visualization aborted.")
            return

    # Prepare to capture attention weights from each layer
    attention_by_layer = []

    # Create hook functions to capture attention weights
    hooks = []

    def get_attention_hook(layer_idx):
        def hook(module, input, output):
            # Check if this module has attention weights
            if hasattr(module, 'attn_weights') and module.attn_weights is not None:
                # Store (layer_idx, attention_weights)
                attention_weights = module.attn_weights
                if isinstance(attention_weights, torch.Tensor):
                    attention_weights = attention_weights.detach().cpu()
                attention_by_layer.append((layer_idx, attention_weights))
        return hook

    # Register hooks for each transformer layer
    for i, layer in enumerate(model.transformer):
        hooks.append(layer.register_forward_hook(get_attention_hook(i)))

    # Run a forward pass to capture attention weights
    try:
        with torch.no_grad():
            if hasattr(config, 'use_gnn_pre_transformer') and config.use_gnn_pre_transformer and adjacency_matrix is not None:
                _ = model(input_batch.to(device), adjacency_matrix.to(device))
            else:
                _ = model(input_batch.to(device))
    except Exception as e:
        print(f"Error during model forward pass: {e}")
        # Remove hooks to prevent memory leaks
        for hook in hooks:
            hook.remove()
        return

    # Remove hooks to prevent memory leaks
    for hook in hooks:
        hook.remove()

    # Check if we captured any attention weights
    if not attention_by_layer:
        print("No attention weights were captured. Check model implementation.")
        return

    # Sort by layer index in case hooks fired out of order
    attention_by_layer.sort(key=lambda x: x[0])

    # Process attention weights for visualization
    processed_attentions = []
    for layer_idx, attn in attention_by_layer:
        # Handle different attention weight shapes
        if len(attn.shape) == 4:  # [batch, heads, seq, seq]
            # Average across batch and heads
            attn_avg = attn.mean(dim=(0, 1)).numpy()
        elif len(attn.shape) == 3:  # [batch, seq, seq]
            # Average across batch
            attn_avg = attn.mean(dim=0).numpy()
        else:
            # Already 2D [seq, seq]
            attn_avg = attn.numpy() if isinstance(attn, torch.Tensor) else attn

        processed_attentions.append((layer_idx, attn_avg))

    # Create visualization
    n_layers = len(processed_attentions)
    fig = plt.figure(figsize=(min(20, n_layers * 4), 10))

    # Create a grid layout: layers in row 1, attention diff in row 2
    gs = GridSpec(2, n_layers, height_ratios=[1, 1])

    # Define feature groups if requested
    feature_groups = []
    if show_feature_groups and config:
        # Traffic data features (always present)
        num_sensors = getattr(config, 'num_features', processed_attentions[0][1].shape[0])
        feature_groups.append(("Traffic Data", num_sensors))

        # Time features
        if hasattr(config, 'use_time_features') and config.use_time_features:
            time_features = 4  # hour, dayofweek, weekofyear, month
            feature_groups.append(("Time Features", time_features))

        # Holiday feature
        if hasattr(config, 'use_holiday_feature') and config.use_holiday_feature:
            feature_groups.append(("Holiday", 1))

        # Weather features
        if hasattr(config, 'use_weather_feature') and config.use_weather_feature:
            if config.weather_feature_type == 'all_features':
                weather_features = 8  # All weather features
            elif config.weather_feature_type == 'temperature':
                weather_features = 1
            elif config.weather_feature_type == 'wind':
                weather_features = 2  # wind_speed, wind_direction
            else:
                weather_features = 1  # Default for other single weather features
            feature_groups.append(("Weather", weather_features))

        # Lagged features
        if hasattr(config, 'use_lagged_features') and config.use_lagged_features:
            lagged_features = getattr(config, 'num_lags', 24)
            feature_groups.append(("Lagged Data", lagged_features))

        # Spatial features
        if hasattr(config, 'use_spatial_features') and config.use_spatial_features:
            spatial_features = getattr(config, 'spatial_feature_dim', 16)
            feature_groups.append(("Spatial", spatial_features))

    # Define colors for feature group boundaries
    boundary_colors = ['red', 'blue', 'green', 'purple', 'orange', 'cyan']

    # Create plots for each layer's attention
    for i, (layer_idx, attn) in enumerate(processed_attentions):
        # Plot attention heatmap
        ax = fig.add_subplot(gs[0, i])
        im = ax.imshow(attn, cmap='viridis', interpolation='none')
        ax.set_title(f"Layer {layer_idx+1}")

        # Remove ticks for cleaner visualization
        ax.set_xticks([])
        ax.set_yticks([])

        # Add feature group boundaries if requested
        if show_feature_groups and feature_groups:
            current_pos = 0
            color_idx = 0
            for group_name, group_size in feature_groups:
                if current_pos + group_size <= attn.shape[0]:  # Check boundaries
                    # Add vertical and horizontal lines
                    ax.axvline(x=current_pos, color=boundary_colors[color_idx], linestyle='-', linewidth=1.5)
                    ax.axhline(y=current_pos, color=boundary_colors[color_idx], linestyle='-', linewidth=1.5)

                    # Add small label
                    if i == 0:  # Only add labels on the first plot
                        ax.text(-0.5, current_pos + group_size/2, group_name,
                                rotation=90, verticalalignment='center', fontsize=8,
                                transform=ax.transData, color=boundary_colors[color_idx])

                    current_pos += group_size
                    color_idx = (color_idx + 1) % len(boundary_colors)

    # For second row, show attention differences between consecutive layers
    for i in range(n_layers - 1):
        ax = fig.add_subplot(gs[1, i])

        # Get consecutive attention patterns
        _, curr_attn = processed_attentions[i]
        _, next_attn = processed_attentions[i+1]

        # Calculate difference (how attention changes from this layer to the next)
        diff = next_attn - curr_attn

        # Plot difference with diverging colormap (blue-white-red)
        # Blue = negative change, Red = positive change
        vmax = max(abs(diff.min()), abs(diff.max()))
        im = ax.imshow(diff, cmap='coolwarm', vmin=-vmax, vmax=vmax, interpolation='none')
        ax.set_title(f"Change: Layer {i+1} → {i+2}")

        # Remove ticks for cleaner visualization
        ax.set_xticks([])
        ax.set_yticks([])

        # Add feature group boundaries if requested
        if show_feature_groups and feature_groups:
            current_pos = 0
            color_idx = 0
            for group_name, group_size in feature_groups:
                if current_pos + group_size <= diff.shape[0]:  # Check boundaries
                    # Add vertical and horizontal lines
                    ax.axvline(x=current_pos, color=boundary_colors[color_idx], linestyle='-', linewidth=1.5)
                    ax.axhline(y=current_pos, color=boundary_colors[color_idx], linestyle='-', linewidth=1.5)
                    current_pos += group_size
                    color_idx = (color_idx + 1) % len(boundary_colors)

    # Add last subplot as a legend explaining differences
    if n_layers > 1:
        ax = fig.add_subplot(gs[1, n_layers-1])
        ax.text(0.5, 0.5, "Attention Changes:\n\n" +
                "Blue = Decreasing Attention\n" +
                "White = No Change\n" +
                "Red = Increasing Attention",
                ha='center', va='center', fontsize=10)
        ax.axis('off')

    # Add overall title and adjust layout
    plt.suptitle("Evolution of Attention Patterns Across Transformer Layers", fontsize=16, y=0.98)
    plt.tight_layout(rect=[0, 0, 1, 0.96])  # Leave space for suptitle

    # Add model details as a footer
    if config:
        model_details = (
            f"Model: Transformer | Layers: {config.num_layers} | Heads: {config.num_heads} | "
            f"Hidden Dim: {config.hidden_dim} | Generated: {get_maputo_timestamp()}"
        )
        plt.figtext(0.5, 0.01, model_details, ha='center', fontsize=9)

    # Save visualization
    timestamp = get_maputo_timestamp()
    plt.savefig(os.path.join(results_dir, f'attention_layer_progression_{timestamp}.png'), dpi=300)
    plt.close()

    print(f"Layer-wise attention progression visualization saved to: {os.path.join(results_dir, f'attention_layer_progression_{timestamp}.png')}")


def visualize_feature_attribution(
    model: nn.Module,
    test_loader: DataLoader,
    config: TrainingConfig,
    results_dir: str,
    adjacency_matrix: Optional[torch.Tensor] = None,  # Add adjacency matrix parameter
    n_samples: int = 20,
    sensor_idx: int = 0,
) -> None:
    """
    Visualizes which input features have the most influence on the model's predictions
    using gradient-based feature attribution.

    Args:
        model: The trained model
        test_loader: DataLoader with test data
        config: Training configuration containing feature information
        results_dir: Directory to save results
        adjacency_matrix: Adjacency matrix for GNN (required if model uses GNN pre-transformer)
        n_samples: Number of samples to use for attribution (more is more stable)
        sensor_idx: Index of the sensor to analyze attributions for
    """
    device = next(model.parameters()).device
    model.eval()

    # Check if the model uses GNN pre-transformer
    uses_gnn = hasattr(config, 'use_gnn_pre_transformer') and config.use_gnn_pre_transformer

    # Ensure adjacency matrix is available if needed
    if uses_gnn and adjacency_matrix is None:
        # Try to get adjacency matrix from the test dataset
        if hasattr(test_loader.dataset, 'get_adjacency_matrix'):
            try:
                adjacency_matrix = test_loader.dataset.get_adjacency_matrix()
                print("Retrieved adjacency matrix from dataset")
            except Exception as e:
                print(f"Error retrieving adjacency matrix: {e}")
                print("Attempting to continue without adjacency matrix")

        # If still None, warn and return
        if adjacency_matrix is None:
            print("ERROR: Model requires adjacency matrix for GNN but none was provided.")
            print("Please pass adjacency_matrix parameter or ensure dataset has get_adjacency_matrix method.")
            return

    # Ensure adjacency matrix is on the correct device
    if uses_gnn and adjacency_matrix is not None:
        adjacency_matrix = adjacency_matrix.to(device)

    # Create a mapping of feature indices to feature names
    feature_names = create_feature_mapping(config)

    # Dictionary to store attribution scores
    attribution_scores = {}
    n_processed = 0

    for data, target in test_loader:
        if n_processed >= n_samples:
            break

        data, target = data.to(device), target.to(device)

        # Enable gradient calculation for inputs
        data.requires_grad = True

        # Forward pass (with adjacency matrix if needed)
        if uses_gnn:
            output = model(data, adjacency_matrix)
        else:
            output = model(data)

        # We'll focus on the prediction for the first timestep and specified sensor
        prediction = output[:, 0, sensor_idx].mean()  # Mean across batch

        # Backward pass to get gradients
        prediction.backward()

        # Get the gradients with respect to inputs
        input_gradients = data.grad.abs().mean(dim=0)  # Average over batch

        # Aggregate gradients across sequence dimension to get per-feature attribution
        feature_attributions = input_gradients.mean(dim=0).cpu().numpy()

        # Update running attributions
        for i, attribution in enumerate(feature_attributions):
            if i in attribution_scores:
                attribution_scores[i] += attribution
            else:
                attribution_scores[i] = attribution

        # Reset gradients for next iteration
        data.grad = None
        model.zero_grad()

        n_processed += data.shape[0]

    # Normalize attribution scores
    for i in attribution_scores:
        attribution_scores[i] /= n_processed

    # Create feature groups based on configuration
    feature_groups = []

    # Define the starting index for each feature group
    start_idx = 0

    # Traffic data features (always present)
    num_sensors = getattr(config, 'num_features', 207)  # Default to 207 for METR-LA
    feature_groups.append(("Traffic Data", start_idx, num_sensors))
    start_idx += num_sensors

    # Time features
    if hasattr(config, 'use_time_features') and config.use_time_features:
        time_features = 4  # hour, dayofweek, weekofyear, month
        feature_groups.append(("Time Features", start_idx, time_features))
        start_idx += time_features

    # Holiday feature
    if hasattr(config, 'use_holiday_feature') and config.use_holiday_feature:
        feature_groups.append(("Holiday", start_idx, 1))
        start_idx += 1

    # Weather features
    if hasattr(config, 'use_weather_feature') and config.use_weather_feature:
        if hasattr(config, 'weather_feature_type'):
            if config.weather_feature_type == 'all_features':
                weather_features = 8  # All weather features
            elif config.weather_feature_type == 'temperature':
                weather_features = 1
            elif config.weather_feature_type == 'wind':
                weather_features = 2  # wind_speed, wind_direction
            else:
                weather_features = 1  # Default for other single weather features
        else:
            weather_features = 8  # Default to all features if not specified

        feature_groups.append(("Weather", start_idx, weather_features))
        start_idx += weather_features

    # Lagged features
    if hasattr(config, 'use_lagged_features') and config.use_lagged_features:
        lagged_features = getattr(config, 'num_lags', 24)
        feature_groups.append(("Lagged Data", start_idx, lagged_features))
        start_idx += lagged_features

    # Spatial features
    if hasattr(config, 'use_spatial_features') and config.use_spatial_features:
        spatial_features = getattr(config, 'spatial_feature_dim', 16)
        feature_groups.append(("Spatial", start_idx, spatial_features))
        start_idx += spatial_features

    # Create visualization
    plt.figure(figsize=(15, 10))

    # Plot 1: Overall feature attribution by group
    plt.subplot(2, 1, 1)
    group_names = []
    group_scores = []

    for group_name, start, count in feature_groups:
        # Calculate average attribution for this group
        group_attribution = 0
        for i in range(start, start + count):
            if i in attribution_scores:
                group_attribution += attribution_scores[i]

        group_attribution /= count if count > 0 else 1
        group_names.append(group_name)
        group_scores.append(group_attribution)

    # Create bar chart of group attributions
    colors = plt.cm.viridis(np.linspace(0, 1, len(group_names)))
    bars = plt.bar(group_names, group_scores, color=colors)

    # Add attribution values on top of bars
    for bar, score in zip(bars, group_scores):
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., height + 0.002,
                f'{score:.4f}', ha='center', va='bottom', fontsize=10)

    plt.title(f'Feature Group Attribution for Sensor {sensor_idx}', fontsize=14)
    plt.ylabel('Attribution Score (Higher = More Important)')
    plt.grid(axis='y', alpha=0.3)

    # Plot 2: Detailed attributions within each group
    plt.subplot(2, 1, 2)

    # Create subplots for each feature group
    fig, axes = plt.subplots(len(feature_groups), 1, figsize=(15, 4 * len(feature_groups)))

    for idx, (group_name, start, count) in enumerate(feature_groups):
        ax = axes[idx] if len(feature_groups) > 1 else axes

        # Get attributions for this group
        indices = range(start, start + count)
        scores = [attribution_scores.get(i, 0) for i in indices]

        # Get feature names
        labels = [feature_names.get(i, f"Feature {i}") for i in indices]

        # Sort by attribution score for better visualization
        sorted_data = sorted(zip(labels, scores), key=lambda x: x[1], reverse=True)
        sorted_labels, sorted_scores = zip(*sorted_data) if sorted_data else ([], [])

        # Create bar chart
        bars = ax.bar(sorted_labels, sorted_scores, color=colors[idx])

        # Add values on top of bars for significant features
        for bar, score in zip(bars, sorted_scores):
            if score > max(sorted_scores) * 0.1:  # Only label significant features
                height = bar.get_height()
                ax.text(bar.get_x() + bar.get_width()/2., height + 0.001,
                        f'{score:.4f}', ha='center', va='bottom', fontsize=8, rotation=45)

        ax.set_title(f'{group_name} Feature Attribution', fontsize=12)
        ax.set_ylabel('Attribution Score')
        ax.grid(axis='y', alpha=0.3)

        # Rotate x-labels for readability
        ax.set_xticklabels(sorted_labels, rotation=90)

    plt.tight_layout()

    # Save the visualizations
    timestamp = get_maputo_timestamp()

    # Save group overview
    plt.figure(1)
    plt.tight_layout()
    plt.savefig(os.path.join(results_dir, f'feature_attribution_groups_sensor{sensor_idx}_{timestamp}.png'), dpi=300)

    # Save detailed view
    plt.figure(2)
    plt.tight_layout()
    plt.savefig(os.path.join(results_dir, f'feature_attribution_detailed_sensor{sensor_idx}_{timestamp}.png'), dpi=300)

    plt.close('all')

    print(f"Feature attribution visualizations saved to {results_dir}")

    # Also save the raw attribution scores as CSV for further analysis
    attribution_df = pd.DataFrame({
        'Feature_Index': list(attribution_scores.keys()),
        'Feature_Name': [feature_names.get(i, f"Feature {i}") for i in attribution_scores.keys()],
        'Attribution_Score': list(attribution_scores.values())
    })
    attribution_df.to_csv(os.path.join(results_dir, f'feature_attribution_scores_sensor{sensor_idx}_{timestamp}.csv'), index=False)


def visualize_prediction_explanation(
    model: nn.Module,
    test_loader: DataLoader,
    data_scaler: Any,
    config: TrainingConfig,
    results_dir: str,
    adjacency_matrix: Optional[torch.Tensor] = None,  # Added adjacency_matrix parameter
    sensor_idx: int = 0,
    pred_timestep: int = 0,
    batch_idx: int = 0,
    num_examples: int = 1
):
    """
    Visualize which input features and timesteps most influence specific predictions.

    Args:
        model: The trained Traffic Transformer model
        test_loader: DataLoader with test data
        data_scaler: Scaler used to normalize data
        config: Training configuration object
        results_dir: Directory to save visualizations
        adjacency_matrix: Adjacency matrix for GNN (required if use_gnn_pre_transformer=True)
        sensor_idx: Index of the sensor to analyze (default: 0)
        pred_timestep: Index of prediction timestep to analyze (default: 0 = first prediction step)
        batch_idx: Which batch item to analyze (default: 0 = first item in batch)
        num_examples: Number of examples to generate (with different sensors/timesteps)
    """
    device = next(model.parameters()).device
    model.eval()

    # Check if we need adjacency matrix
    needs_adjacency = False
    if hasattr(config, 'use_gnn_pre_transformer') and config.use_gnn_pre_transformer:
        needs_adjacency = True
        # Ensure we have adjacency matrix if needed
        if adjacency_matrix is None:
            print("Model requires adjacency matrix but none provided. Attempting to retrieve from dataset...")

            # Try to get adjacency matrix from dataset
            if hasattr(test_loader.dataset, 'get_adjacency_matrix'):
                try:
                    adjacency_matrix = test_loader.dataset.get_adjacency_matrix()
                    print("Successfully retrieved adjacency matrix from dataset.")
                except Exception as e:
                    print(f"Error retrieving adjacency matrix: {e}")
                    print("Cannot proceed with prediction explanation. Aborting.")
                    return
            else:
                print("Cannot find adjacency matrix and dataset doesn't provide one.")
                print("Please provide adjacency_matrix parameter or disable use_gnn_pre_transformer.")
                return

        # Ensure adjacency matrix is on the correct device
        adjacency_matrix = adjacency_matrix.to(device)

    # Get a batch of data
    for i, (data, target) in enumerate(test_loader):
        if i == 0:  # Just use the first batch
            break

    data, target = data.to(device), target.to(device)

    # Get the correct feature dimensions
    # This is the number of output features (typically number of sensors)
    output_dim = target.shape[2]

    # Check scaler dimensions to ensure compatibility
    scaler_n_features = data_scaler.n_features_in_ if hasattr(data_scaler, 'n_features_in_') else output_dim

    print(f"Scaler trained on {scaler_n_features} features")
    print(f"Target has {output_dim} features")

    # Define feature groups based on config for labeling
    feature_groups = []
    feature_names = []

    # Traffic data features (always present)
    num_sensors = output_dim  # Use the actual output dimension
    feature_groups.append(("Traffic Data", num_sensors))
    feature_names.extend([f"Sensor {i+1}" for i in range(num_sensors)])

    # Time features
    if hasattr(config, 'use_time_features') and config.use_time_features:
        feature_groups.append(("Time", 4))  # hour, dayofweek, weekofyear, month
        feature_names.extend(["Hour", "Day of Week", "Week of Year", "Month"])

    # Holiday feature
    if hasattr(config, 'use_holiday_feature') and config.use_holiday_feature:
        feature_groups.append(("Holiday", 1))
        feature_names.append("Holiday")

    # Weather features
    if hasattr(config, 'use_weather_feature') and config.use_weather_feature:
        if config.weather_feature_type == 'all_features':
            feature_groups.append(("Weather", 8))
            feature_names.extend(["Temp", "Weather", "Visibility", "Wind Speed",
                                "Wind Dir", "Humidity", "Dew Point", "Cloud Cover"])
        elif config.weather_feature_type == 'temperature':
            feature_groups.append(("Weather", 1))
            feature_names.append("Temperature")
        elif config.weather_feature_type == 'wind':
            feature_groups.append(("Weather", 2))
            feature_names.extend(["Wind Speed", "Wind Direction"])
        else:
            feature_groups.append(("Weather", 1))
            feature_names.append(config.weather_feature_type)

    # Generate multiple examples if requested
    for example in range(num_examples):
        # For multiple examples, analyze different sensors/timesteps
        if num_examples > 1:
            curr_sensor = (sensor_idx + example) % output_dim
            curr_timestep = (pred_timestep + example) % config.pred_length
        else:
            curr_sensor = sensor_idx
            curr_timestep = pred_timestep

        # Create a clone of the data that requires gradients
        data_requires_grad = data.clone().detach().requires_grad_(True)

        # Forward pass - with adjacency matrix if needed
        try:
            if needs_adjacency:
                output = model(data_requires_grad, adjacency_matrix)
            else:
                output = model(data_requires_grad)

            # Select the specific prediction to explain (single value)
            specific_prediction = output[batch_idx, curr_timestep, curr_sensor]

            # Backward pass to get gradients w.r.t inputs
            model.zero_grad()
            specific_prediction.backward()

            # Get the gradients as importance values
            input_gradients = data_requires_grad.grad[batch_idx].abs().cpu().numpy()
        except Exception as e:
            print(f"Error during forward/backward pass: {e}")
            print(f"Unable to generate explanation for Sensor {curr_sensor+1}, Timestep +{curr_timestep+1}")
            continue  # Skip to next example if there's an error

        # Create visualization
        plt.figure(figsize=(15, 10))

        # Setup for the main heatmap
        gs = GridSpec(4, 4, figure=plt.gcf())
        ax_main = plt.subplot(gs[:3, :])

        # Create heatmap of gradients (absolute values for clearer visualization)
        im = ax_main.imshow(input_gradients.T, aspect='auto', cmap='viridis',
                          interpolation='nearest')

        # Add colorbar
        plt.colorbar(im, ax=ax_main, label='Absolute Gradient Magnitude')

        # Label axes
        ax_main.set_xlabel('Input Timestep')
        ax_main.set_ylabel('Feature')

        # Create y-tick labels with feature names
        if len(feature_names) == input_gradients.shape[1]:
            ax_main.set_yticks(range(len(feature_names)))
            ax_main.set_yticklabels(feature_names)

        # Add feature group boundaries
        current_pos = 0
        for group_name, group_size in feature_groups:
            if current_pos + group_size <= input_gradients.shape[1]:
                # Add horizontal line
                ax_main.axhline(y=current_pos - 0.5, color='red', linestyle='-', linewidth=1.5)

                # Add group label to the right of the heatmap
                ax_main.text(input_gradients.shape[0] + 1, current_pos + group_size/2 - 0.5,
                           group_name, verticalalignment='center', fontsize=10,
                           bbox=dict(facecolor='white', alpha=0.7, boxstyle='round'))

                current_pos += group_size

        # Add a horizontal line after the last group
        ax_main.axhline(y=current_pos - 0.5, color='red', linestyle='-', linewidth=1.5)

        # Main title
        plt.suptitle(f'Prediction Explanation for Sensor {curr_sensor+1}, Timestep +{curr_timestep+1}',
                   fontsize=16, fontweight='bold')

        # Get the original value of the prediction and target
        pred_value = specific_prediction.item()
        target_value = target[batch_idx, curr_timestep, curr_sensor].item()

        # USE DIRECT COMPARISON WITHOUT INVERSE TRANSFORM IF SCALER DIMENSIONS DON'T MATCH
        if scaler_n_features != output_dim:
            # If scaler dimensions don't match, skip inverse transform
            ax_pred = plt.subplot(gs[3, :2])
            ax_pred.axis('off')
            pred_text = (f"Prediction (normalized): {pred_value:.4f}\n"
                        f"Actual (normalized): {target_value:.4f}\n"
                        f"Error (normalized): {abs(pred_value - target_value):.4f}")
            ax_pred.text(0.1, 0.5, pred_text, fontsize=12,
                       bbox=dict(facecolor='lightyellow', alpha=0.5, boxstyle='round'))

            # Add note about skipping inverse transform
            ax_pred.text(0.1, 0.2, "(Note: Values shown in normalized scale due to scaler dimension mismatch)",
                       fontsize=10, color='red')
        else:
            # If scaler dimensions match, we can do proper inverse transform
            try:
                # First create arrays with the proper shape for the scaler
                pred_array = np.zeros((1, scaler_n_features))
                pred_array[0, curr_sensor] = pred_value

                target_array = np.zeros((1, scaler_n_features))
                target_array[0, curr_sensor] = target_value

                # Inverse transform
                pred_orig = data_scaler.inverse_transform(pred_array)[0, curr_sensor]
                target_orig = data_scaler.inverse_transform(target_array)[0, curr_sensor]

                # Add prediction info
                ax_pred = plt.subplot(gs[3, :2])
                ax_pred.axis('off')
                pred_text = (f"Prediction: {pred_orig:.2f}\n"
                            f"Actual: {target_orig:.2f}\n"
                            f"Error: {abs(pred_orig - target_orig):.2f} ({abs(pred_orig - target_orig)/max(0.01, abs(target_orig))*100:.1f}%)")
                ax_pred.text(0.1, 0.5, pred_text, fontsize=12,
                           bbox=dict(facecolor='lightyellow', alpha=0.5, boxstyle='round'))
            except Exception as e:
                # Fallback if inverse transform fails
                print(f"Error in inverse transform: {e}")
                ax_pred = plt.subplot(gs[3, :2])
                ax_pred.axis('off')
                pred_text = (f"Prediction (normalized): {pred_value:.4f}\n"
                            f"Actual (normalized): {target_value:.4f}\n"
                            f"Error (normalized): {abs(pred_value - target_value):.4f}")
                ax_pred.text(0.1, 0.5, pred_text, fontsize=12,
                           bbox=dict(facecolor='lightyellow', alpha=0.5, boxstyle='round'))

                # Add note about error in inverse transform
                ax_pred.text(0.1, 0.2, f"(Note: Values shown in normalized scale due to: {str(e)})",
                           fontsize=10, color='red')

        # Add explanation of the visualization
        ax_exp = plt.subplot(gs[3, 2:])
        ax_exp.axis('off')
        explanation = ("This visualization shows which input features and timesteps\n"
                      "most strongly influenced this specific prediction.\n"
                      "Brighter colors indicate stronger influence.")
        ax_exp.text(0.1, 0.5, explanation, fontsize=10,
                  bbox=dict(facecolor='lightblue', alpha=0.5, boxstyle='round'))

        # Calculate timestep with highest influence
        max_timestep = np.unravel_index(np.argmax(input_gradients), input_gradients.shape)[0]
        max_feature = np.unravel_index(np.argmax(input_gradients), input_gradients.shape)[1]

        # Mark the max influence point
        ax_main.plot(max_timestep, max_feature, 'rx', markersize=10)

        # Add annotation about most influential point
        if max_feature < len(feature_names):
            feature_name = feature_names[max_feature]
            ax_main.set_title(f"Strongest influence: {feature_name} at timestep {max_timestep}",
                            fontsize=12)

        # Save the figure
        timestamp = get_maputo_timestamp()
        filename = f'prediction_explanation_s{curr_sensor+1}_t{curr_timestep+1}_{timestamp}.png'
        plt.tight_layout()
        plt.savefig(os.path.join(results_dir, filename), dpi=300)
        plt.close()

        print(f"Generated explanation for Sensor {curr_sensor+1}, Timestep +{curr_timestep+1}")

    return

def visualize_hidden_states(model, input_batch, results_dir, config=None):
    """
    Visualize how hidden representations evolve throughout the transformer model.

    Args:
        model: The transformer model
        input_batch: A batch of input data (tensor)
        results_dir: Directory to save visualizations
        config: Optional model configuration
    """
    import os
    import numpy as np
    import matplotlib.pyplot as plt
    import torch
    import torch.nn.functional as F
    import seaborn as sns
    from datetime import datetime

    # Try importing PCA - if not available, we'll handle it in the visualization
    try:
        from sklearn.decomposition import PCA
        pca_available = True
    except ImportError:
        print("Warning: sklearn PCA not available. Some visualizations will be skipped.")
        pca_available = False

    # Helper function for timestamps if not available in globals
    def get_timestamp():
        if 'get_maputo_timestamp' in globals():
            return get_maputo_timestamp()
        else:
            return datetime.now().strftime("%Y%m%d_%H%M%S")

    device = next(model.parameters()).device
    hidden_states = []

    # Register hooks to capture hidden states
    def get_hidden_state_hook(layer_name):
        def hook(module, input, output):
            # For some layers, output might be a tuple
            if isinstance(output, tuple):
                output = output[0]
            hidden_states.append((layer_name, output.detach()))
        return hook

    # Create hooks for all relevant components
    hooks = []

    # Capture input embedding
    try:
        hooks.append(model.embedding.register_forward_hook(
            get_hidden_state_hook('1-Embedding')))
    except Exception as e:
        print(f"Warning: Could not hook embedding layer: {e}")

    # Capture after positional encoding if it exists
    if hasattr(model, 'pos_encoder'):
        try:
            hooks.append(model.pos_encoder.register_forward_hook(
                get_hidden_state_hook('2-PosEncoding')))
        except Exception as e:
            print(f"Warning: Could not hook positional encoding layer: {e}")

    # Capture after each transformer layer
    if hasattr(model, 'transformer'):
        for i, layer in enumerate(model.transformer):
            try:
                hooks.append(layer.register_forward_hook(
                    get_hidden_state_hook(f'3-Layer{i+1}')))
            except Exception as e:
                print(f"Warning: Could not hook transformer layer {i+1}: {e}")

    # Capture final output (before decoder)
    def capture_final(module, input, output):
        try:
            if hasattr(module, 'decoder') and isinstance(input, tuple):
                # This captures the input to the decoder (final hidden state)
                hidden_states.append(('4-FinalHidden', input[0].detach()))
        except Exception as e:
            print(f"Warning: Error capturing final hidden state: {e}")

    try:
        hooks.append(model.register_forward_hook(capture_final))
    except Exception as e:
        print(f"Warning: Could not hook model for final hidden state: {e}")

    # Run forward pass to collect hidden states
    model.eval()
    try:
        with torch.no_grad():
            _ = model(input_batch.to(device))
    except Exception as e:
        print(f"Error during model forward pass: {e}")
        # Remove hooks before returning
        for hook in hooks:
            hook.remove()
        return None

    # Remove hooks
    for hook in hooks:
        hook.remove()

    # Check if we captured any hidden states
    if not hidden_states:
        print("No hidden states were captured. Check model architecture.")
        return None

    # Create directory for hidden state visualizations
    timestamp = get_timestamp()
    hidden_states_dir = os.path.join(results_dir, f'hidden_states_{timestamp}')
    os.makedirs(hidden_states_dir, exist_ok=True)

    # ---------- 1. PCA Visualization of Hidden States ----------
    if pca_available:
        try:
            plt.figure(figsize=(14, 10))

            # Define colors and markers for different layers
            colors = plt.cm.viridis(np.linspace(0, 1, len(hidden_states)))
            markers = ['o', 's', '^', 'D', 'v', '<', '>', 'p', '*', 'h', 'H', '+', 'x', '|', '_']

            # Setup PCA
            pca = PCA(n_components=2)
            all_pca_data = []

            # Plot all hidden states in PCA space
            for i, (layer_name, state) in enumerate(hidden_states):
                try:
                    # Get a sample of hidden states (first batch item, all positions)
                    state_sample = state[0].cpu().numpy()  # Shape: [seq_len, hidden_dim]

                    # Flatten if needed
                    if len(state_sample.shape) > 2:
                        state_sample = state_sample.reshape(-1, state_sample.shape[-1])

                    # Apply PCA, but first check if we have enough samples and features
                    if state_sample.shape[0] > 1 and state_sample.shape[1] > 1:
                        state_pca = pca.fit_transform(state_sample)

                        # Store PCA data for later aggregate plot
                        all_pca_data.append((layer_name, state_pca, colors[i]))

                        # Plot each position in the sequence
                        plt.scatter(
                            state_pca[:, 0], state_pca[:, 1],
                            alpha=0.7,
                            color=colors[i],
                            marker=markers[i % len(markers)],
                            label=f"{layer_name} (mean)"
                        )

                        # Plot the mean position with a larger marker
                        mean_pos = state_pca.mean(axis=0)
                        plt.scatter(
                            mean_pos[0], mean_pos[1],
                            s=200,
                            color=colors[i],
                            marker=markers[i % len(markers)],
                            edgecolors='black',
                            linewidths=1.5
                        )
                except Exception as e:
                    print(f"Warning: Error processing PCA for layer {layer_name}: {e}")

            plt.title("PCA Projection of Hidden States", fontsize=16)
            plt.xlabel("Principal Component 1", fontsize=12)
            plt.ylabel("Principal Component 2", fontsize=12)
            plt.legend(loc='best')
            plt.grid(alpha=0.3)

            # Add arrows between consecutive layer means to show evolution
            if len(all_pca_data) > 1:
                try:
                    layer_means = [data[1].mean(axis=0) for data in all_pca_data]
                    for i in range(len(layer_means) - 1):
                        plt.annotate(
                            "",
                            xy=layer_means[i+1],
                            xytext=layer_means[i],
                            arrowprops=dict(arrowstyle="->", color="gray", alpha=0.6, lw=1.5)
                        )
                except Exception as e:
                    print(f"Warning: Error drawing evolution arrows: {e}")

            plt.tight_layout()
            plt.savefig(os.path.join(hidden_states_dir, '1_pca_projection.png'), dpi=300)
            plt.close()
        except Exception as e:
            print(f"Error creating PCA visualization: {e}")

    # ---------- 2. Evolution of Hidden State Statistics ----------
    try:
        plt.figure(figsize=(15, 10))
        grid = plt.GridSpec(2, 2)

        # 2.1 Hidden state norms across layers
        ax1 = plt.subplot(grid[0, 0])
        layer_names = [name for name, _ in hidden_states]

        # Calculate norms for each layer
        norms = []
        for _, state in hidden_states:
            try:
                # Calculate norm of each position's hidden state
                state_norms = torch.norm(state, dim=-1).cpu().numpy()
                # Average across positions and batch
                avg_norm = state_norms.mean()
                norms.append(avg_norm)
            except Exception as e:
                print(f"Warning: Error calculating norm: {e}")
                norms.append(0)  # Use 0 as fallback

        # Plot norms
        sns.barplot(x=layer_names, y=norms, ax=ax1)
        ax1.set_title("Average Hidden State Norm by Layer", fontsize=14)
        ax1.set_xlabel("")
        ax1.set_ylabel("Norm", fontsize=12)
        ax1.tick_params(axis='x', rotation=45)

        # 2.2 Hidden state variance across layers
        ax2 = plt.subplot(grid[0, 1])

        # Calculate variance for each layer
        variances = []
        for _, state in hidden_states:
            try:
                # Calculate variance of hidden states
                layer_var = torch.var(state, dim=-1).cpu().numpy()
                # Average across positions and batch
                avg_var = layer_var.mean()
                variances.append(avg_var)
            except Exception as e:
                print(f"Warning: Error calculating variance: {e}")
                variances.append(0)  # Use 0 as fallback

        # Plot variances
        sns.barplot(x=layer_names, y=variances, ax=ax2)
        ax2.set_title("Average Hidden State Variance by Layer", fontsize=14)
        ax2.set_xlabel("")
        ax2.set_ylabel("Variance", fontsize=12)
        ax2.tick_params(axis='x', rotation=45)

        # 2.3 Similarity between consecutive layers
        ax3 = plt.subplot(grid[1, 0])

        similarities = []
        sim_labels = []

        for i in range(len(hidden_states) - 1):
            name1 = hidden_states[i][0]
            name2 = hidden_states[i+1][0]
            state1 = hidden_states[i][1]
            state2 = hidden_states[i+1][1]

            try:
                # Flatten states for comparison
                state1_flat = state1.reshape(-1, state1.shape[-1])
                state2_flat = state2.reshape(-1, state2.shape[-1])

                # If dimensions don't match, use mean vectors
                if state1_flat.shape[0] != state2_flat.shape[0] or state1_flat.shape[1] != state2_flat.shape[1]:
                    state1_mean = torch.mean(state1_flat, dim=0, keepdim=True)
                    state2_mean = torch.mean(state2_flat, dim=0, keepdim=True)

                    # Handle different feature dimensions by padding the smaller one
                    if state1_mean.shape[1] != state2_mean.shape[1]:
                        max_dim = max(state1_mean.shape[1], state2_mean.shape[1])
                        if state1_mean.shape[1] < max_dim:
                            padding = torch.zeros(1, max_dim - state1_mean.shape[1], device=state1_mean.device)
                            state1_mean = torch.cat([state1_mean, padding], dim=1)
                        if state2_mean.shape[1] < max_dim:
                            padding = torch.zeros(1, max_dim - state2_mean.shape[1], device=state2_mean.device)
                            state2_mean = torch.cat([state2_mean, padding], dim=1)

                    # Calculate cosine similarity between mean vectors
                    sim = F.cosine_similarity(state1_mean, state2_mean, dim=1)
                    avg_sim = sim.item()
                else:
                    # Calculate row-wise cosine similarity and average
                    # Normalize each row first
                    state1_norm = F.normalize(state1_flat, p=2, dim=1)
                    state2_norm = F.normalize(state2_flat, p=2, dim=1)

                    # Compute cosine similarity
                    cos_sim = torch.sum(state1_norm * state2_norm, dim=1)
                    avg_sim = cos_sim.mean().item()
            except Exception as e:
                print(f"Warning: Error calculating similarity between {name1} and {name2}: {e}")
                avg_sim = 0  # Use 0 as fallback

            similarities.append(avg_sim)
            sim_labels.append(f"{name1}\n→\n{name2}")

        # Plot similarities
        sns.barplot(x=sim_labels, y=similarities, ax=ax3)
        ax3.set_title("Cosine Similarity Between Consecutive Layers", fontsize=14)
        ax3.set_xlabel("")
        ax3.set_ylabel("Similarity", fontsize=12)
        ax3.axhline(y=0, color='r', linestyle='--', alpha=0.5)

        # 2.4 Distribution of final hidden state values
        ax4 = plt.subplot(grid[1, 1])

        try:
            # Get the final hidden state
            final_hidden = hidden_states[-1][1].cpu().numpy()

            # Plot histogram of values
            ax4.hist(final_hidden.flatten(), bins=50, alpha=0.7)
            ax4.set_title("Distribution of Final Hidden State Values", fontsize=14)
            ax4.set_xlabel("Value", fontsize=12)
            ax4.set_ylabel("Frequency", fontsize=12)

            # Add vertical line at 0
            ax4.axvline(x=0, color='r', linestyle='--')

            # Add statistics as text
            stats_text = (
                f"Mean: {final_hidden.mean():.4f}\n"
                f"Std: {final_hidden.std():.4f}\n"
                f"Min: {final_hidden.min():.4f}\n"
                f"Max: {final_hidden.max():.4f}"
            )
            ax4.text(0.05, 0.95, stats_text, transform=ax4.transAxes,
                    verticalalignment='top', bbox=dict(boxstyle='round', alpha=0.1))
        except Exception as e:
            print(f"Warning: Error creating final hidden state distribution: {e}")
            ax4.text(0.5, 0.5, f"Error creating distribution: {e}",
                    ha='center', va='center', transform=ax4.transAxes)

        plt.tight_layout()
        plt.savefig(os.path.join(hidden_states_dir, '2_hidden_state_statistics.png'), dpi=300)
        plt.close()
    except Exception as e:
        print(f"Error creating hidden state statistics: {e}")

    # ---------- 3. Hidden State Attention Analysis ----------
    # This analysis shows how attention patterns correlate with hidden state changes
    try:
        if hasattr(model, 'attention_weights') and model.attention_weights is not None:
            plt.figure(figsize=(12, 10))

            # Get attention weights
            attn_weights = model.attention_weights
            if isinstance(attn_weights, torch.Tensor):
                attn_weights = attn_weights.cpu().numpy()

            # Find last transformer layer's hidden states
            transformer_states = [state for name, state in hidden_states
                                if name.startswith('3-Layer')]

            if transformer_states:
                try:
                    last_state = transformer_states[-1].cpu().numpy()
                    last_state_2d = last_state[0]  # First batch item

                    # Calculate hidden state changes between positions
                    state_change = np.zeros((last_state_2d.shape[0], last_state_2d.shape[0]))
                    for i in range(last_state_2d.shape[0]):
                        for j in range(last_state_2d.shape[0]):
                            # L2 distance between hidden states
                            state_change[i, j] = np.linalg.norm(
                                last_state_2d[i] - last_state_2d[j])

                    # Plot hidden state distances
                    plt.subplot(2, 2, 1)
                    sns.heatmap(state_change, cmap='viridis')
                    plt.title("Hidden State Distances")
                    plt.xlabel("Position j")
                    plt.ylabel("Position i")

                    # Plot attention weights
                    plt.subplot(2, 2, 2)

                    # Handle different attention weight shapes
                    if len(attn_weights.shape) > 2:
                        # Average across batch and heads if necessary
                        attn_plot = np.mean(attn_weights, axis=tuple(
                            range(len(attn_weights.shape) - 2)))
                    else:
                        attn_plot = attn_weights

                    # Check if dimensions match and adjust if needed
                    if attn_plot.shape != state_change.shape:
                        print(f"Warning: Attention weights shape {attn_plot.shape} doesn't match state change shape {state_change.shape}")
                        # Reshape or crop to match
                        if len(attn_plot.shape) == 2:
                            h, w = attn_plot.shape
                            h2, w2 = state_change.shape
                            h_min, w_min = min(h, h2), min(w, w2)
                            attn_plot = attn_plot[:h_min, :w_min]
                            state_change = state_change[:h_min, :w_min]

                    sns.heatmap(attn_plot, cmap='viridis')
                    plt.title("Attention Weights")
                    plt.xlabel("Key Position")
                    plt.ylabel("Query Position")

                    # Plot correlation between attention and state change
                    plt.subplot(2, 2, 3)

                    # Flatten both matrices
                    attn_flat = attn_plot.flatten()
                    state_change_flat = state_change.flatten()

                    plt.scatter(attn_flat, state_change_flat, alpha=0.3)
                    plt.title("Attention vs. Hidden State Change")
                    plt.xlabel("Attention Weight")
                    plt.ylabel("Hidden State Distance")

                    # Add trend line
                    if len(attn_flat) > 1:
                        z = np.polyfit(attn_flat, state_change_flat, 1)
                        p = np.poly1d(z)
                        plt.plot(sorted(attn_flat), p(sorted(attn_flat)),
                                "r--", alpha=0.8)

                        # Add correlation coefficient
                        corr = np.corrcoef(attn_flat, state_change_flat)[0, 1]
                        plt.text(0.05, 0.95, f"Correlation: {corr:.4f}",
                                transform=plt.gca().transAxes,
                                verticalalignment='top',
                                bbox=dict(boxstyle='round', alpha=0.1))
                except Exception as e:
                    print(f"Warning: Error creating attention vs state change plots: {e}")

            # Add additional analysis in the 4th subplot
            plt.subplot(2, 2, 4)

            try:
                # Analyze hidden state evolution by position
                pos_norms = []
                for name, state in hidden_states:
                    if name.startswith(('1-', '2-', '3-')):  # Skip final output
                        state_sample = state[0]  # First batch item
                        pos_norm = torch.norm(state_sample, dim=-1).cpu().numpy()
                        pos_norms.append((name, pos_norm))

                # Plot norm evolution for different positions
                if pos_norms:
                    num_positions = min(5, len(pos_norms[0][1]) if pos_norms[0][1].size > 0 else 0)
                    for pos_idx in range(num_positions):  # Show first 5 positions
                        try:
                            pos_values = [norms[1][pos_idx] if pos_idx < len(norms[1]) else 0
                                        for norms in pos_norms]
                            plt.plot(range(len(pos_values)), pos_values,
                                    marker='o', label=f"Pos {pos_idx+1}")
                        except Exception as e:
                            print(f"Warning: Error plotting position {pos_idx}: {e}")

                plt.title("Hidden State Norm Evolution by Position")
                plt.xlabel("Layer")
                plt.xticks(range(len(pos_norms)),
                        [name.split('-')[0] for name, _ in pos_norms],
                        rotation=45)
                plt.ylabel("Norm")
                plt.legend()
            except Exception as e:
                print(f"Warning: Error creating position norm evolution plot: {e}")
                plt.text(0.5, 0.5, f"Error: {e}", ha='center', va='center')

            plt.tight_layout()
            plt.savefig(os.path.join(hidden_states_dir, '3_attention_analysis.png'), dpi=300)
            plt.close()
    except Exception as e:
        print(f"Error creating attention analysis: {e}")

    # ---------- 4. Hidden State Trajectory Visualization ----------
    # This shows how individual positions evolve through the network
    if pca_available:
        try:
            # Try importing 3D plotting components
            from mpl_toolkits.mplot3d import Axes3D

            plt.figure(figsize=(12, 10))
            ax = plt.subplot(111, projection='3d')

            # Apply PCA to all hidden states together to get consistent components
            all_states = []
            for _, state in hidden_states:
                try:
                    # Take first batch item, all sequence positions
                    state_np = state[0].cpu().numpy()
                    # Reshape if needed
                    if len(state_np.shape) > 2:
                        state_np = state_np.reshape(-1, state_np.shape[-1])
                    all_states.append(state_np)
                except Exception as e:
                    print(f"Warning: Error processing state for PCA: {e}")

            if not all_states:
                raise ValueError("No valid states for PCA")

            # Combine all hidden states
            try:
                all_states_combined = np.vstack(all_states)

                # Fit PCA on combined data for consistent dimensions
                pca = PCA(n_components=3)
                pca.fit(all_states_combined)

                # Track specific positions through the layers
                num_positions_to_track = min(5, input_batch.shape[1])
                layer_indices = range(len(hidden_states))

                # Create consistent marker styles and colors
                position_colors = plt.cm.tab10(np.linspace(0, 1, num_positions_to_track))
                layer_markers = ['o', 's', '^', 'D', 'v', '<', '>', 'p', '*', 'h']

                # Plot each position's trajectory through layers
                for pos_idx in range(num_positions_to_track):
                    # Extract this position's hidden state at each layer
                    pos_trajectory = []
                    for layer_idx, (_, state) in enumerate(hidden_states):
                        try:
                            # Check if the state has enough positions
                            if pos_idx < state.shape[1]:
                                # Extract hidden state for this position
                                pos_state = state[0, pos_idx].cpu().numpy()
                                # Transform with pre-fit PCA
                                pos_pca = pca.transform(pos_state.reshape(1, -1))[0]
                                pos_trajectory.append(pos_pca)
                            else:
                                print(f"Warning: Position {pos_idx} out of bounds for layer {layer_idx}")
                        except Exception as e:
                            print(f"Warning: Error processing position {pos_idx} in layer {layer_idx}: {e}")

                    # Plot trajectory only if we have data
                    if pos_trajectory:
                        # Convert to numpy for easier indexing
                        pos_trajectory = np.array(pos_trajectory)

                        if len(pos_trajectory) > 1:  # Need at least 2 points for a line
                            # Plot trajectory line
                            ax.plot(pos_trajectory[:, 0],
                                    pos_trajectory[:, 1],
                                    pos_trajectory[:, 2],
                                    color=position_colors[pos_idx],
                                    alpha=0.5,
                                    label=f"Position {pos_idx+1}")

                        # Plot points for each layer with different markers
                        for layer_idx in range(len(pos_trajectory)):
                            ax.scatter(pos_trajectory[layer_idx, 0],
                                    pos_trajectory[layer_idx, 1],
                                    pos_trajectory[layer_idx, 2],
                                    color=position_colors[pos_idx],
                                    marker=layer_markers[layer_idx % len(layer_markers)],
                                    s=50)

                # Add layer marker legend
                legend_elements = []
                for layer_idx, (layer_name, _) in enumerate(hidden_states):
                    legend_elements.append(plt.Line2D([0], [0],
                                                    marker=layer_markers[layer_idx % len(layer_markers)],
                                                    color='gray',
                                                    linestyle='None',
                                                    markersize=8,
                                                    label=layer_name))

                # Create two legends - one for positions, one for layers
                ax.legend(loc='upper left', title="Sequence Positions")
                ax2 = plt.gca().add_artist(plt.legend(handles=legend_elements,
                                                    loc='upper right',
                                                    title="Model Layers"))

                ax.set_title("3D Hidden State Trajectories Across Layers", fontsize=14)
                ax.set_xlabel("PC1", fontsize=12)
                ax.set_ylabel("PC2", fontsize=12)
                ax.set_zlabel("PC3", fontsize=12)
            except Exception as e:
                plt.clf()  # Clear the figure
                plt.text(0.5, 0.5, f"Error creating 3D trajectories: {e}",
                        ha='center', va='center', transform=plt.gca().transAxes)
                plt.title("Error in Trajectory Visualization")

            plt.tight_layout()
            plt.savefig(os.path.join(hidden_states_dir, '4_hidden_state_trajectories.png'), dpi=300)
            plt.close()
        except Exception as e:
            print(f"Error creating hidden state trajectories: {e}")

    # Create index.html to view all plots
    try:
        html_content = f"""
        <!DOCTYPE html>
        <html>
        <head>
            <title>Hidden State Analysis - {timestamp}</title>
            <style>
                body {{ font-family: Arial, sans-serif; margin: 20px; }}
                h1 {{ color: #333; }}
                .plot-container {{ margin-bottom: 30px; }}
                img {{ max-width: 100%; border: 1px solid #ddd; }}
            </style>
        </head>
        <body>
            <h1>Hidden State Evolution Analysis</h1>
            <p>Generated on: {timestamp}</p>

            <div class="plot-container">
                <h2>1. PCA Projection of Hidden States</h2>
                <img src="1_pca_projection.png" alt="PCA Projection">
                <p>This visualization shows how hidden states from different layers cluster in 2D space after PCA reduction.</p>
            </div>

            <div class="plot-container">
                <h2>2. Hidden State Statistics</h2>
                <img src="2_hidden_state_statistics.png" alt="Hidden State Statistics">
                <p>This visualization shows statistical properties of hidden states across layers, including norms, variances, and similarities.</p>
            </div>

            <div class="plot-container">
                <h2>3. Attention and Hidden State Correlation</h2>
                <img src="3_attention_analysis.png" alt="Attention Analysis">
                <p>This visualization shows the relationship between attention weights and hidden state changes.</p>
            </div>

            <div class="plot-container">
                <h2>4. Hidden State Trajectories</h2>
                <img src="4_hidden_state_trajectories.png" alt="Hidden State Trajectories">
                <p>This 3D visualization shows how individual sequence positions' hidden states evolve through the network.</p>
            </div>
        </body>
        </html>
        """

        # Write HTML file
        with open(os.path.join(hidden_states_dir, 'index.html'), 'w') as f:
            f.write(html_content)

        print(f"Hidden state evolution analysis saved to: {hidden_states_dir}")
        print(f"View the HTML report at: {os.path.join(hidden_states_dir, 'index.html')}")
    except Exception as e:
        print(f"Error creating HTML report: {e}")

    return hidden_states_dir

def plot_attention_heads(
    model: nn.Module,
    input_batch: torch.Tensor,
    results_dir: str,
    config: Optional[TrainingConfig] = None,
    layer_idx: int = -1,
    feature_names: Optional[List[str]] = None,
    adjacency_matrix: Optional[torch.Tensor] = None  # Add adjacency_matrix parameter
) -> None:
    """
    Visualize individual attention heads in the transformer model.

    Args:
        model: The trained Traffic Transformer model
        input_batch: A batch of input data [batch_size, seq_len, features]
        results_dir: Directory to save visualizations
        config: Training configuration
        layer_idx: Which layer to visualize (-1 for last layer)
        feature_names: Optional list of feature names for better labeling
        adjacency_matrix: Adjacency matrix for GNN pre-transformer (required if model uses GNN)
    """
    device = next(model.parameters()).device
    model.eval()

    # Check if model requires adjacency_matrix
    requires_adjacency = (
        hasattr(config, 'use_gnn_pre_transformer') and
        config.use_gnn_pre_transformer
    )

    if requires_adjacency and adjacency_matrix is None:
        print("Model requires adjacency_matrix but none was provided.")
        print("Please provide the adjacency matrix when calling this function.")
        return  # Exit the function if no adjacency matrix is available

    # Define a hook to capture multi-head attention weights before they're averaged
    pre_softmax_attentions = {}
    post_softmax_attentions = {}

    def attention_hook(module, input, output, layer_id):
        # For most PyTorch transformer implementations:
        # - The attention scores are in the first element of the tuple
        # - For MultiheadAttention, need_weights=True returns (output, attn_weights)
        if isinstance(output, tuple) and len(output) > 1:
            # This captures post-softmax attention weights
            attn_weights = output[1]
            if isinstance(attn_weights, torch.Tensor):
                post_softmax_attentions[layer_id] = attn_weights.detach()

        # Try to capture pre-softmax attention scores if available
        # This requires knowing the internal variable names in the MultiheadAttention implementation
        if hasattr(module, 'attn_output_weights'):
            pre_softmax_attentions[layer_id] = module.attn_output_weights.detach()

    # Register hooks for each transformer layer's self-attention module
    hooks = []
    for i, layer in enumerate(model.transformer):
        # Find the self-attention module (structure depends on your implementation)
        if hasattr(layer, 'self_attn'):
            hook = layer.self_attn.register_forward_hook(
                lambda mod, inp, out, layer_id=i: attention_hook(mod, inp, out, layer_id)
            )
            hooks.append(hook)
        elif hasattr(layer, 'mha'):  # Some implementations use different naming
            hook = layer.mha.register_forward_hook(
                lambda mod, inp, out, layer_id=i: attention_hook(mod, inp, out, layer_id)
            )
            hooks.append(hook)
        else:
            # If structure is different, try to traverse the module to find MultiheadAttention
            for name, module in layer.named_modules():
                if isinstance(module, nn.MultiheadAttention):
                    hook = module.register_forward_hook(
                        lambda mod, inp, out, layer_id=i: attention_hook(mod, inp, out, layer_id)
                    )
                    hooks.append(hook)
                    break

    # Run forward pass to capture attention weights
    try:
        with torch.no_grad():
            input_batch = input_batch.to(device)
            # Pass adjacency_matrix if the model requires it
            if requires_adjacency:
                adjacency_matrix = adjacency_matrix.to(device)
                _ = model(input_batch, adjacency_matrix)
            else:
                _ = model(input_batch)
    except Exception as e:
        print(f"Error during model forward pass: {e}")
        # Clean up hooks before returning
        for hook in hooks:
            hook.remove()
        return

    # Remove hooks after forward pass
    for hook in hooks:
        hook.remove()

    # Select the layer to visualize
    if layer_idx < 0:
        layer_idx = len(model.transformer) + layer_idx  # Convert negative index

    # Get attention weights for the selected layer
    if layer_idx in post_softmax_attentions:
        attention_weights = post_softmax_attentions[layer_idx]
    elif layer_idx in pre_softmax_attentions:
        attention_weights = pre_softmax_attentions[layer_idx]
    else:
        print(f"No attention weights captured for layer {layer_idx}")
        return

    # Process the attention weights based on shape
    # [batch, heads, seq, seq] or [batch * heads, seq, seq]
    if len(attention_weights.shape) == 4:
        # Shape is [batch, heads, seq, seq]
        batch_size, num_heads, seq_len, _ = attention_weights.shape
        # Use only the first batch element
        attention_weights = attention_weights[0]
    elif len(attention_weights.shape) == 3:
        # Shape might be [batch * heads, seq, seq]
        # Need to reshape based on number of heads
        if hasattr(model, 'num_heads'):
            num_heads = model.num_heads
        elif config and hasattr(config, 'num_heads'):
            num_heads = config.num_heads
        else:
            # Try to infer from the model architecture or config
            num_heads = 8  # Default fallback
            print(f"Assuming {num_heads} attention heads (couldn't detect from model)")

        batch_size = attention_weights.shape[0] // num_heads
        seq_len = attention_weights.shape[1]
        attention_weights = attention_weights.view(batch_size, num_heads, seq_len, seq_len)[0]
    else:
        print(f"Unexpected attention weights shape: {attention_weights.shape}")
        return

    # Move to CPU and convert to numpy for plotting
    attention_weights = attention_weights.cpu().numpy()

    # Prepare a grid of heatmaps, one for each attention head
    num_rows = int(np.ceil(num_heads / 4))
    num_cols = min(4, num_heads)
    fig, axes = plt.subplots(num_rows, num_cols, figsize=(16, 3 * num_rows))

    # Make axes indexable if there's only one row
    if num_rows == 1 and num_cols == 1:
        axes = np.array([axes])
    elif num_rows == 1 or num_cols == 1:
        axes = axes.reshape(-1)

    # Create feature labels if available
    if feature_names and len(feature_names) == seq_len:
        labels = feature_names
    else:
        labels = [str(i+1) for i in range(seq_len)]

    # Plot each attention head
    for h in range(num_heads):
        if h < len(axes.flat):
            row, col = h // num_cols, h % num_cols
            ax = axes[row, col] if num_rows > 1 and num_cols > 1 else axes[h]

            # Plot the attention heatmap
            sns.heatmap(
                attention_weights[h],
                cmap='viridis',
                xticklabels=labels if seq_len <= 20 else False,  # Only show labels if not too many
                yticklabels=labels if seq_len <= 20 else False,
                ax=ax
            )

            ax.set_title(f"Head {h+1}")

            # Add feature group boundaries if enabled
            if config and hasattr(config, 'use_time_features') and seq_len <= 20:
                # Define feature groups similar to plot_attention_weights
                feature_groups = []

                # Traffic data features (always present)
                num_sensors = getattr(config, 'num_features', seq_len)
                feature_groups.append(("Traffic", num_sensors))

                # Time features
                if hasattr(config, 'use_time_features') and config.use_time_features:
                    time_features = 4  # hour, dayofweek, weekofyear, month
                    feature_groups.append(("Time", time_features))

                # Holiday feature
                if hasattr(config, 'use_holiday_feature') and config.use_holiday_feature:
                    feature_groups.append(("Holiday", 1))

                # Weather features
                if hasattr(config, 'use_weather_feature') and config.use_weather_feature:
                    if config.weather_feature_type == 'all_features':
                        weather_features = 8  # All weather features
                    elif config.weather_feature_type == 'temperature':
                        weather_features = 1
                    elif config.weather_feature_type == 'wind':
                        weather_features = 2  # wind_speed, wind_direction
                    else:
                        weather_features = 1  # Default for other single weather features
                    feature_groups.append(("Weather", weather_features))

                # Lagged features
                if hasattr(config, 'use_lagged_features') and config.use_lagged_features:
                    lagged_features = getattr(config, 'num_lags', 24)
                    feature_groups.append(("Lagged", lagged_features))

                # Spatial features
                if hasattr(config, 'use_spatial_features') and config.use_spatial_features:
                    spatial_features = getattr(config, 'spatial_feature_dim', 16)
                    feature_groups.append(("Spatial", spatial_features))

                # Add vertical and horizontal lines at group boundaries
                colors = ['red', 'blue', 'green', 'purple', 'orange', 'cyan']
                current_pos = 0
                color_idx = 0

                for group_name, group_size in feature_groups:
                    if current_pos + group_size <= seq_len:
                        # Add vertical & horizontal lines
                        ax.axvline(x=current_pos, color=colors[color_idx], linestyle='--', linewidth=0.8)
                        ax.axhline(y=current_pos, color=colors[color_idx], linestyle='--', linewidth=0.8)

                        current_pos += group_size
                        color_idx = (color_idx + 1) % len(colors)

            # Clean up axis labels if too many features
            if seq_len > 20:
                ax.set_xlabel("Query Position")
                ax.set_ylabel("Key Position")

    # Hide any unused subplots
    for i in range(num_heads, len(axes.flat)):
        axes.flat[i].set_visible(False)

    # Add title and other metadata
    plt.suptitle(f"Attention Heads in Layer {layer_idx+1}", fontsize=16)

    # Add explanation text
    if config:
        transformer_info = (
            f"Model: {config.hidden_dim} hidden dims, {config.num_layers} layers\n"
            f"Prediction Length: {config.pred_length}, Sequence Length: {config.seq_length}"
        )
        plt.figtext(0.5, 0.01, transformer_info, ha='center', fontsize=10)

    # Save the visualization
    timestamp = get_maputo_timestamp()
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])  # Make room for suptitle
    save_path = os.path.join(results_dir, f'attention_heads_layer{layer_idx+1}_{timestamp}.png')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()

    print(f"Attention heads visualization saved to {save_path}")

    # Create another visualization showing the diversity/specialization of heads
    plt.figure(figsize=(10, 6))

    # Calculate head diversity metrics
    head_entropies = []
    for h in range(num_heads):
        # Normalize attention weights to sum to 1
        head_weights = attention_weights[h].flatten()
        head_weights = head_weights / head_weights.sum()

        # Calculate entropy (higher entropy = more uniform attention)
        epsilon = 1e-10  # Avoid log(0)
        entropy = -np.sum(head_weights * np.log(head_weights + epsilon))
        head_entropies.append(entropy)

    # Plot head entropies (measure of attention dispersion)
    plt.bar(range(1, num_heads+1), head_entropies, alpha=0.7)
    plt.axhline(y=np.mean(head_entropies), color='r', linestyle='--', label='Average Entropy')
    plt.xlabel('Attention Head')
    plt.ylabel('Entropy (Higher = More Uniform Attention)')
    plt.title('Attention Head Specialization Analysis')
    plt.legend()

    # Save the specialization analysis
    spec_path = os.path.join(results_dir, f'head_specialization_layer{layer_idx+1}_{timestamp}.png')
    plt.savefig(spec_path, dpi=300)
    plt.close()

    print(f"Head specialization analysis saved to {spec_path}")

def visualize_feature_importance(
    model: nn.Module,
    test_loader: DataLoader,
    results_dir: str,
    config: Optional[TrainingConfig] = None
) -> None:
    """
    Visualize the learned importance weights for each feature group.

    Args:
        model: Trained model with feature-wise attention
        test_loader: DataLoader with test data
        results_dir: Directory to save visualizations
        config: Training configuration
    """
    # Check if model has feature importances
    if not hasattr(model, 'feature_importances') or not model.feature_importances:
        print("Model does not have feature importance information. Skipping visualization.")
        return

    # Get feature importances
    feature_names = list(model.feature_importances.keys())
    importance_values = list(model.feature_importances.values())

    # Create visualization
    plt.figure(figsize=(10, 6))
    bars = plt.bar(feature_names, importance_values, color=plt.cm.viridis(np.linspace(0, 1, len(feature_names))))

    # Add values on top of bars
    for bar, value in zip(bars, importance_values):
        plt.text(
            bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.01,
            f'{value:.4f}',
            ha='center', va='bottom'
        )

    plt.title('Feature Group Importance Weights', fontsize=16)
    plt.ylabel('Importance Weight')
    plt.ylim(0, max(importance_values) * 1.2)  # Add some space for text
    plt.grid(axis='y', alpha=0.3)

    # Add model details
    if config:
        plt.figtext(
            0.5, 0.01,
            f"Model: {config.num_layers} layers, {config.num_heads} heads, {config.hidden_dim} hidden dim | "
            f"Generated: {get_maputo_timestamp()}",
            ha='center', fontsize=10
        )

    # Save visualization
    timestamp = get_maputo_timestamp()
    plt.tight_layout()
    save_path = os.path.join(results_dir, f'feature_importance_weights_{timestamp}.png')
    plt.savefig(save_path, dpi=300)
    plt.close()

    print(f"Feature importance visualization saved to {save_path}")

def create_feature_mapping(config: TrainingConfig) -> Dict[int, str]:
    """
    Creates a mapping of feature indices to human-readable feature names
    based on the configuration.

    Args:
        config: Training configuration with feature information

    Returns:
        Dictionary mapping feature indices to feature names
    """
    feature_names = {}
    idx = 0

    # Traffic data features
    num_sensors = getattr(config, 'num_features', 207)
    for i in range(num_sensors):
        feature_names[idx] = f"Traffic_Sensor_{i+1}"
        idx += 1

    # Time features
    if hasattr(config, 'use_time_features') and config.use_time_features:
        time_feature_names = ["Hour", "DayOfWeek", "WeekOfYear", "Month"]
        for name in time_feature_names:
            feature_names[idx] = name
            idx += 1

    # Holiday feature
    if hasattr(config, 'use_holiday_feature') and config.use_holiday_feature:
        feature_names[idx] = "IsHoliday"
        idx += 1

    # Weather features
    if hasattr(config, 'use_weather_feature') and config.use_weather_feature:
        if hasattr(config, 'weather_feature_type'):
            if config.weather_feature_type == 'all_features':
                weather_names = ["Temperature", "WeatherCondition", "Visibility",
                                "WindSpeed", "WindDirection", "Humidity",
                                "DewPoint", "CloudCover"]
                for name in weather_names:
                    feature_names[idx] = name
                    idx += 1
            elif config.weather_feature_type == 'temperature':
                feature_names[idx] = "Temperature"
                idx += 1
            elif config.weather_feature_type == 'wind':
                feature_names[idx] = "WindSpeed"
                idx += 1
                feature_names[idx] = "WindDirection"
                idx += 1
            else:
                feature_names[idx] = config.weather_feature_type.capitalize()
                idx += 1
        else:
            # Default to all weather features
            weather_names = ["Temperature", "WeatherCondition", "Visibility",
                            "WindSpeed", "WindDirection", "Humidity",
                            "DewPoint", "CloudCover"]
            for name in weather_names:
                feature_names[idx] = name
                idx += 1

    # Lagged features
    if hasattr(config, 'use_lagged_features') and config.use_lagged_features:
        num_lags = getattr(config, 'num_lags', 24)
        for i in range(num_lags):
            feature_names[idx] = f"Lag_{i+1}"
            idx += 1

    # Spatial features
    if hasattr(config, 'use_spatial_features') and config.use_spatial_features:
        spatial_dim = getattr(config, 'spatial_feature_dim', 16)
        for i in range(spatial_dim):
            feature_names[idx] = f"Spatial_{i+1}"
            idx += 1

    return feature_names

def plot_predictions_vs_actual(
    actuals: np.ndarray,
    predictions: np.ndarray,
    sensor_index: int,
    sensor_id: str,  # Parameter for actual sensor ID
    fold: int,
    results_dir: str,
    pred_len: int,
    config: Optional['TrainingConfig'] = None, # Use string hint if TrainingConfig defined later
) -> None:
    """
    Plots actual vs predicted traffic flow with enhanced details using actual sensor ID.
    Compares the first 288 (24 hours) time steps or the available length if shorter.

    Args:
        actuals: Numpy array of actual values [num_samples, num_sensors].
        predictions: Numpy array of predicted values [num_samples, num_sensors].
        sensor_index: The column index of the sensor to plot.
        sensor_id: The actual string identifier of the sensor (e.g., '773869').
        fold: The cross-validation fold number (0-based).
        results_dir: The directory to save the plot image.
        pred_len: The prediction length (horizon) used by the model.
        config: The TrainingConfig object containing model hyperparameters.
    """
    # Ensure sensor_index is valid before accessing data
    if sensor_index >= actuals.shape[1] or sensor_index >= predictions.shape[1]:
         print(f"Warning: sensor_index {sensor_index} is out of bounds for data shapes "
               f"actuals:{actuals.shape}, predictions:{predictions.shape}. Skipping plot for sensor ID {sensor_id}.")
         return
    if actuals.shape[0] == 0 or predictions.shape[0] == 0:
        print(f"Warning: Actuals or predictions array is empty. Skipping plot for sensor ID {sensor_id}.")
        return

    # Determine the number of time steps to plot (up to 288)
    plot_length = min(288, actuals.shape[0], predictions.shape[0])

    # Calculate R-squared specifically for this plot (subset of data)
    # Ensure there are enough points to calculate R2 score
    if plot_length <= 1:
        print(f"Warning: Not enough data points ({plot_length}) to calculate R-squared for sensor {sensor_id}. Setting R2 to N/A.")
        r2 = np.nan
    else:
        try:
            # Slice data up to plot_length for R2 calculation
            r2 = r2_score(actuals[:plot_length, sensor_index], predictions[:plot_length, sensor_index])
        except ValueError as e:
            # Catch potential errors during R2 calculation (e.g., constant input)
            print(f"Warning: Could not calculate R-squared for sensor {sensor_id}. Error: {e}. Setting R2 to NaN.")
            r2 = np.nan

    plt.figure(figsize=(14, 8))

    # --- Main Plot ---
    plt.subplot(4, 1, (1, 3))  # Use top 3/4 of the figure for the main plot

    plt.plot(actuals[:plot_length, sensor_index], label='Actual', linewidth=2)
    plt.plot(predictions[:plot_length, sensor_index], label='Predicted', linewidth=2, alpha=0.8)

    # Format R-squared part of the title, handling potential NaN
    title_r2_part = f'R² = {r2:.4f}' if not np.isnan(r2) else 'R² = N/A'
    # Use actual sensor_id in the title
    plt.title(f'Fold {fold+1} - Actual vs Predicted Traffic Flow (Sensor {sensor_id})\n'
              f'{title_r2_part}', fontsize=14, fontweight='bold')

    # plt.xlabel(f'Time Steps (First {plot_length})')
    plt.ylabel('Speed (mph)')
    plt.legend(loc='upper right')
    plt.grid(True, alpha=0.3)
    plt.xlim(0, plot_length) # Ensure x-axis limit matches plotted data

    # --- Model Details Section ---
    plt.subplot(4, 1, 4) # Use the bottom 1/4 of the figure for model details
    plt.axis('off')     # Turn off axes for the text area
    model_details = "Configuration details not available." # Default text
    if config:
        # Format model details string if config is provided
        model_details = (
            f"Model: Transformer | Layers: {config.num_layers} | Heads: {config.num_heads} | "
            f"Hidden Dim: {config.hidden_dim} | Prediction Length: {pred_len} | \n" # Added newline for better wrapping
            f"Learning Rate: {config.learning_rate} | Optimizer: {config.optimizer_type} | "
            f"Loss: {config.loss_function} | Time Features: {config.use_time_features} | \n" # Added newline
            f"Batch Size: {config.batch_size} | Generated: {get_maputo_timestamp()}"
        )
    # Display the model details text
    plt.text(0.01, 0.9, model_details, wrap=True, fontsize=9, va='top') # Adjust vertical alignment

    # --- Save the Plot ---
    timestamp = get_maputo_timestamp()
    # Sanitize sensor_id for use in filename (replace non-alphanumeric chars with '_')
    safe_sensor_id = "".join(c if c.isalnum() else "_" for c in str(sensor_id)) # Ensure sensor_id is string
    # Construct filename using the sanitized sensor ID
    filename = f'predictions_fold{fold+1}_sensor{safe_sensor_id}_step{pred_len}_{timestamp}.png'
    save_path = os.path.join(results_dir, filename)

    try:
        plt.tight_layout(rect=[0, 0.03, 1, 0.95]) # Adjust layout slightly to prevent title overlap
        plt.savefig(save_path, dpi=300)
        # print(f"Saved plot: {save_path}") # Optional: uncomment for verbose output
    except Exception as e:
        print(f"Error saving plot {save_path}: {e}")
    finally:
        plt.close() # Ensure the figure is closed to free memory


def plot_training_history(
    train_losses: List[float],
    val_losses: List[float],
    results_dir: str,
    config: Optional[TrainingConfig] = None  # Add config parameter
) -> None:
    """
    Plots training and validation loss history with enhanced details
    """
    plt.figure(figsize=(12, 8))

    # Use 3/4 of the figure for the main plot
    plt.subplot(4, 1, (1, 3))
    plt.plot(train_losses, label='Training Loss', linewidth=2)
    plt.plot(val_losses, label='Validation Loss', linewidth=2)
    plt.title('Training History', fontsize=16, fontweight='bold')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True, alpha=0.3)

    # Calculate improvement metrics
    if len(train_losses) > 0:
        initial_loss = train_losses[0]
        final_loss = train_losses[-1]
        best_val_loss = min(val_losses) if val_losses else 0

        improvement = 100 * (initial_loss - final_loss) / initial_loss if initial_loss > 0 else 0

        # Add text annotation for improvement
        plt.annotate(
            f'Training loss reduced by {improvement:.2f}%',
            xy=(len(train_losses) * 0.6, (initial_loss + final_loss) / 2),
            xytext=(len(train_losses) * 0.4, final_loss + (initial_loss - final_loss) * 0.6),
            arrowprops=dict(facecolor='black', shrink=0.05, width=1.5, headwidth=8),
            fontsize=10
        )

    # Add model details at the bottom 1/4 of the figure
    model_details = ""
    if config:
        model_details = (
            f"Model: Transformer | Epochs: {len(train_losses)} | Batch Size: {config.batch_size} | "
            f"Learning Rate: {config.learning_rate} | Optimizer: {config.optimizer_type} | "
            f"Layers: {config.num_layers} | Heads: {config.num_heads} | Hidden Dim: {config.hidden_dim} | "
            f"Loss Function: {config.loss_function} | Scheduler: {config.scheduler_type or 'None'} | "
            f"Generated: {get_maputo_timestamp()}"
        )

    plt.subplot(4, 1, 4)
    plt.axis('off')
    plt.text(0.01, 0.5, model_details, wrap=True, fontsize=9)

    # Save with timestamp
    timestamp = get_maputo_timestamp()
    plt.tight_layout()
    plt.savefig(os.path.join(results_dir, f'training_history_{timestamp}.png'), dpi=300)
    plt.close()

def create_summary_comparison_plot(
    transformer_metrics: List[Tuple[float, float, float, float]],
    baseline_metrics: Dict[str, List[Tuple[float, float, float, float]]],
    results_dir: str,
    config: TrainingConfig
) -> None:
    """Creates a simplified comparison plot without the removed baselines"""
    plt.figure(figsize=(12, 8))

    # Calculate average metrics for transformer
    avg_transformer = np.mean(transformer_metrics, axis=0)

    # Plot only transformer metrics
    plt.subplot(2, 1, 1)
    metrics_names = ['MAE', 'RMSE', 'R²', 'MAPE']
    metrics_values = avg_transformer

    x = range(len(metrics_names))
    bars = plt.bar(x, metrics_values)
    plt.xticks(x, metrics_names)
    plt.title('Transformer Model Performance Metrics')

    # Add values on top of bars
    for bar in bars:
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                f'{height:.2f}', ha='center', va='bottom', fontsize=10)

    # Add model configuration info
    plt.subplot(2, 1, 2)
    plt.axis('off')
    model_info = (
        f"MODEL CONFIGURATION\n\n"
        f"Model: Transformer\n"
        f"Layers: {config.num_layers}\n"
        f"Attention Heads: {config.num_heads}\n"
        f"Hidden Dimension: {config.hidden_dim}\n"
        f"Prediction Length: {config.pred_length}\n"
        f"Learning Rate: {config.learning_rate}\n"
        f"Optimizer: {config.optimizer_type}\n"
        f"Loss Function: {config.loss_function}\n"
        f"Using {config.data_scaler_type} scaling"
        f"{', time features' if config.use_time_features else ''}"
    )
    plt.text(0.1, 0.9, model_info, va='top', fontsize=12)

    # Add run timestamp
    timestamp = get_maputo_timestamp()
    plt.figtext(0.5, 0.01, f"Generated: {timestamp}", ha='center', fontsize=10)

    plt.tight_layout(rect=[0, 0.03, 1, 0.97])
    plt.suptitle('Model Performance', fontsize=18, fontweight='bold')

    # Save the figure
    plt.savefig(os.path.join(results_dir, f'model_performance_{timestamp}.png'), dpi=300)
    plt.close()

## Report Generation Module

In [ ]:

# =============================================================================
# Report Generation Module
# =============================================================================
def _create_title_page(model, config, timestamp, pdf):
    """
    Creates title page for the traffic prediction report.

    Args:
        model: The trained model
        config: Training configuration
        timestamp: Timestamp for the report
        pdf: PDF object to save the page
    """
    plt.figure(figsize=(12, 8))
    plt.axis('off')

    # Title
    plt.text(0.5, 0.85, "Traffic Prediction Analysis Report",
             fontsize=24, fontweight='bold', ha='center')

    # Subtitle with timestamp
    plt.text(0.5, 0.75, f"Generated on: {timestamp}",
             fontsize=14, ha='center')

    # Model information
    model_info = f"Model: TrafficTransformer"
    if hasattr(model, 'num_layers'):
        model_info += f"\nLayers: {model.num_layers}, Heads: {model.num_heads}"
    if hasattr(model, 'hidden_dim'):
        model_info += f", Hidden Dim: {model.hidden_dim}"
    plt.text(0.5, 0.65, model_info, fontsize=12, ha='center')

    # Configuration highlights
    config_highlights = (
        f"Sequence Length: {config.seq_length}, Prediction Window: {config.pred_length}\n"
        f"Batch Size: {config.batch_size}, Learning Rate: {config.learning_rate}\n"
        f"Optimizer: {config.optimizer_type}, Loss: {config.loss_function}\n"
        f"Using Time Features: {config.use_time_features}, "
        f"Using Holiday Features: {config.use_holiday_feature}"
    )
    plt.text(0.5, 0.55, config_highlights, fontsize=12, ha='center')

    # Footer
    plt.text(0.5, 0.2, "Transformer-Based Traffic Flow Prediction",
             fontsize=16, ha='center', fontstyle='italic')

    pdf.savefig()
    plt.close()

def _create_training_analysis(train_losses, val_losses, pdf):
    """
    Creates training analysis page showing loss curves and convergence patterns.

    Args:
        train_losses: List of training losses per epoch
        val_losses: List of validation losses per epoch
        pdf: PDF object to save the page
    """
    plt.figure(figsize=(12, 8))

    # Plot training and validation loss curves
    epochs = range(1, len(train_losses) + 1)
    plt.subplot(2, 1, 1)
    plt.plot(epochs, train_losses, 'b-', label='Training Loss')
    plt.plot(epochs, val_losses, 'r-', label='Validation Loss')
    plt.title('Training and Validation Loss Curves')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True, alpha=0.3)

    # Plot convergence patterns (loss improvement rate)
    plt.subplot(2, 1, 2)
    if len(train_losses) > 5:  # Need enough epochs for moving average
        # Calculate moving average of loss improvement
        window_size = min(5, len(train_losses) // 4)
        train_improvements = [train_losses[i] - train_losses[i+window_size]
                             for i in range(len(train_losses) - window_size)]
        val_improvements = [val_losses[i] - val_losses[i+window_size]
                           for i in range(len(val_losses) - window_size)]

        # Plot improvement rates
        plt.plot(range(window_size + 1, len(train_losses) + 1),
                 train_improvements, 'b--', label='Training Improvement')
        plt.plot(range(window_size + 1, len(val_losses) + 1),
                 val_improvements, 'r--', label='Validation Improvement')
        plt.title('Loss Improvement Over Time (Higher is Better)')
        plt.xlabel('Epochs')
        plt.ylabel('Loss Reduction')
        plt.legend()
        plt.grid(True, alpha=0.3)
    else:
        # Not enough epochs for improvement analysis
        plt.text(0.5, 0.5, "Insufficient epochs for convergence analysis",
                 ha='center', va='center', fontsize=14)

    plt.tight_layout()
    pdf.savefig()
    plt.close()

    # Create additional training insights page if enough data
    if len(train_losses) > 10:
        plt.figure(figsize=(12, 8))

        # Early vs Late convergence
        plt.subplot(2, 2, 1)
        early_epochs = len(train_losses) // 3
        early_improvement = train_losses[0] - train_losses[early_epochs]
        late_improvement = train_losses[early_epochs] - train_losses[-1]

        bars = plt.bar(['Early Phase', 'Late Phase'],
                      [early_improvement, late_improvement])
        plt.title('Loss Improvement: Early vs Late Training')
        plt.ylabel('Loss Reduction')

        # Add values on bars
        for bar in bars:
            height = bar.get_height()
            plt.text(bar.get_x() + bar.get_width()/2., height + 0.001,
                    f'{height:.4f}', ha='center', va='bottom')

        # Train-Val Loss Gap
        plt.subplot(2, 2, 2)
        loss_gaps = [val - train for train, val in zip(train_losses, val_losses)]
        plt.plot(epochs, loss_gaps)
        plt.title('Validation-Training Loss Gap')
        plt.xlabel('Epochs')
        plt.ylabel('Gap')
        plt.grid(True, alpha=0.3)

        # Loss distribution
        plt.subplot(2, 2, 3)
        plt.hist(train_losses, bins=10, alpha=0.5, label='Training')
        plt.hist(val_losses, bins=10, alpha=0.5, label='Validation')
        plt.title('Loss Distribution')
        plt.xlabel('Loss Value')
        plt.ylabel('Frequency')
        plt.legend()

        # Stability analysis (loss variance in last 1/3 of training)
        plt.subplot(2, 2, 4)
        stability_start = 2 * len(train_losses) // 3
        train_stability = np.std(train_losses[stability_start:])
        val_stability = np.std(val_losses[stability_start:])

        bars = plt.bar(['Training Stability', 'Validation Stability'],
                      [train_stability, val_stability])
        plt.title('Training Stability (Lower is Better)')
        plt.ylabel('Loss Standard Deviation')

        # Add values on bars
        for bar in bars:
            height = bar.get_height()
            plt.text(bar.get_x() + bar.get_width()/2., height + 0.0001,
                    f'{height:.6f}', ha='center', va='bottom')

        plt.tight_layout()
        pdf.savefig()
        plt.close()

def _create_performance_analysis(predictions, actuals, pdf):
    """
    Creates performance analysis page showing actual vs predicted values and error analysis.

    Args:
        predictions: Predicted values
        actuals: Actual values
        pdf: PDF object to save the page
    """
    # Ensure we have proper arrays
    predictions = np.array(predictions).ravel()
    actuals = np.array(actuals).ravel()

    plt.figure(figsize=(12, 10))
    gs = GridSpec(3, 1, figure=plt.gcf())

    # 1. Actual vs Predicted Scatter Plot
    ax1 = plt.subplot(gs[0, 0])
    ax1.scatter(actuals, predictions, alpha=0.5, s=10)

    # Add perfect prediction line
    min_val = min(np.min(actuals), np.min(predictions))
    max_val = max(np.max(actuals), np.max(predictions))
    ax1.plot([min_val, max_val], [min_val, max_val], 'r--')

    ax1.set_title('Actual vs Predicted Values')
    ax1.set_xlabel('Actual')
    ax1.set_ylabel('Predicted')
    ax1.grid(True, alpha=0.3)

    # 2. Error Distribution Histogram
    ax2 = plt.subplot(gs[0, 1])
    errors = predictions - actuals
    ax2.hist(errors, bins=30, alpha=0.7)
    ax2.set_title('Error Distribution')
    ax2.set_xlabel('Prediction Error')
    ax2.set_ylabel('Frequency')
    ax2.grid(True, alpha=0.3)

    # Add mean and std as vertical lines
    mean_error = np.mean(errors)
    std_error = np.std(errors)
    ax2.axvline(mean_error, color='r', linestyle='--', label=f'Mean: {mean_error:.4f}')
    ax2.axvline(mean_error + std_error, color='g', linestyle=':', label=f'Std: {std_error:.4f}')
    ax2.axvline(mean_error - std_error, color='g', linestyle=':')
    ax2.legend()

    # 3. Predicted vs Actual Time Series (sample)
    ax3 = plt.subplot(gs[1, :])
    sample_size = min(288, len(actuals))
    indices = range(sample_size)
    ax3.plot(indices, actuals[:sample_size], 'b-', label='Actual')
    ax3.plot(indices, predictions[:sample_size], 'r-', label='Predicted')
    ax3.set_title('Actual vs Predicted (Sample Time Series)')
    ax3.set_xlabel('Time Step')
    ax3.set_ylabel('Value')
    ax3.legend()
    ax3.grid(True, alpha=0.3)

    # 4. Residual Plot
    ax4 = plt.subplot(gs[2, 0])
    ax4.scatter(actuals, errors, alpha=0.5, s=10)
    ax4.axhline(y=0, color='r', linestyle='--')
    ax4.set_title('Residual Plot')
    ax4.set_xlabel('Actual Value')
    ax4.set_ylabel('Residual (Error)')
    ax4.grid(True, alpha=0.3)

    # 5. Q-Q Plot for Error Normality
    # ax5 = plt.subplot(gs[2, 1])
    # stats.probplot(errors, dist="norm", plot=ax5)
    # ax5.set_title('Q-Q Plot of Residuals')
    # ax5.grid(True, alpha=0.3)

    # Overall metrics text
    mae = mean_absolute_error(actuals, predictions)
    rmse = np.sqrt(mean_squared_error(actuals, predictions))
    r2 = r2_score(actuals, predictions)
    mape = 100 * np.mean(np.abs((actuals - predictions) / (actuals + 1e-8)))

    metrics_text = (
        f"MAE: {mae:.4f}\n"
        f"RMSE: {rmse:.4f}\n"
        f"R²: {r2:.4f}\n"
        f"MAPE: {mape:.2f}%"
    )

    plt.figtext(0.5, 0.01, metrics_text, ha="center", fontsize=12,
               bbox={"facecolor":"orange", "alpha":0.2, "pad":5})

    plt.tight_layout(rect=[0, 0.05, 1, 0.95])  # Adjust layout to make room for text
    pdf.savefig()
    plt.close()

def generate_traffic_report(
    model: nn.Module,
    test_loader: DataLoader,
    scaler: Any,
    config: TrainingConfig,
    device: str,
    train_losses: List[float],
    val_losses: List[float],
    fold_metrics: List[Tuple[float, float, float, float]],
    baseline_metrics: Dict[str, List[Tuple[float, float, float, float]]],
    results_dir: str,
    timestamp: Optional[str] = None
) -> str:
    """
    Generate comprehensive report for traffic prediction model analysis with improved error handling

    Args:
        model: Trained model
        test_loader: Test data loader
        scaler: Scaler used for normalization
        config: Training configuration
        device: Device used for model
        train_losses: Training loss history
        val_losses: Validation loss history
        fold_metrics: Metrics for each fold
        baseline_metrics: Metrics for baseline models
        results_dir: Directory to save report
        timestamp: Optional timestamp for the report

    Returns:
        Path to generated report
    """
    timestamp = get_maputo_timestamp() if timestamp is None else timestamp
    pdf_path = os.path.join(results_dir, f'traffic_prediction_report_{timestamp}.pdf')

    # Memory cleanup before generating report
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # Set plotting style
    try:
        plt.style.use('seaborn-v0_8')
    except:
        try:
            plt.style.use('seaborn')
        except:
            plt.style.use('default')

    # Set publication-quality figure parameters
    plt.rcParams.update({
        'figure.figsize': (10, 6),
        'font.size': 12,
        'axes.labelsize': 14,
        'axes.titlesize': 16,
        'figure.titlesize': 20,
        'axes.grid': True,
        'grid.alpha': 0.3,
        'lines.linewidth': 2,
        'savefig.dpi': 300,
        'savefig.bbox': 'tight'
    })

    # Utility function for error pages
    def _create_error_page(error_message: str, pdf: PdfPages) -> None:
        """Creates an error page for the PDF when a section fails."""
        plt.figure(figsize=(12, 8))
        plt.axis('off')
        plt.text(0.5, 0.5, f"Error: {error_message}",
                ha='center', va='center', color='red', fontsize=14, wrap=True)
        pdf.savefig()
        plt.close()

    # Collect predictions and attention weights
    model.eval()
    predictions = []
    actuals = []
    attention_weights = []

    try:
        with torch.no_grad():
            for data, target in test_loader:
                data, target = data.to(device), target.to(device)
                output = model(data)

                # Store attention weights if available
                if hasattr(model, 'attention_weights') and model.attention_weights is not None:
                    # Handle different attention weight shapes
                    weights = model.attention_weights
                    if isinstance(weights, torch.Tensor):
                        weights = weights.cpu().numpy()
                    attention_weights.append(weights)

                # Store predictions and actuals
                pred = output.cpu().numpy()
                predictions.append(pred)
                actuals.append(target.cpu().numpy())

        predictions = np.concatenate(predictions)
        actuals = np.concatenate(actuals)

        # Reshape and inverse transform if needed
        if len(predictions.shape) > 2:
            predictions = predictions.reshape(-1, predictions.shape[-1])
            actuals = actuals.reshape(-1, actuals.shape[-1])
    except Exception as e:
        print(f"Error collecting predictions: {str(e)}")
        # Create minimal emergency report with available data
        _create_emergency_report(
            predictions or np.array([]),
            actuals or np.array([]),
            train_losses,
            val_losses,
            pdf_path
        )
        return pdf_path

    # Create PDF report with section-by-section error handling
    try:
        with PdfPages(pdf_path) as pdf:
            # 1. Title Page
            try:
                _create_title_page(model, config, timestamp, pdf)
            except Exception as e:
                _create_error_page(f"Error in title page: {str(e)}", pdf)
                print(f"Error in title page: {str(e)}")

            # 2. Training Analysis
            try:
                _create_training_analysis(train_losses, val_losses, pdf)
            except Exception as e:
                _create_error_page(f"Error in training analysis: {str(e)}", pdf)
                print(f"Error in training analysis: {str(e)}")

            # 3. Performance Analysis
            try:
                if len(predictions) > 0 and len(actuals) > 0:
                    _create_performance_analysis(predictions, actuals, pdf)
                else:
                    _create_error_page("Insufficient prediction data for performance analysis", pdf)
            except Exception as e:
                _create_error_page(f"Error in performance analysis: {str(e)}", pdf)
                print(f"Error in performance analysis: {str(e)}")

            # 4. Attention Analysis
            try:
                if attention_weights:
                    _create_attention_analysis(attention_weights, config, pdf)
                else:
                    _create_error_page("No attention weights available for analysis", pdf)
            except Exception as e:
                _create_error_page(f"Error in attention analysis: {str(e)}", pdf)
                print(f"Error in attention analysis: {str(e)}")

            # 5. Cross Validation Analysis
            try:
                if fold_metrics:
                    _create_cross_validation_analysis(fold_metrics, pdf)
                else:
                    _create_error_page("No fold metrics available for cross-validation analysis", pdf)
            except Exception as e:
                _create_error_page(f"Error in cross validation analysis: {str(e)}", pdf)
                print(f"Error in cross validation analysis: {str(e)}")

            # 6. Baseline Comparison
            try:
                if fold_metrics and any(metrics for metrics in baseline_metrics.values()):
                    _create_baseline_comparison(baseline_metrics, fold_metrics, pdf)
                else:
                    _create_error_page("Insufficient data for baseline comparison", pdf)
            except Exception as e:
                _create_error_page(f"Error in baseline comparison: {str(e)}", pdf)
                print(f"Error in baseline comparison: {str(e)}")

            # 7. Model Configuration
            try:
                _create_config_summary(config, pdf)
            except Exception as e:
                _create_error_page(f"Error in config summary: {str(e)}", pdf)
                print(f"Error in config summary: {str(e)}")

            # 8. Summary Statistics
            try:
                _create_summary_statistics(
                    predictions, actuals, train_losses, val_losses,
                    fold_metrics, baseline_metrics, pdf
                )
            except Exception as e:
                _create_error_page(f"Error in summary statistics: {str(e)}", pdf)
                print(f"Error in summary statistics: {str(e)}")

    except Exception as e:
        print(f"Error generating report: {str(e)}")
        # Create minimal emergency report
        _create_emergency_report(predictions, actuals, train_losses, val_losses, pdf_path)

    return pdf_path


def _create_attention_analysis(
    attention_weights: List[np.ndarray],
    config: TrainingConfig,
    pdf: PdfPages
) -> None:
    """Creates attention analysis visualizations with improved error handling."""
    plt.figure(figsize=(12, 8))

    # Guard against empty attention weights
    if not attention_weights or all(w is None for w in attention_weights):
        plt.text(0.5, 0.5, "No attention weights available",
                 ha='center', va='center', fontsize=14)
        pdf.savefig()
        plt.close()
        return

    try:
        # Handle different possible shapes of attention weights
        sample_weights = attention_weights[0]
        attention_array = np.array(attention_weights)

        # Detect shape and process accordingly
        if len(attention_array.shape) == 4:  # [batch, heads, seq, seq]
            attention_mean = np.mean(attention_array, axis=(0, 1))  # Average across batch and heads
        elif len(attention_array.shape) == 3:  # [batch, seq, seq]
            attention_mean = np.mean(attention_array, axis=0)  # Average across batch
        elif len(attention_array.shape) == 2:  # Already [seq, seq]
            attention_mean = attention_array
        else:
            # If we have a list of tensors with different shapes
            reshaped_weights = []
            for weights in attention_weights:
                if hasattr(weights, 'shape'):
                    if len(weights.shape) == 3:  # [batch, seq, seq]
                        weights = np.mean(weights, axis=0)
                    elif len(weights.shape) > 3:  # More dimensions than expected
                        weights = np.mean(weights, axis=tuple(range(len(weights.shape)-2)))
                reshaped_weights.append(weights)

            attention_mean = np.mean(reshaped_weights, axis=0)

        # Create heatmap
        sns.heatmap(attention_mean, cmap="viridis", annot=False)
        plt.title('Average Attention Weights')
        plt.xlabel('Key Position')
        plt.ylabel('Query Position')

    except Exception as e:
        # Create a fallback visualization with error message
        plt.clf()  # Clear the figure
        plt.text(0.5, 0.5, f"Error visualizing attention weights: {str(e)}\n"
                           f"Shape info: {[w.shape if hasattr(w, 'shape') else type(w) for w in attention_weights[:3]]}...",
                 ha='center', va='center', wrap=True)
        plt.title('Attention Visualization Error')

    pdf.savefig()
    plt.close()


def _create_cross_validation_analysis(
    fold_metrics: List[Tuple[float, float, float, float]],
    pdf: PdfPages
) -> None:
    """Creates cross validation analysis visualizations."""
    metrics_names = ['MAE', 'RMSE', 'R²', 'MAPE']
    metrics_values = np.array(fold_metrics)

    plt.figure(figsize=(12, 6))
    for i, metric in enumerate(metrics_names):
        plt.subplot(2, 2, i+1)
        plt.boxplot(metrics_values[:, i])
        plt.title(f'{metric} Across Folds')
        plt.grid(True, alpha=0.3)

    plt.tight_layout()
    pdf.savefig()
    plt.close()


def _create_baseline_comparison(
    baseline_metrics: Dict[str, List[Tuple[float, float, float, float]]],
    fold_metrics: List[Tuple[float, float, float, float]],
    pdf: PdfPages
) -> None:
    """Creates simplified baseline comparison visualization without ARIMA, LSTM, and ExpSmoothing."""

    plt.figure(figsize=(12, 8))
    plt.axis('off')  # Turn off axes for the message version

    # Check if there are any metrics in baseline_metrics
    if not any(metrics for metrics in baseline_metrics.values()):
        # Display message that baselines are disabled
        plt.text(0.5, 0.5,
                "Baseline models (ARIMA, LSTM, ExpSmoothing) have been disabled",
                ha='center', va='center', fontsize=14,
                bbox={'facecolor': 'lightgray', 'alpha': 0.5, 'pad': 10})
        plt.title('Baseline Comparison', fontsize=16)

        # Add a note about transformer performance
        if fold_metrics:
            avg_metrics = np.mean(fold_metrics, axis=0)
            metrics_text = (
                f"\n\nTransformer Model Metrics:\n"
                f"MAE: {avg_metrics[0]:.4f}\n"
                f"RMSE: {avg_metrics[1]:.4f}\n"
                f"R²: {avg_metrics[2]:.4f}\n"
                f"MAPE: {avg_metrics[3]:.2f}%"
            )
            plt.text(0.5, 0.3, metrics_text, ha='center', fontsize=12)
    else:
        # If there are any baseline metrics (like naive forecast), show comparison
        transformer_metrics = np.mean(fold_metrics, axis=0)
        metrics_names = ['MAE', 'RMSE', 'R²', 'MAPE']

        # Reset axis settings for plots
        plt.clf()

        # Setup subplots for metrics
        fig, axes = plt.subplots(2, 2, figsize=(12, 8))
        plt.suptitle('Transformer vs Available Baselines', fontsize=16)

        # Flatten axes for easier access
        axes = axes.flatten()

        # Get available models
        models = ['Transformer'] + list(baseline_metrics.keys())

        # Prepare data for plotting
        metrics_data = [transformer_metrics]
        for model in baseline_metrics.keys():
            if baseline_metrics[model]:
                metrics_data.append(np.mean(baseline_metrics[model], axis=0))
            else:
                metrics_data.append([np.nan, np.nan, np.nan, np.nan])

        # Create plots for each metric
        for i, (metric, ax) in enumerate(zip(metrics_names, axes)):
            metric_values = [data[i] for data in metrics_data]
            ax.bar(models, metric_values)
            ax.set_title(f'{metric} Comparison')
            ax.tick_params(axis='x', rotation=45)
            ax.grid(True, alpha=0.3)

            # Add values on bars
            for j, value in enumerate(metric_values):
                if not np.isnan(value):
                    ax.text(j, value + (max(metric_values) * 0.05),
                           f"{value:.3f}", ha='center')

    # Ensure proper layout
    plt.tight_layout()

    # Save to PDF
    pdf.savefig()
    plt.close()


def _create_config_summary(config: TrainingConfig, pdf: PdfPages) -> None:
    """Creates configuration summary page."""
    plt.figure(figsize=(12, 8))
    plt.axis('off')

    config_text = "Model Configuration:\n\n"
    for key, value in vars(config).items():
        if not key.startswith('_') and key not in ['input_dir', 'output_dir', 'model_dir', 'results_dir']:  # Skip directories for brevity
            config_text += f"{key}: {value}\n"

    plt.text(0.1, 0.9, config_text, fontsize=10, va='top')
    pdf.savefig()
    plt.close()


def _create_summary_statistics(
    predictions: np.ndarray,
    actuals: np.ndarray,
    train_losses: List[float],
    val_losses: List[float],
    fold_metrics: List[Tuple[float, float, float, float]],
    baseline_metrics: Dict[str, List[Tuple[float, float, float, float]]],
    pdf: PdfPages
) -> None:
    """Creates summary statistics page."""
    plt.figure(figsize=(12, 8))
    plt.axis('off')

    # Calculate overall metrics
    mae = mean_absolute_error(actuals.ravel(), predictions.ravel())
    rmse = np.sqrt(mean_squared_error(actuals.ravel(), predictions.ravel()))
    r2 = r2_score(actuals.ravel(), predictions.ravel())

    summary_text = f"""
    Overall Model Performance:

    Mean Absolute Error: {mae:.4f}
    Root Mean Squared Error: {rmse:.4f}
    R² Score: {r2:.4f}

    Training Summary:
    Initial Training Loss: {train_losses[0]:.6f}
    Final Training Loss: {train_losses[-1]:.6f}
    Loss Improvement: {train_losses[0] - train_losses[-1]:.6f}

    Cross-Validation Summary:
    Number of Folds: {len(fold_metrics)}
    Average MAE across folds: {np.mean([m[0] for m in fold_metrics]):.4f}
    Average RMSE across folds: {np.mean([m[1] for m in fold_metrics]):.4f}

    Baseline Comparison:
    """

    for model_name, metrics in baseline_metrics.items():
        if metrics:  # Check if metrics list is not empty
            avg_metrics = np.mean(metrics, axis=0)
            summary_text += f"\n{model_name.upper()} - MAE: {avg_metrics[0]:.4f}, RMSE: {avg_metrics[1]:.4f}"
        else:
            summary_text += f"\n{model_name.upper()} - No metrics available"

    plt.text(0.1, 0.9, summary_text, fontsize=12, va='top')
    pdf.savefig()
    plt.close()


def _create_emergency_report(
    predictions: np.ndarray,
    actuals: np.ndarray,
    train_losses: List[float],
    val_losses: List[float],
    pdf_path: str
) -> None:
    """Creates a minimal emergency report if the full report fails."""
    try:
        with PdfPages(pdf_path) as pdf:
            plt.figure(figsize=(12, 8))
            plt.axis('off')

            mae = mean_absolute_error(actuals.ravel(), predictions.ravel())
            rmse = np.sqrt(mean_squared_error(actuals.ravel(), predictions.ravel()))
            r2 = r2_score(actuals.ravel(), predictions.ravel())

            emergency_text = f"""
            Emergency Report (Error in full report generation)

            Basic Metrics:
            MAE: {mae:.4f}
            RMSE: {rmse:.4f}
            R² Score: {r2:.4f}

            Final Losses:
            Training: {train_losses[-1]:.6f}
            Validation: {val_losses[-1]:.6f}
            """
            plt.text(0.1, 0.9, emergency_text, fontsize=12, va='top')
            pdf.savefig()
            plt.close()
    except Exception as e2:
        warnings.warn(f"Emergency report also failed: {str(e2)}")

## Transfer Learning Module

In [ ]:
# =============================================================================
# Transfer Learning Module for Traffic Transformers
# =============================================================================

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, TensorDataset
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler
from typing import Dict, List, Tuple, Optional, Union, Any, Callable
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
import gc

class TransferLearningModule:
    """
    Module for transfer learning with Traffic Transformer models.
    Enables fine-tuning models pre-trained on METR-LA for African traffic datasets.
    """

    def __init__(
        self,
        base_model: nn.Module,
        config: 'TrainingConfig',
        target_dataset_name: str = 'mozambique',
        freeze_encoder: bool = True,
        freeze_layers: int = 1,
        adapter_dim: int = 64
    ):
        """
        Initialize transfer learning module

        Args:
            base_model: Pre-trained Traffic Transformer model
            config: Configuration object
            target_dataset_name: Name of target dataset ('mozambique' or 'south_africa')
            freeze_encoder: Whether to freeze encoder layers
            freeze_layers: Number of transformer layers to freeze (if freeze_encoder is True)
            adapter_dim: Dimension of adapter layers (if used)
        """
        self.base_model = base_model
        self.config = config
        self.target_dataset_name = target_dataset_name
        self.freeze_encoder = freeze_encoder
        self.freeze_layers = freeze_layers
        self.adapter_dim = adapter_dim
        self.device = next(base_model.parameters()).device

        # Apply freezing based on parameters
        self._apply_parameter_freezing()

        # Add adapter layers if needed
        self.has_adapters = adapter_dim > 0
        if self.has_adapters:
            self._add_adapter_layers()

        # Store metrics for tracking
        self.transfer_history = {
            'train_loss': [],
            'val_loss': [],
            'source_metrics': None,  # Will store metrics on source dataset
            'target_metrics': None   # Will store metrics on target dataset
        }

    def _apply_parameter_freezing(self):
        """Apply freezing to specified parts of the model"""
        if self.freeze_encoder:
            # Freeze embedding layer
            for param in self.base_model.embedding.parameters():
                param.requires_grad = False

            # Freeze positional encoding (not trainable by default, but being explicit)
            if hasattr(self.base_model.pos_encoder, 'pe'):
                self.base_model.pos_encoder.pe.requires_grad = False

            # Freeze specified transformer layers
            for i, layer in enumerate(self.base_model.transformer):
                if i < self.freeze_layers:
                    for param in layer.parameters():
                        param.requires_grad = False

            print(f"Froze embedding layer and {self.freeze_layers} transformer layers")
        else:
            print("No parameter freezing applied - full fine-tuning")

    def _add_adapter_layers(self):
        """
        Add adapter layers to the model for more efficient transfer learning.
        Adapters are small bottleneck layers added after transformer layers.
        """
        hidden_dim = self.base_model.transformer[0].linear1.out_features

        # Create adapter layers
        self.down_adapters = nn.ModuleList([
            nn.Linear(hidden_dim, self.adapter_dim)
            for _ in range(len(self.base_model.transformer) - self.freeze_layers)
        ])

        self.up_adapters = nn.ModuleList([
            nn.Linear(self.adapter_dim, hidden_dim)
            for _ in range(len(self.base_model.transformer) - self.freeze_layers)
        ])

        # Add activations
        self.adapter_act = nn.GELU()

        # Patch the forward method of non-frozen transformer layers
        for i in range(self.freeze_layers, len(self.base_model.transformer)):
            layer = self.base_model.transformer[i]
            adapter_idx = i - self.freeze_layers

            # Store the original forward method
            orig_forward = layer.forward

            # Define the new forward method with adapter
            def make_new_forward(orig_f, a_idx):
                def new_forward(x):
                    # Call original forward pass
                    output = orig_f(x)

                    # Apply adapter (down projection → activation → up projection)
                    residual = output
                    output = self.down_adapters[a_idx](output)
                    output = self.adapter_act(output)
                    output = self.up_adapters[a_idx](output)

                    # Residual connection
                    output = output + residual
                    return output

                return new_forward

            # Replace the forward method
            layer.forward = make_new_forward(orig_forward, adapter_idx)

        print(f"Added adapter layers with dimension {self.adapter_dim}")

    def load_african_dataset(
        self,
        file_path: str,
        sequence_length: Optional[int] = None,
        prediction_length: Optional[int] = None,
        test_split: float = 0.2
    ) -> Tuple[DataLoader, DataLoader, Any]:
        """
        Load and prepare an African traffic dataset for transfer learning

        Args:
            file_path: Path to the dataset CSV file
            sequence_length: Length of input sequence (defaults to config value)
            prediction_length: Length of prediction window (defaults to config value)
            test_split: Portion of data to use for testing

        Returns:
            Tuple of (train_loader, test_loader, scaler)
        """
        seq_length = sequence_length or self.config.seq_length
        pred_length = prediction_length or self.config.pred_length

        try:
            # Load dataset
            df = pd.read_csv(file_path, parse_dates=True, index_col=0)
            print(f"Loaded dataset with shape: {df.shape}")

            # Handle null values
            df.replace(0.0, np.nan, inplace=True)
            df.ffill(inplace=True)
            df.bfill(inplace=True)

            # Get timestamps and data
            timestamps = df.index
            sensor_data = df.values
            num_features = sensor_data.shape[1]

            # Scale data
            if self.config.data_scaler_type == 'minmax':
                data_scaler = MinMaxScaler()
            elif self.config.data_scaler_type == 'standard':
                data_scaler = StandardScaler()
            elif self.config.data_scaler_type == 'robust':
                data_scaler = RobustScaler()
            else:
                warnings.warn(f"Invalid scaler type: {self.config.data_scaler_type}, using MinMaxScaler")
                data_scaler = MinMaxScaler()

            data_normalized = data_scaler.fit_transform(sensor_data)

            # Split into train and test sets
            total_samples = len(data_normalized)
            test_size = int(total_samples * test_split)
            train_size = total_samples - test_size

            train_data = data_normalized[:train_size]
            test_data = data_normalized[train_size:]
            train_times = timestamps[:train_size]
            test_times = timestamps[train_size:]

            # Create datasets
            train_dataset = TrafficDataset(
                train_data, train_times, seq_length, pred_length, self.config
            )
            test_dataset = TrafficDataset(
                test_data, test_times, seq_length, pred_length, self.config
            )

            # Create data loaders
            batch_size = min(32, self.config.batch_size)  # Smaller batch size for smaller datasets

            train_loader = DataLoader(
                train_dataset,
                batch_size=batch_size,
                shuffle=True,
                num_workers=min(2, self.config.num_workers),  # Reduce workers for smaller datasets
                pin_memory=self.config.pin_memory
            )

            test_loader = DataLoader(
                test_dataset,
                batch_size=batch_size,
                shuffle=False,
                num_workers=min(2, self.config.num_workers),
                pin_memory=self.config.pin_memory
            )

            return train_loader, test_loader, data_scaler

        except Exception as e:
            raise RuntimeError(f"Error loading African dataset: {str(e)}")

    def fine_tune(
        self,
        train_loader: DataLoader,
        val_loader: DataLoader,
        learning_rate: float = 1e-4,
        num_epochs: int = 30,
        patience: int = 5,
        output_dir: Optional[str] = None
    ) -> nn.Module:
        """
        Fine-tune the model on a new dataset

        Args:
            train_loader: DataLoader with training data
            val_loader: DataLoader with validation data
            learning_rate: Learning rate for fine-tuning
            num_epochs: Maximum number of epochs
            patience: Early stopping patience
            output_dir: Directory to save checkpoints and results

        Returns:
            Fine-tuned model
        """
        model = self.base_model
        model.train()

        # Only optimize parameters that require gradients
        optimizer = optim.AdamW(
            [p for p in model.parameters() if p.requires_grad],
            lr=learning_rate
        )

        scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', patience=patience//2, factor=0.5
        )

        criterion = nn.MSELoss()
        best_loss = float('inf')
        no_improve = 0
        train_losses = []
        val_losses = []

        # Training loop
        for epoch in range(num_epochs):
            model.train()
            epoch_loss = 0

            for batch_idx, (data, target) in enumerate(train_loader):
                data, target = data.to(self.device), target.to(self.device)

                optimizer.zero_grad()
                output = model(data)
                loss = criterion(output, target)
                loss.backward()

                # Gradient clipping
                if self.config.gradient_clip:
                    nn.utils.clip_grad_norm_(model.parameters(), self.config.gradient_clip)

                optimizer.step()
                epoch_loss += loss.item()

            avg_train_loss = epoch_loss / len(train_loader)
            train_losses.append(avg_train_loss)

            # Validation
            model.eval()
            val_loss = 0

            with torch.no_grad():
                for data, target in val_loader:
                    data, target = data.to(self.device), target.to(self.device)
                    output = model(data)
                    loss = criterion(output, target)
                    val_loss += loss.item()

            avg_val_loss = val_loss / len(val_loader)
            val_losses.append(avg_val_loss)

            # Update learning rate scheduler
            scheduler.step(avg_val_loss)

            # Early stopping check
            if avg_val_loss < best_loss:
                best_loss = avg_val_loss
                no_improve = 0

                # Save best model if output_dir is provided
                if output_dir:
                    os.makedirs(output_dir, exist_ok=True)
                    torch.save({
                        'epoch': epoch,
                        'model_state_dict': model.state_dict(),
                        'optimizer_state_dict': optimizer.state_dict(),
                        'loss': best_loss,
                        'target_dataset': self.target_dataset_name
                    }, os.path.join(output_dir, f'fine_tuned_{self.target_dataset_name}_best.pth'))
            else:
                no_improve += 1
                if no_improve >= patience:
                    print(f'Early stopping at epoch {epoch+1}')
                    break

            print(f'Epoch {epoch+1}/{num_epochs} | Train Loss: {avg_train_loss:.6f} | Val Loss: {avg_val_loss:.6f}')

        # Store training history
        self.transfer_history['train_loss'] = train_losses
        self.transfer_history['val_loss'] = val_losses

        # Plot training history if output_dir is provided
        if output_dir:
            self._plot_transfer_learning_curve(train_losses, val_losses, output_dir)

        return model

    def evaluate_transfer(
        self,
        source_loader: DataLoader,
        target_loader: DataLoader,
        source_scaler: Any,
        target_scaler: Any,
        output_dir: Optional[str] = None
    ) -> Dict[str, Dict[str, float]]:
        """
        Evaluate transfer learning performance on both source and target datasets

        Args:
            source_loader: DataLoader with source dataset
            target_loader: DataLoader with target dataset
            source_scaler: Scaler for source data
            target_scaler: Scaler for target data
            output_dir: Directory to save results

        Returns:
            Dictionary with evaluation metrics
        """
        model = self.base_model
        model.eval()

        # Evaluate on source dataset
        source_metrics = self._evaluate_on_dataset(model, source_loader, source_scaler, "METR-LA")

        # Evaluate on target dataset
        target_metrics = self._evaluate_on_dataset(model, target_loader, target_scaler, self.target_dataset_name)

        # Store metrics in history
        self.transfer_history['source_metrics'] = source_metrics
        self.transfer_history['target_metrics'] = target_metrics

        # Compare and visualize if output_dir is provided
        if output_dir:
            self._plot_transfer_comparison(source_metrics, target_metrics, output_dir)

        # Return both metrics
        return {
            'source': source_metrics,
            'target': target_metrics
        }

    def _evaluate_on_dataset(
        self,
        model: nn.Module,
        dataloader: DataLoader,
        scaler: Any,
        dataset_name: str
    ) -> Dict[str, float]:
        """Helper method to evaluate model on a dataset"""
        model.eval()
        all_preds = []
        all_targets = []

        with torch.no_grad():
            for data, target in dataloader:
                data, target = data.to(self.device), target.to(self.device)
                output = model(data)

                all_preds.append(output.cpu().numpy())
                all_targets.append(target.cpu().numpy())

        predictions = np.concatenate(all_preds)
        actuals = np.concatenate(all_targets)

        # Reshape for inverse transformation
        num_samples, pred_window, num_features = predictions.shape
        predictions_2d = predictions.reshape(-1, num_features)
        actuals_2d = actuals.reshape(-1, num_features)

        # Inverse transform
        predictions_inv = scaler.inverse_transform(predictions_2d)
        actuals_inv = scaler.inverse_transform(actuals_2d)

        # Calculate metrics
        mae = mean_absolute_error(actuals_inv.ravel(), predictions_inv.ravel())
        rmse = np.sqrt(mean_squared_error(actuals_inv.ravel(), predictions_inv.ravel()))
        r2 = r2_score(actuals_inv.ravel(), predictions_inv.ravel())

        # Calculate MAPE with handling for zeros
        mape = np.mean(np.abs((actuals_inv.ravel() - predictions_inv.ravel()) /
                               np.maximum(np.abs(actuals_inv.ravel()), 1e-10))) * 100

        print(f'Evaluation on {dataset_name}: MAE={mae:.4f}, RMSE={rmse:.4f}, R²={r2:.4f}, MAPE={mape:.2f}%')

        return {'mae': mae, 'rmse': rmse, 'r2': r2, 'mape': mape}

    def _plot_transfer_learning_curve(
        self,
        train_losses: List[float],
        val_losses: List[float],
        output_dir: str
    ):
        """Plot transfer learning curves"""
        plt.figure(figsize=(10, 6))
        plt.plot(train_losses, label='Training Loss')
        plt.plot(val_losses, label='Validation Loss')
        plt.title(f'Transfer Learning to {self.target_dataset_name.title()} Dataset')
        plt.xlabel('Epochs')
        plt.ylabel('Loss')
        plt.legend()
        plt.grid(True, alpha=0.3)

        # Add details about freezing and adapters
        plt.annotate(
            f"Frozen layers: {self.freeze_layers if self.freeze_encoder else 'None'}\n"
            f"Adapters: {'Yes' if self.has_adapters else 'No'}",
            xy=(0.02, 0.02), xycoords='figure fraction'
        )

        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, f'transfer_learning_curve_{self.target_dataset_name}.png'), dpi=300)
        plt.close()

    def _plot_transfer_comparison(
        self,
        source_metrics: Dict[str, float],
        target_metrics: Dict[str, float],
        output_dir: str
    ):
        """Plot comparison between source and target dataset performance"""
        metrics = ['mae', 'rmse', 'r2', 'mape']
        source_values = [source_metrics[m] for m in metrics]
        target_values = [target_metrics[m] for m in metrics]

        # For better visualization, normalize R² separately (higher is better)
        if metrics.index('r2') == 2:  # If r2 is the third metric
            # Invert R² so lower is better for visualization consistency
            source_values[2] = 1 - source_values[2]
            target_values[2] = 1 - target_values[2]
            metrics[2] = 'r2 (inverted)'

        plt.figure(figsize=(12, 8))

        x = range(len(metrics))
        width = 0.35

        plt.bar([i - width/2 for i in x], source_values, width, label='METR-LA (Source)')
        plt.bar([i + width/2 for i in x], target_values, width, label=f'{self.target_dataset_name.title()} (Target)')

        plt.xlabel('Metrics')
        plt.ylabel('Value (Lower is Better)')
        plt.title('Transfer Learning Performance Comparison')
        plt.xticks(x, metrics)
        plt.legend()
        plt.grid(True, alpha=0.3)

        # Add text showing improvement/degradation percentages
        for i, (source, target) in enumerate(zip(source_values, target_values)):
            if metrics[i] != 'r2 (inverted)':
                change_pct = (target - source) / source * 100
                color = 'green' if change_pct < 0 else 'red'
                plt.annotate(
                    f"{change_pct:.1f}%",
                    xy=(i, max(source, target) * 1.05),
                    ha='center',
                    color=color
                )

        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, f'transfer_comparison_{self.target_dataset_name}.png'), dpi=300)
        plt.close()

    @staticmethod
    def visualize_attention_transfer(
        base_model: nn.Module,
        source_loader: DataLoader,
        target_loader: DataLoader,
        output_dir: str
    ):
        """
        Visualize and compare attention patterns between source and target datasets

        Args:
            base_model: The fine-tuned model
            source_loader: DataLoader with source dataset
            target_loader: DataLoader with target dataset
            output_dir: Directory to save visualizations
        """
        device = next(base_model.parameters()).device
        base_model.eval()

        # Get sample batch from each dataset
        source_batch = next(iter(source_loader))
        target_batch = next(iter(target_loader))

        # Function to extract attention weights from a batch
        def get_attention_weights(batch):
            data, _ = batch
            data = data.to(device)

            # Clear any previous attention weights
            if hasattr(base_model, 'attention_weights'):
                base_model.attention_weights = None

            # Forward pass to capture attention weights
            with torch.no_grad():
                _ = base_model(data)

            # Get attention weights
            if hasattr(base_model, 'attention_weights') and base_model.attention_weights is not None:
                weights = base_model.attention_weights
                if isinstance(weights, torch.Tensor):
                    weights = weights.cpu().numpy()
                return weights
            return None

        # Get attention weights for both datasets
        source_attention = get_attention_weights(source_batch)
        target_attention = get_attention_weights(target_batch)

        if source_attention is not None and target_attention is not None:
            plt.figure(figsize=(15, 6))

            # Plot source attention
            plt.subplot(1, 3, 1)
            sns.heatmap(source_attention, cmap='viridis')
            plt.title('Source Dataset (METR-LA)\nAttention Pattern')
            plt.xlabel('Key Position')
            plt.ylabel('Query Position')

            # Plot target attention
            plt.subplot(1, 3, 2)
            sns.heatmap(target_attention, cmap='viridis')
            plt.title('Target Dataset\nAttention Pattern')
            plt.xlabel('Key Position')
            plt.ylabel('Query Position')

            # Plot difference
            plt.subplot(1, 3, 3)
            diff = target_attention - source_attention
            sns.heatmap(diff, cmap='coolwarm', center=0)
            plt.title('Attention Difference\n(Target - Source)')
            plt.xlabel('Key Position')
            plt.ylabel('Query Position')

            plt.tight_layout()
            plt.savefig(os.path.join(output_dir, 'attention_transfer_comparison.png'), dpi=300)
            plt.close()
        else:
            print("Could not extract attention weights for visualization")

## Main Execution

In [ ]:
# Monkey Patch
# Complete replacement of evaluate_model and predict functions

def fixed_evaluate_model(
    model,
    dataloader,
    criterion,
    data_scaler,
    device='cpu',
    config=None,
    adjacency_matrix=None
):
    """Fixed version of evaluate_model that correctly handles dictionary inputs"""
    print("=== Using patched evaluate_model function ===")
    model.eval()
    total_loss = 0
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for batch_data in dataloader:
            # Handle different data formats
            if isinstance(batch_data, (tuple, list)) and len(batch_data) == 2:
                data, target = batch_data

                # Proper handling of dictionary data
                if isinstance(data, dict):
                    data = {k: v.to(device) for k, v in data.items()}
                else:
                    data = data.to(device)
                target = target.to(device)
            else:
                print(f"Unexpected batch_data format: {type(batch_data)}")
                continue

            # Run model forward pass
            output = model(data)
            loss = criterion(output, target)

            total_loss += loss.item()
            all_preds.append(output.cpu().numpy())
            all_targets.append(target.cpu().numpy())

    # After we're done, check if we have any predictions
    if not all_preds:
        print("No predictions were made! Check the error messages above.")
        # Return dummy values
        return 999.0, (999.0, 999.0, 0.0, 999.0)

    predictions = np.concatenate(all_preds)
    actuals = np.concatenate(all_targets)

    # Reshape for inverse transformation
    num_samples, pred_window, num_features = predictions.shape
    predictions_2d = predictions.reshape(-1, num_features)
    actuals_2d = actuals.reshape(-1, num_features)

    # Inverse transform
    predictions_inv = data_scaler.inverse_transform(predictions_2d)
    actuals_inv = data_scaler.inverse_transform(actuals_2d)

    # Calculate metrics
    mae = mean_absolute_error(actuals_inv.ravel(), predictions_inv.ravel())
    rmse = np.sqrt(mean_squared_error(actuals_inv.ravel(), predictions_inv.ravel()))
    r2 = r2_score(actuals_inv.ravel(), predictions_inv.ravel())
    mape = robust_mape(actuals_inv.ravel(), predictions_inv.ravel())

    print(f'MAE: {mae:.2f}, RMSE: {rmse:.2f}, R²: {r2:.2f}, MAPE: {mape:.2f}%')

    return total_loss / len(dataloader), (mae, rmse, r2, mape)

def fixed_predict(
    model,
    dataloader,
    scaler,
    device='cpu',
    adjacency_matrix=None,
    config=None
):
    """Fixed version of predict function that correctly handles dictionary inputs"""
    print("=== Using patched predict function ===")
    model.eval()
    all_preds = []
    all_targets = []

    # Use AMP for prediction if available on GPU
    use_amp = hasattr(torch.cuda, 'amp') and device != 'cpu'

    with torch.no_grad():
        for batch_data in dataloader:
            data, target = batch_data

            # Proper handling of dictionary data
            if isinstance(data, dict):
                data = {k: v.to(device) for k, v in data.items()}
            else:
                data = data.to(device)
            target = target.to(device)

            if use_amp:
                if DEVICE_TYPE_SUPPORTED:
                    with autocast(device_type='cuda'):
                        # For prediction, don't use teacher forcing
                        if hasattr(dataloader.dataset, 'config') and dataloader.dataset.config.use_spatial_features and dataloader.dataset.config.use_gnn_pre_transformer:
                            output = model(data, adjacency_matrix=adjacency_matrix.to(device))
                        else:
                            output = model(data)
                else:
                    # Older PyTorch versions
                    with autocast():
                        # For prediction, don't use teacher forcing
                        if hasattr(dataloader.dataset, 'config') and dataloader.dataset.config.use_spatial_features and dataloader.dataset.config.use_gnn_pre_transformer:
                            output = model(data, adjacency_matrix=adjacency_matrix.to(device))
                        else:
                            output = model(data)
            else:
                # For prediction, don't use teacher forcing
                if hasattr(dataloader.dataset, 'config') and dataloader.dataset.config.use_spatial_features and dataloader.dataset.config.use_gnn_pre_transformer:
                    output = model(data, adjacency_matrix=adjacency_matrix.to(device))
                else:
                    output = model(data)

            all_preds.append(output.cpu().numpy())
            all_targets.append(target.cpu().numpy())

    predictions = np.concatenate(all_preds)
    actuals = np.concatenate(all_targets)

    # Reshape for inverse transformation
    num_samples, pred_window, num_features = predictions.shape
    predictions_2d = predictions.reshape(-1, num_features)
    actuals_2d = actuals.reshape(-1, num_features)

    # Inverse transform
    predictions_inv = scaler.inverse_transform(predictions_2d)
    actuals_inv = scaler.inverse_transform(actuals_2d)

    return predictions_inv, actuals_inv

# Replace both the evaluate_model and predict functions with our fixed versions
import sys
current_module = sys.modules[__name__]

# Store originals for potential restoration
original_evaluate_model = current_module.evaluate_model
original_predict = current_module.predict

# Replace with fixed versions
current_module.evaluate_model = fixed_evaluate_model
current_module.predict = fixed_predict

print("Successfully patched both evaluate_model and predict functions")

In [ ]:
# =============================================================================
# Main Execution with Full Transfer Learning Support
# =============================================================================

def main():
    """Main execution function with enhanced transfer learning support"""
    # --- Load Configuration ---
    config_path = '' #'/content/drive/Shareddrives/Almo-2002-R&D/1 - RD-Traffic-Prediction/config.yaml'
    if os.path.exists(config_path):
        config = load_config(config_path)
        print("Configuration loaded from config.yaml")
    else:
        print("config.yaml not found, using default parameters.")
        config = TrainingConfig(base_output_dir='/content/drive/Shareddrives/Almo-2002-R&D/1 - RD-Traffic-Prediction/Transformer_Versions/COLAB_NOBASELINE_V14-6')

    # Check if transfer learning is enabled
    is_transfer_learning = hasattr(config, 'enable_transfer_learning') and config.enable_transfer_learning
    # Check if we should run ONLY transfer learning
    run_only_transfer = hasattr(config, 'run_only_transfer_learning') and config.run_only_transfer_learning

    # --- Setup Directories ---
    input_dir, output_dir, model_dir, results_dir = setup_directories(config)

    # --- Mount Google Drive if in Colab ---
    if IN_COLAB:
        try:
            drive.mount('/content/drive')
            print("Google Drive mounted successfully")
        except:
            warnings.warn("Failed to mount Google Drive, using local directories")

    # --- Set Device ---
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    # Start GPU memory monitoring
    monitor_stop_flag = start_gpu_memory_monitor(config, interval=15)

    try:
        if is_transfer_learning:
            # =====================================================
            # TRANSFER LEARNING PATH
            # =====================================================
            print("\n=== Running Transfer Learning Workflow ===\n")

            # 1. Load source (METR-LA) dataset for creating model and comparison
            try:
                source_df = pd.read_csv(os.path.join(input_dir, 'METR-LA.csv'), index_col=0, parse_dates=True)
                print(f"Loaded METR-LA data with shape: {source_df.shape}")

                # Store sensor IDs for later use in plotting
                source_sensor_ids = source_df.columns.tolist()

                # Prepare source data
                source_data, source_timestamps, source_scaler, source_num_features = prepare_data(source_df, config)

                # Create test dataset for source data (for evaluation comparison)
                source_test_size = int(len(source_data) * 0.2)
                source_test_data = source_data[-source_test_size:]
                source_test_times = source_timestamps[-source_test_size:]

                source_test_dataset = TrafficDataset(
                    source_test_data, source_test_times, config.seq_length, config.pred_length, config
                )

                source_test_loader = DataLoader(
                    source_test_dataset,
                    batch_size=config.batch_size,
                    shuffle=False,
                    num_workers=config.num_workers,
                    pin_memory=config.pin_memory
                )

                print(f"Prepared source test dataset with {len(source_test_dataset)} samples")

                # Set up source adjacency matrix if needed
                if config.use_spatial_features and config.use_gnn_pre_transformer:
                    min_data_len_adj = config.seq_length + config.pred_length
                    dataset_instance_adj = TrafficDataset(
                        source_data[:min_data_len_adj],
                        source_timestamps[:min_data_len_adj],
                        config.seq_length,
                        config.pred_length,
                        config
                    )
                    source_adjacency_matrix = dataset_instance_adj.get_adjacency_matrix()
                else:
                    source_adjacency_matrix = None

            except FileNotFoundError:
                raise FileNotFoundError(f"METR-LA.csv not found in {input_dir}, required for transfer learning")

            # 2. Load target dataset (Mozambique or South Africa)
            target_dataset_name = getattr(config, 'target_dataset_name', 'mozambique')
            target_data_path = getattr(config, 'target_data_path', f'{target_dataset_name}_traffic.csv')

            try:
                target_data_full_path = os.path.join(input_dir, target_data_path)
                if not os.path.exists(target_data_full_path):
                    raise FileNotFoundError(f"Target dataset file {target_data_path} not found in {input_dir}")

                target_df = pd.read_csv(target_data_full_path, index_col=0, parse_dates=True)
                print(f"Loaded {target_dataset_name} data with shape: {target_df.shape}")

                # Store target sensor IDs
                target_sensor_ids = target_df.columns.tolist()

                # Prepare target data
                target_data, target_timestamps, target_scaler, target_num_features = prepare_data(target_df, config)

                # Split into train and validation sets
                target_val_size = int(len(target_data) * 0.2)
                target_train_data = target_data[:-target_val_size]
                target_val_data = target_data[-target_val_size:]
                target_train_times = target_timestamps[:-target_val_size]
                target_val_times = target_timestamps[-target_val_size:]

                # Create datasets
                target_train_dataset = TrafficDataset(
                    target_train_data, target_train_times, config.seq_length, config.pred_length, config
                )

                target_val_dataset = TrafficDataset(
                    target_val_data, target_val_times, config.seq_length, config.pred_length, config
                )

                # Create data loaders
                target_train_loader = DataLoader(
                    target_train_dataset,
                    batch_size=config.batch_size,
                    shuffle=True,
                    num_workers=config.num_workers,
                    pin_memory=config.pin_memory
                )

                target_val_loader = DataLoader(
                    target_val_dataset,
                    batch_size=config.batch_size,
                    shuffle=False,
                    num_workers=config.num_workers,
                    pin_memory=config.pin_memory
                )

                print(f"Prepared target datasets - Train: {len(target_train_dataset)}, Val: {len(target_val_dataset)}")

                # Set up target adjacency matrix if needed (could reuse source if structure matches)
                if config.use_spatial_features and config.use_gnn_pre_transformer:
                    # For simplicity, reuse source adjacency matrix if dimensions match
                    if source_adjacency_matrix is not None and source_adjacency_matrix.shape[0] == target_df.shape[1]:
                        target_adjacency_matrix = source_adjacency_matrix
                        print("Reusing source adjacency matrix for target dataset")
                    else:
                        # Create new adjacency matrix
                        min_data_len_adj = config.seq_length + config.pred_length
                        dataset_instance_adj = TrafficDataset(
                            target_data[:min_data_len_adj],
                            target_timestamps[:min_data_len_adj],
                            config.seq_length,
                            config.pred_length,
                            config
                        )
                        target_adjacency_matrix = dataset_instance_adj.get_adjacency_matrix()
                        print("Created new adjacency matrix for target dataset")
                else:
                    target_adjacency_matrix = None

            except Exception as e:
                raise RuntimeError(f"Error loading target dataset: {str(e)}")

            # 3. Initialize model parameters
            model_params = {
                'input_dim': source_test_dataset[0][0].shape[1],  # Use sample from source dataset
                'hidden_dim': config.hidden_dim,
                'num_layers': config.num_layers,
                'num_heads': config.num_heads,
                'num_features': source_num_features,
                'dropout': config.dropout,
                'ff_dim_multiplier': config.ff_dim_multiplier,
                'activation': config.activation,
                'decoder_type': config.decoder_type,
                'use_gnn_pre_transformer': config.use_gnn_pre_transformer,
                'spatial_feature_dim': config.spatial_feature_dim,
                'gnn_type': config.gnn_type,
                'pred_len': config.pred_length,
            }

            # Ensure model dimensions are compatible
            model_params, config = ensure_compatible_dimensions(model_params, config)

            # 4. Load pre-trained model
            source_model_path = getattr(config, 'source_model_path', None)
            if source_model_path:
                pretrained_path = os.path.join(model_dir, source_model_path)
            else:
                # Try to find the most recent best model file
                model_files = [f for f in os.listdir(model_dir) if f.startswith('best_model_') and f.endswith('.pth')]
                if model_files:
                    # Sort by modification time (newest first)
                    model_files.sort(key=lambda x: os.path.getmtime(os.path.join(model_dir, x)), reverse=True)
                    pretrained_path = os.path.join(model_dir, model_files[0])
                else:
                    raise FileNotFoundError("No pre-trained model found. Please specify source_model_path or ensure a best_model_*.pth file exists.")

            try:
                checkpoint = torch.load(pretrained_path, map_location=device)
                model = TrafficTransformer(**model_params).to(device)
                model.load_state_dict(checkpoint['model_state_dict'])
                print(f"Loaded pre-trained model from {pretrained_path}")
            except Exception as e:
                raise RuntimeError(f"Error loading pre-trained model: {str(e)}")

            # 5. Set up transfer learning configuration
            freeze_encoder = getattr(config, 'freeze_encoder', True)
            freeze_layers = getattr(config, 'freeze_layers', 1)
            adapter_dim = getattr(config, 'adapter_dim', 0)

            # Initialize transfer learning module
            transfer_module = TransferLearningModule(
                base_model=model,
                config=config,
                target_dataset_name=target_dataset_name,
                freeze_encoder=freeze_encoder,
                freeze_layers=freeze_layers,
                adapter_dim=adapter_dim
            )

            # 6. Fine-tune the model
            print("\n=== Starting Transfer Learning Fine-tuning ===")
            fine_tuned_model, history, metrics = train_transfer_model(
                model=model,
                train_loader=target_train_loader,
                val_loader=target_val_loader,
                source_loader=source_test_loader,
                optimizer=None,  # Will be created in the function
                scheduler=None,  # Will be created in the function
                criterion=None,  # Will be created in the function
                config=config,
                data_scaler=target_scaler,
                source_scaler=source_scaler,
                device=device,
                adjacency_matrix=target_adjacency_matrix
            )

            # 7. Generate visualizations for specific sensors
            print("\n=== Generating Sensor-specific Visualizations ===")

            # Make predictions on target validation set
            print("Generating predictions for target dataset...")
            target_predictions, target_actuals = predict(
                model=fine_tuned_model,
                dataloader=target_val_loader,
                scaler=target_scaler,
                device=device,
                adjacency_matrix=target_adjacency_matrix,
                config=config
            )

            # Plot predictions for a few target sensors
            for sensor_idx in range(min(10, target_actuals.shape[1])):
                if sensor_idx < len(target_sensor_ids):
                    actual_sensor_id = target_sensor_ids[sensor_idx]
                    plot_predictions_vs_actual(
                        actuals=target_actuals,
                        predictions=target_predictions,
                        sensor_index=sensor_idx,
                        sensor_id=actual_sensor_id,
                        fold=0,  # Not using folds for transfer learning
                        results_dir=results_dir,
                        pred_len=config.pred_length,
                        config=config
                    )

            # Make predictions on source test set
            print("Generating predictions for source dataset...")
            source_predictions, source_actuals = predict(
                model=fine_tuned_model,
                dataloader=source_test_loader,
                scaler=source_scaler,
                device=device,
                adjacency_matrix=source_adjacency_matrix,
                config=config
            )

            # Plot attention weights (using model from current fold)
            plot_attention_weights(
                fine_tuned_model,  # This would be the fine-tuned model in transfer learning
                config.seq_length,
                results_dir,
                config,
                show_feature_groups=True  # Enable feature group visualization
            )

            # 8. Generate summary report
            print("\n=== Generating Transfer Learning Summary Report ===")

            # Create a comparison dataframe
            if metrics['source'] and metrics['target']:
                report_data = {
                    'Dataset': ['METR-LA (Source)', f'{target_dataset_name.title()} (Target)'],
                    'MAE': [metrics['source']['mae'], metrics['target']['mae']],
                    'RMSE': [metrics['source']['rmse'], metrics['target']['rmse']],
                    'R²': [metrics['source']['r2'], metrics['target']['r2']],
                    'MAPE (%)': [metrics['source']['mape'], metrics['target']['mape']]
                }

                # Calculate improvement percentages
                improvement = {
                    'MAE': (metrics['target']['mae'] - metrics['source']['mae']) / metrics['source']['mae'] * 100,
                    'RMSE': (metrics['target']['rmse'] - metrics['source']['rmse']) / metrics['source']['rmse'] * 100,
                    'R²': (metrics['target']['r2'] - metrics['source']['r2']) / max(0.001, abs(metrics['source']['r2'])) * 100,
                    'MAPE': (metrics['target']['mape'] - metrics['source']['mape']) / max(0.001, metrics['source']['mape']) * 100
                }

                # Add a row for improvement percentages
                report_data['Dataset'].append('Improvement (%)')
                report_data['MAE'].append(improvement['MAE'])
                report_data['RMSE'].append(improvement['RMSE'])
                report_data['R²'].append(improvement['R²'])
                report_data['MAPE (%)'].append(improvement['MAPE'])

                # Create dataframe and save to CSV
                df_report = pd.DataFrame(report_data)
                timestamp = get_maputo_timestamp()
                report_path = os.path.join(results_dir, f'transfer_learning_report_{target_dataset_name}_{timestamp}.csv')
                df_report.to_csv(report_path, index=False)

                print(f"Transfer learning summary report saved to {report_path}")

                # Print summary to console
                print("\n=== TRANSFER LEARNING SUMMARY ===")
                print(f"Source dataset: METR-LA, Target dataset: {target_dataset_name.title()}")
                print(f"MAE  - Source: {metrics['source']['mae']:.4f}, Target: {metrics['target']['mae']:.4f}, Change: {improvement['MAE']:.2f}%")
                print(f"RMSE - Source: {metrics['source']['rmse']:.4f}, Target: {metrics['target']['rmse']:.4f}, Change: {improvement['RMSE']:.2f}%")
                print(f"R²   - Source: {metrics['source']['r2']:.4f}, Target: {metrics['target']['r2']:.4f}, Change: {improvement['R²']:.2f}%")
                print(f"MAPE - Source: {metrics['source']['mape']:.2f}%, Target: {metrics['target']['mape']:.2f}%, Change: {improvement['MAPE']:.2f}%")

            else:
                print("Incomplete metrics for source or target dataset, cannot generate complete report")

        elif not run_only_transfer:  # Skip standard training if run_only_transfer is True
            # =====================================================
            # STANDARD TRAINING PATH
            # =====================================================
            print("\n=== Running Standard Training Workflow ===\n")

            # --- Load and Preprocess Data ---
            try:
                # Load the main dataset based on the dataset_name in config
                dataset_filename = f"{config.dataset_name}.csv"

                try:
                    # First try with a specific format
                    df = pd.read_csv(os.path.join(input_dir, dataset_filename), index_col=0,
                                    parse_dates=True, date_format='%Y-%m-%d %H:%M:%S')
                    print(f"Successfully loaded {dataset_filename} with date format '%Y-%m-%d %H:%M:%S'")
                except ValueError:
                    try:
                        # Try another common format
                        df = pd.read_csv(os.path.join(input_dir, dataset_filename), index_col=0,
                                        parse_dates=True, date_format='%Y-%m-%d')
                        print(f"Successfully loaded {dataset_filename} with date format '%Y-%m-%d'")
                    except ValueError:
                        # If both formats fail, fall back to automatic detection
                        print(f"Could not parse dates with common formats, falling back to automatic detection")
                        df = pd.read_csv(os.path.join(input_dir, dataset_filename), index_col=0, parse_dates=True)

                print(f"Loaded {config.dataset_name} data with shape: {df.shape}")
                print(f"Sensor IDs (columns): {df.columns.tolist()[:5]}...") # Print first few sensor IDs

            except FileNotFoundError:
                raise FileNotFoundError(f"{config.dataset_name}.csv not found in {input_dir}, please place the dataset file there")

            data_normalized, timestamps, data_scaler, num_features = prepare_data(df, config)

            # --- Get Sensor IDs ---
            # Store sensor IDs for later use in plotting filenames/titles
            sensor_ids = df.columns.tolist() # Get the list of sensor IDs from the DataFrame

            # --- Adjacency Matrix Preparation ---
            if config.use_spatial_features and config.use_gnn_pre_transformer:
                if not TORCH_GEOMETRIC_AVAILABLE:
                    raise ImportError("PyTorch Geometric is required for GNN pre-transformer but not available")

                # Use a small slice of data just for getting adjacency matrix if needed
                min_data_len_adj = config.seq_length + config.pred_length
                if len(data_normalized) >= min_data_len_adj:
                    dataset_instance_adj = TrafficDataset(data_normalized[:min_data_len_adj], timestamps[:min_data_len_adj], config.seq_length, config.pred_length, config)
                    try:
                        adjacency_matrix = dataset_instance_adj.get_adjacency_matrix()
                        print(f"Successfully loaded adjacency matrix for {config.dataset_name}")
                    except Exception as e:
                        print(f"Error loading adjacency matrix: {e}")
                        adjacency_matrix = None
                else:
                    print("Warning: Not enough data to create dataset for adjacency matrix. Setting adjacency_matrix to None.")
                    adjacency_matrix = None
            else:
                adjacency_matrix = None

            # --- Model Parameters ---
            # Create a test dataset instance to get actual feature dimensions
            min_data_len = config.seq_length + config.pred_length
            if len(data_normalized) < min_data_len:
                 raise ValueError(f"Not enough data ({len(data_normalized)}) to create a sample with seq_length={config.seq_length} and pred_length={config.pred_length}.")

            # Make sure weather file path in config points to the correct location in input_dir if needed for dataset creation
            # This path might have been updated by setup_directories, double check it points to input_dir
            potential_weather_path_in_input = os.path.join(input_dir, os.path.basename(config.weather_data_file))
            if os.path.exists(potential_weather_path_in_input):
                config.weather_data_file = potential_weather_path_in_input
            else:
                print(f"Warning: Weather file {potential_weather_path_in_input} not found in input_dir for test dataset creation.")


            print(f"Creating test dataset to determine input dim with weather file: {config.weather_data_file}")
            try:
                test_dataset = TrafficDataset(data_normalized[:min_data_len], timestamps[:min_data_len], config.seq_length, config.pred_length, config)
                if len(test_dataset) == 0:
                    raise ValueError("Test dataset is unexpectedly empty after creation.")

                # Get feature dimensions from the dataset
                feature_dims, feature_names = get_feature_dimensions(test_dataset)
                print(f"Detected feature groups: {feature_names}")
                print(f"Feature dimensions: {feature_dims}")

                # Add the gnn_traffic dimension based on whether max pooling is enabled
                if hasattr(config, 'gnn_max_pooling') and config.gnn_max_pooling:
                    # With max pooling, gnn_traffic dimension is just hidden_dim
                    feature_dims['gnn_traffic'] = config.hidden_dim
                    print(f"Using max pooling for GNN: gnn_traffic dimension set to {config.hidden_dim}")
                else:
                    # Without max pooling, flatten all sensors (original behavior)
                    gnn_dim = num_features * config.hidden_dim
                    feature_dims['gnn_traffic'] = gnn_dim
                    print(f"Using flattened GNN features: gnn_traffic dimension set to {gnn_dim}")

                feature_names.append('gnn_traffic')

                # Ensure traffic dimension is available
                if 'traffic' not in feature_dims:
                    # Get from feature_groups directly as a fallback
                    feature_dims['traffic'] = test_dataset.feature_groups['traffic']['dim']
                    print(f"Added traffic dimension from feature_groups: {feature_dims['traffic']}")

                # Define input dimension based on traffic features
                input_dim = feature_dims['traffic']
                print(f"Using input_dim={input_dim} for model creation")

            except Exception as dataset_init_error:
                raise RuntimeError(f"Failed to create test TrafficDataset: {dataset_init_error}")

            # Use the actual_input_dim derived from the dataset sample directly
            model_params = {
                'input_dim': input_dim,
                'hidden_dim': config.hidden_dim,
                'num_layers': config.num_layers,
                'num_heads': config.num_heads,
                'num_features': num_features,
                'dropout': config.dropout,
                'ff_dim_multiplier': config.ff_dim_multiplier,
                'activation': config.activation,
                'decoder_type': config.decoder_type,
                'use_gnn_pre_transformer': config.use_gnn_pre_transformer,
                'spatial_feature_dim': config.spatial_feature_dim,
                'gnn_type': config.gnn_type,
                'pred_len': config.pred_length,
                'feature_dims': feature_dims,
                'gnn_max_pooling': config.gnn_max_pooling,
                'spatial_max_pooling': config.spatial_max_pooling,
                'seq_length': config.seq_length
            }

            # Dimensions log update
            print(f"Actual input dimension determined from DataLoader sample: {input_dim}")
            print(f"Using input_dim={model_params['input_dim']} for model creation") # Should now match actual_input_dim

            # Automatically find optimal batch size if enabled
            if config.find_optimal_batch_size and device == 'cuda' and torch.cuda.is_available():
                print("\n=== Finding Optimal Batch Size ===")
                # Create a dummy input/target on CPU first
                dummy_input_cpu = torch.randn(1, config.seq_length, model_params['input_dim'])
                dummy_target_cpu = torch.randn(1, config.pred_length, num_features)

                # Create a temporary model instance for testing on CPU first
                temp_model_params_bs, config_bs = ensure_compatible_dimensions(model_params.copy(), config) # Use copies
                try:
                    temp_model = TrafficTransformer(**temp_model_params_bs) # Create on CPU

                    # Find optimal batch size (function handles moving model/data to GPU)
                    optimal_batch_size = find_optimal_batch_size(
                        temp_model,
                        dummy_input_cpu, # Pass CPU tensor
                        dummy_target_cpu, # Pass CPU tensor
                        max_batch_size=2048,  # Upper limit to test
                        start_batch=32        # Starting test size
                    )
                    print(f"Optimal batch size for GPU memory: {optimal_batch_size}")
                     # Update configuration with optimal batch size
                    config.batch_size = optimal_batch_size

                except Exception as bs_error:
                     print(f"Error during optimal batch size search: {bs_error}. Using default batch size {config.batch_size}.")
                finally:
                     # Clean up
                    del temp_model, dummy_input_cpu, dummy_target_cpu, temp_model_params_bs, config_bs
                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()
                    gc.collect()


            # --- Optuna Hyperparameter Optimization ---
            if OPTUNA_AVAILABLE and config.optuna_trials is not None and config.optuna_trials > 0:
                # Define the objective function with proper config access
                def objective(trial):
                    # We need to declare config as nonlocal since it's from the outer scope
                    nonlocal config, model_params, data_normalized, timestamps, device, adjacency_matrix, num_features

                    # Define hyperparameter search space
                    num_heads = trial.suggest_categorical('num_heads', [4, 8, 16])

                    # Ensure hidden_dim is both divisible by num_heads and is even
                    hidden_dim_base = trial.suggest_int('hidden_dim', 64, 1024)
                    # Adjust to ensure divisibility by num_heads
                    hidden_dim = (hidden_dim_base // num_heads) * num_heads
                    if hidden_dim == 0: hidden_dim = num_heads # Ensure not zero
                    # Adjust to ensure it's even
                    if hidden_dim % 2 != 0:
                        hidden_dim += num_heads # Add num_heads to maintain divisibility and make it even

                    # Add GNN parameters to tune
                    if config.use_gnn_pre_transformer:
                        gnn_type = trial.suggest_categorical('gnn_type', ['gcn', 'gat'])
                        gnn_residual = trial.suggest_categorical('gnn_residual', [True, False])

                        # Common GNN parameters
                        gnn_layers = trial.suggest_int('gnn_layers', 1, 3)  # Number of GNN layers

                        # GAT-specific parameters
                        if gnn_type == 'gat':
                            gat_heads = trial.suggest_int('gat_heads', 1, 8)
                            gat_concat = trial.suggest_categorical('gat_concat', [True, False])
                        else:
                            # Default values if using GCN
                            gat_heads = config.gat_heads
                            gat_concat = config.gat_concat
                    else:
                        # Use original values if GNN pre-transformer is disabled
                        gnn_type = config.gnn_type
                        gnn_residual = config.gnn_residual
                        gnn_layers = 2  # Default value
                        gat_heads = config.gat_heads
                        gat_concat = config.gat_concat

                    # Create a copy of the config to avoid modifying the original
                    trial_config = TrainingConfig( # Re-create with necessary defaults
                        base_output_dir=config.base_output_dir, # Keep base dir
                        # Trial-specific params
                        hidden_dim=hidden_dim,
                        num_layers=trial.suggest_int('num_layers', 2, 6),
                        num_heads=num_heads,
                        dropout=trial.suggest_float('dropout', 0.0, 0.5),
                        learning_rate=trial.suggest_float('learning_rate', 1e-5, 1e-3, log=True),
                        decoder_type=trial.suggest_categorical('decoder_type', ['linear', 'mlp']),
                        # Inherited params
                        batch_size=config.batch_size, # Use found or default batch size
                        seq_length=config.seq_length,
                        pred_length=config.pred_length,
                        num_epochs=min(config.num_epochs, 10), # Limit epochs for Optuna trial
                        patience=config.patience // 2, # Shorter patience for trials
                        ff_dim_multiplier=config.ff_dim_multiplier,
                        activation=config.activation,
                        data_scaler_type=config.data_scaler_type,
                        optimizer_type=config.optimizer_type, # Can be tuned if needed
                        loss_function=config.loss_function, # Can be tuned if needed
                        use_time_features=config.use_time_features,
                        use_holiday_feature=config.use_holiday_feature,
                        holiday_country_code=config.holiday_country_code,
                        use_weather_feature=config.use_weather_feature,
                        weather_feature_type=config.weather_feature_type,
                        weather_data_file=config.weather_data_file, # Use updated path
                        gradient_clip=config.gradient_clip,
                        scheduler_type=config.scheduler_type, # Can be tuned if needed
                        scheduler_patience=config.scheduler_patience // 2,
                        scheduler_factor=config.scheduler_factor,
                        step_scheduler_step_size=config.step_scheduler_step_size,
                        step_scheduler_gamma=config.step_scheduler_gamma,
                        use_lagged_features=config.use_lagged_features,
                        num_lags=config.num_lags,
                        use_spatial_features=config.use_spatial_features,
                        spatial_feature_dim=hidden_dim_base,
                        use_gnn_pre_transformer=config.use_gnn_pre_transformer,
                        gnn_type=gnn_type,
                        gat_heads=gat_heads,
                        gat_concat=gat_concat,
                        gnn_residual=gnn_residual,
                        use_quantile_regression=config.use_quantile_regression,
                        quantiles=config.quantiles,
                        optuna_trials=0, # Disable nested Optuna
                        warmup_epochs=config.warmup_epochs // 2,
                        use_mixed_precision=config.use_mixed_precision,
                        accumulation_steps=config.accumulation_steps,
                        num_workers=min(config.num_workers, 2), # Reduce workers for trials
                        pin_memory=config.pin_memory,
                        find_optimal_batch_size=False, # Already done
                        monitor_gpu_usage=False, # Disable monitoring for trials
                    )

                    # Copy directory paths from main config
                    trial_config.input_dir = config.input_dir
                    trial_config.output_dir = config.output_dir
                    trial_config.model_dir = config.model_dir # Use main model dir (or could create subdirs)
                    trial_config.results_dir = config.results_dir # Use main results dir

                    print(f"\n--- Optuna Trial ---")
                    print(f"Params: hidden_dim={hidden_dim}, num_layers={trial_config.num_layers}, num_heads={num_heads}, lr={trial_config.learning_rate:.5f}, dropout={trial_config.dropout:.3f}, decoder={trial_config.decoder_type}")
                    if config.use_gnn_pre_transformer:
                        # Include all GNN parameters (for both GCN and GAT)
                        gnn_params = f"GNN params: gnn_type={gnn_type}, gnn_residual={gnn_residual}"

                        # Include GNN-specific parameters
                        if 'gnn_dropout' in locals():
                            gnn_params += f", gnn_dropout={gnn_dropout}"
                        if 'gnn_layers' in locals():
                            gnn_params += f", gnn_layers={gnn_layers}"

                        # Include GAT-specific parameters (even if not currently using GAT)
                        gnn_params += f", gat_heads={gat_heads}, gat_concat={gat_concat}"

                        print(gnn_params)

                    # Copy model parameters but with trial values
                    trial_model_params = model_params.copy()
                    trial_model_params['hidden_dim'] = hidden_dim
                    trial_model_params['num_layers'] = trial_config.num_layers
                    trial_model_params['num_heads'] = num_heads
                    trial_model_params['dropout'] = trial_config.dropout
                    trial_model_params['decoder_type'] = trial_config.decoder_type

                    # GNN PARAMETERS
                    if config.use_gnn_pre_transformer:
                        trial_model_params['gnn_type'] = gnn_type
                        trial_model_params['gnn_residual'] = gnn_residual
                        # Pass these to the GCNEncoder via TrafficTransformer
                        if gnn_type == 'gat':
                            trial_model_params['gat_heads'] = gat_heads
                            trial_model_params['gat_concat'] = gat_concat

                    # Ensure dimensions are compatible FOR THE TRIAL
                    trial_model_params, trial_config = ensure_compatible_dimensions(trial_model_params, trial_config)

                    # Check constraints again after potential adjustments
                    assert trial_model_params['hidden_dim'] % trial_model_params['num_heads'] == 0, \
                        f"Trial Error: Hidden dimension {trial_model_params['hidden_dim']} must be divisible by number of heads {trial_model_params['num_heads']}"
                    assert trial_model_params['hidden_dim'] % 2 == 0, \
                        f"Trial Error: Hidden dimension {trial_model_params['hidden_dim']} must be even."

                    try:
                        # Create model with trial parameters
                        model_optuna = TrafficTransformer(**trial_model_params).to(device)

                        # Initialize optimizer and scheduler
                        optimizer_optuna = optim.AdamW(model_optuna.parameters(), lr=trial_config.learning_rate)
                        scheduler_optuna = optim.lr_scheduler.ReduceLROnPlateau(
                            optimizer_optuna, patience=trial_config.patience // 2, factor=0.5
                        ) if trial_config.scheduler_type == 'plateau' else None # Example scheduler

                        # Choose criterion based on loss function
                        if trial_config.loss_function == 'mse':
                            criterion_optuna = nn.MSELoss()
                        elif trial_config.loss_function == 'mae':
                            criterion_optuna = nn.L1Loss()
                        # Add other loss functions if needed for Optuna trials
                        else:
                            criterion_optuna = nn.MSELoss()

                        # Time Series Cross-Validation (Single Split for Optuna speed)
                        tscv_optuna = TimeSeriesSplit(n_splits=2) # Use 2 splits to get one validation set
                        all_indices = np.arange(len(data_normalized))
                        train_idx_optuna, val_idx_optuna = list(tscv_optuna.split(all_indices))[-1] # Take the last split

                        train_data_optuna = data_normalized[train_idx_optuna]
                        val_data_optuna = data_normalized[val_idx_optuna]
                        train_times_optuna = timestamps[train_idx_optuna]
                        val_times_optuna = timestamps[val_idx_optuna]

                        # Ensure dataset creation uses the trial_config
                        train_dataset_optuna = TrafficDataset(
                            train_data_optuna, train_times_optuna, trial_config.seq_length, trial_config.pred_length, trial_config
                        )
                        val_dataset_optuna = TrafficDataset(
                            val_data_optuna, val_times_optuna, trial_config.seq_length, trial_config.pred_length, trial_config
                        )

                        # Use smaller batch size for trials to avoid memory issues if needed
                        trial_batch_size = min(trial_config.batch_size, 64)  # Limiting batch size for trials

                        train_loader_optuna = DataLoader(
                            train_dataset_optuna, batch_size=trial_batch_size, shuffle=True,
                            num_workers=trial_config.num_workers, # Use reduced workers
                            pin_memory=trial_config.pin_memory
                        )
                        val_loader_optuna = DataLoader(
                            val_dataset_optuna, batch_size=trial_batch_size, shuffle=False,
                            num_workers=trial_config.num_workers, # Use reduced workers
                            pin_memory=trial_config.pin_memory
                        )

                        # Train the model (ensure train_model uses trial_config)
                        print(f"Starting Optuna trial training (Max epochs: {trial_config.num_epochs})...")
                        trained_model_optuna, _, _ = train_model(
                            model=model_optuna,
                            train_loader=train_loader_optuna,
                            val_loader=val_loader_optuna,
                            optimizer=optimizer_optuna,
                            scheduler=scheduler_optuna,
                            criterion=criterion_optuna,
                            config=trial_config,
                            device=device,
                            adjacency_matrix=adjacency_matrix, # Pass TRIAL config
                            data_scaler=data_scaler
                        )

                        # Evaluate the model (ensure evaluate_model uses trial_config)
                        avg_val_loss_optuna, _ = evaluate_model(
                            model=trained_model_optuna,
                            dataloader=val_loader_optuna,
                            criterion=criterion_optuna,
                            device=device,
                            config=trial_config,
                            adjacency_matrix=adjacency_matrix, # Pass TRIAL config
                            data_scaler=data_scaler,
                        )

                        print(f"--- Optuna Trial Completed --- Loss: {avg_val_loss_optuna:.6f}\n")
                        # Clean up trial resources
                        del model_optuna, optimizer_optuna, scheduler_optuna, criterion_optuna
                        del train_dataset_optuna, val_dataset_optuna, train_loader_optuna, val_loader_optuna
                        gc.collect()
                        if torch.cuda.is_available(): torch.cuda.empty_cache()

                        return avg_val_loss_optuna

                    except Exception as e:
                        print(f"--- Optuna Trial FAILED --- Error: {str(e)}\n")
                         # Clean up potential partial resources
                        gc.collect()
                        if torch.cuda.is_available(): torch.cuda.empty_cache()
                        # Return a high value to indicate failure
                        # Consider using optuna.TrialPruned() if appropriate
                        return float('inf')


                # Create and run Optuna study
                study = optuna.create_study(direction='minimize')
                print(f"\n=== Starting Optuna Hyperparameter Search ({config.optuna_trials} trials) ===")

                try:
                    # Pass necessary objects (like data) if objective function needs them indirectly
                    study.optimize(objective, n_trials=config.optuna_trials)

                    # Check if we have valid completed trials
                    completed_trials = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]

                    if completed_trials:
                         # Get best parameters from completed trials
                        best_params = study.best_params
                        print(f"\nOptuna finished. Best hyperparameters found: {best_params}")
                        print(f"Best validation loss achieved: {study.best_value:.6f}")

                        # Update MAIN config with best parameters found by Optuna
                        config.num_heads = best_params['num_heads']
                        config.hidden_dim = best_params['hidden_dim'] # Use the adjusted value from objective
                        config.num_layers = best_params['num_layers']
                        config.dropout = best_params['dropout']
                        config.learning_rate = best_params['learning_rate']
                        config.decoder_type = best_params['decoder_type']

                        # Update MAIN model_params dictionary as well
                        model_params['hidden_dim'] = config.hidden_dim
                        model_params['num_layers'] = config.num_layers
                        model_params['num_heads'] = config.num_heads
                        model_params['dropout'] = config.dropout
                        model_params['decoder_type'] = config.decoder_type

                        # CRITICAL: Re-ensure compatibility for the main config after Optuna update
                        print("Re-validating dimensions after Optuna...")
                        model_params, config = ensure_compatible_dimensions(model_params, config)
                        print(f"Final model params after Optuna & validation: Heads={model_params['num_heads']}, Hidden={model_params['hidden_dim']}")

                    else:
                        print("\nNo successful Optuna trials completed. Using original parameters.")

                except Exception as e:
                    print(f"Optuna optimization encountered an error: {str(e)}")
                    import traceback
                    traceback.print_exc()
                    print("Continuing with original parameters...")
                    # Ensure dimensions are still compatible even if Optuna failed midway
                    model_params, config = ensure_compatible_dimensions(model_params, config)

                print("============================================================\n")

            else:
                if not OPTUNA_AVAILABLE:
                    warnings.warn("Optuna not available, skipping hyperparameter tuning")
                else:
                    print("Optuna hyperparameter tuning disabled (optuna_trials is None or 0)")

            # --- Fixed Train/Val/Test Split (70/10/20) ---
            print("\n=== Using Fixed Train/Val/Test Split (70/10/20) ===")
            print(f"Config: {config.data_scaler_type}, {config.optimizer_type}, {config.loss_function}, {config.pred_length}-step ahead")
            print(f"Model Params: Heads={model_params['num_heads']}, Hidden={model_params['hidden_dim']}, Layers={model_params['num_layers']}, Dropout={model_params['dropout']:.3f}")

            # Calculate split indices (maintaining temporal order)
            n_samples = len(data_normalized)
            train_end = int(n_samples * 0.7)
            val_end = int(n_samples * 0.8)  # 70% + 10% = 80%

            # Create the splits
            train_data = data_normalized[:train_end]
            val_data = data_normalized[train_end:val_end]
            test_data = data_normalized[val_end:]

            train_times = timestamps[:train_end]
            val_times = timestamps[train_end:val_end]
            test_times = timestamps[val_end:]

            print(f"Data split - Train: {len(train_data)}, Validation: {len(val_data)}, Test: {len(test_data)} samples")

            # Create datasets
            train_dataset = TrafficDataset(train_data, train_times, config.seq_length, config.pred_length, config)
            val_dataset = TrafficDataset(val_data, val_times, config.seq_length, config.pred_length, config)
            test_dataset = TrafficDataset(test_data, test_times, config.seq_length, config.pred_length, config)

            # Create dataloaders
            train_loader = DataLoader(
                train_dataset,
                batch_size=config.batch_size,
                shuffle=True,
                num_workers=config.num_workers,
                pin_memory=config.pin_memory,
                prefetch_factor=2 if config.num_workers > 0 else None,
                drop_last=True  # Drop last incomplete batch for stability
            )

            val_loader = DataLoader(
                val_dataset,
                batch_size=config.batch_size,
                shuffle=False,
                num_workers=config.num_workers,
                pin_memory=config.pin_memory
            )

            test_loader = DataLoader(
                test_dataset,
                batch_size=config.batch_size,
                shuffle=False,
                num_workers=config.num_workers,
                pin_memory=config.pin_memory
            )

            # Double-check dimension compatibility before creating model
            model_params, config = ensure_compatible_dimensions(model_params, config)

            # Initialize model
            model = TrafficTransformer(**model_params).to(device)

            # Initialize optimizer
            if config.optimizer_type == 'adam':
                optimizer = optim.Adam(model.parameters(), lr=config.learning_rate)
            elif config.optimizer_type == 'adamw':
                optimizer = optim.AdamW(model.parameters(), lr=config.learning_rate)
            else:
                warnings.warn(f"Invalid optimizer type: {config.optimizer_type}, using AdamW")
                optimizer = optim.AdamW(model.parameters(), lr=config.learning_rate)

            # Initialize loss function
            if config.loss_function == 'mse':
                criterion = nn.MSELoss()
            elif config.loss_function == 'mae':
                criterion = nn.L1Loss()
            elif config.loss_function == 'huber':
                criterion = nn.SmoothL1Loss()
            elif config.loss_function == 'quantile':
                # Ensure quantiles are available in config
                if not hasattr(config, 'quantiles') or not config.quantiles:
                    config.quantiles = [0.1, 0.5, 0.9]  # Default quantiles
                    print(f"Warning: Quantiles not found in config, using default: {config.quantiles}")
                criterion = lambda output, target: quantile_loss(output, target, config.quantiles)
            elif config.loss_function == 'hybrid':
                criterion = hybrid_loss
            else:
                warnings.warn(f"Invalid loss function: {config.loss_function}, using MSELoss")
                criterion = nn.MSELoss()

            # Initialize scheduler
            scheduler = None  # Default to None
            if config.scheduler_type == 'plateau':
                scheduler = optim.lr_scheduler.ReduceLROnPlateau(
                    optimizer, mode='min', patience=config.scheduler_patience,
                    factor=config.scheduler_factor, verbose=True
                )
            elif config.scheduler_type == 'cosine':
                scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config.num_epochs)
            elif config.scheduler_type == 'step':
                scheduler = optim.lr_scheduler.StepLR(
                    optimizer, step_size=config.step_scheduler_step_size, gamma=config.step_scheduler_gamma
                )
            elif config.scheduler_type == 'cosine_warmup':
                # Ensure warmup_epochs is valid
                if not hasattr(config, 'warmup_epochs') or config.warmup_epochs <= 0:
                    config.warmup_epochs = max(1, config.num_epochs // 10)  # Default warmup
                    print(f"Warning: Invalid warmup_epochs, setting to {config.warmup_epochs}")
                scheduler = CosineWarmupLR(
                    optimizer,
                    warmup_epochs=config.warmup_epochs,
                    total_epochs=config.num_epochs,
                    base_lr=config.learning_rate
                )
            elif config.scheduler_type is not None:  # Catch invalid but not None types
                warnings.warn(f"Invalid scheduler type: {config.scheduler_type}, no scheduler will be used.")

            # Run garbage collection and clear CUDA cache before training
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

            # Train model
            print("Starting training with fixed split...")
            trained_model, train_losses, val_losses = train_model(
                model=model,
                train_loader=train_loader,
                val_loader=val_loader,  # Use validation set during training
                optimizer=optimizer,
                scheduler=scheduler,
                criterion=criterion,
                config=config,
                device=device,
                adjacency_matrix=adjacency_matrix,
                data_scaler=data_scaler
            )

            # Evaluate on test set
            print("\nEvaluating on test set...")
            test_loss, metrics = evaluate_model(
                model=trained_model,
                dataloader=test_loader,
                criterion=criterion,
                data_scaler=data_scaler,
                device=device,
                config=config,
                adjacency_matrix=adjacency_matrix
            )

            mae, rmse, r2, mape = metrics
            print(f'Test Metrics: MAE={mae:.4f}, RMSE={rmse:.4f}, R²={r2:.4f}, MAPE={mape:.2f}%')

            # Save experiment results
            save_experiment_results(
                config=config,
                model_params=model_params,
                metrics=metrics,
                results_dir=results_dir,
                fold=None  # No fold number for fixed split
            )

            # Generate predictions and plots
            print(f"Generating predictions and plots...")
            predictions, actuals = predict(
                model=trained_model,
                dataloader=test_loader,
                scaler=data_scaler,
                device=device,
                adjacency_matrix=adjacency_matrix,
                config=config
            )

            # Save predictions
            save_predictions_and_actuals(
                predictions=predictions.reshape(-1, num_features),
                actuals=actuals.reshape(-1, num_features),
                results_dir=results_dir,
                filename="fixed_split_predictions"
            )

            # Plot predictions for a subset of sensors
            for sensor_idx in range(min(config.num_sensors_to_plot, actuals.shape[1])):
                if sensor_idx < len(sensor_ids):
                    actual_sensor_id = sensor_ids[sensor_idx]
                    plot_predictions_vs_actual(
                        actuals=actuals,
                        predictions=predictions,
                        sensor_index=sensor_idx,
                        sensor_id=actual_sensor_id,
                        fold=None,  # No fold number for fixed split
                        results_dir=results_dir,
                        pred_len=config.pred_length,
                        config=config,
                    )

            # Plot attention weights
            if config.attention_visualization:
                plot_attention_weights(
                    trained_model,
                    config.seq_length,
                    results_dir,
                    config,
                    show_feature_groups=True
                )

            # Plot training history
            plot_training_history(
                train_losses, val_losses,
                results_dir, config
            )

            # Generate comprehensive report
            print("\nGenerating comprehensive traffic prediction report...")
            report_path = generate_traffic_report(
                model=trained_model,
                test_loader=test_loader,
                scaler=data_scaler,
                config=config,
                device=device,
                train_losses=train_losses,
                val_losses=val_losses,
                fold_metrics=[metrics],  # Wrap single metrics in list for compatibility
                baseline_metrics={'naive_forecast': []},  # Empty baseline for simplicity
                results_dir=results_dir,
                timestamp=get_maputo_timestamp()
            )

            print(f"\nDetailed analysis report generated at: {report_path}")

            # --- Calculate and print average metrics ---
            if fold_metrics:
                avg_mae = np.mean([m[0] for m in fold_metrics])
                avg_rmse = np.mean([m[1] for m in fold_metrics])
                avg_r2 = np.mean([m[2] for m in fold_metrics])
                # Use nanmean for MAPE to handle potential NaNs if division by zero occurred
                avg_mape = np.nanmean([m[3] for m in fold_metrics])

                print(f"\n=== Average Metrics Across {len(fold_metrics)} Folds ===")
                print(f"Transformer MAE: {avg_mae:.4f}, RMSE: {avg_rmse:.4f}, R²: {avg_r2:.4f}, MAPE: {avg_mape:.2f}%")

                # --- Print baseline metrics ---
                print("\n=== Average Baseline Metrics Across Folds ===")
                baseline_models_reported = False
                for model_name, metrics_list in baseline_metrics.items():
                    if metrics_list:  # Only calculate averages if there are metrics for this baseline
                        baseline_models_reported = True
                        avg_baseline_mae = np.mean([m[0] for m in metrics_list])
                        avg_baseline_rmse = np.mean([m[1] for m in metrics_list])
                        avg_baseline_r2 = np.mean([m[2] for m in metrics_list])
                        avg_baseline_mape = np.nanmean([m[3] for m in metrics_list]) # Use nanmean
                        print(f"Model: {model_name.upper()}")
                        print(f"  MAE: {avg_baseline_mae:.4f}, RMSE: {avg_baseline_rmse:.4f}, R²: {avg_baseline_r2:.4f}, MAPE: {avg_baseline_mape:.2f}%")
                if not baseline_models_reported:
                     print("No baseline model metrics were recorded.")

                # --- Generate final report ---
                # Ensure we have necessary components from the last successful fold
                if last_trained_model and last_test_loader:
                    try:
                        print("\nGenerating comprehensive traffic prediction report...")
                        # Use data from the last successful fold for the report context
                        report_path = generate_traffic_report(
                            model=last_trained_model,
                            test_loader=last_test_loader,
                            scaler=data_scaler,
                            config=config, # Pass the final (potentially Optuna-tuned) config
                            device=device,
                            train_losses=last_fold_train_losses, # Use losses from last fold
                            val_losses=last_fold_val_losses,     # Use losses from last fold
                            fold_metrics=fold_metrics,      # Use metrics from ALL folds
                            baseline_metrics=baseline_metrics, # Use ALL baseline metrics
                            results_dir=results_dir,
                            timestamp=get_maputo_timestamp()
                        )

                        if report_path and os.path.exists(report_path):
                            print(f"\nDetailed analysis report generated at: {report_path}")
                        else:
                            print("\nWarning: Failed to generate or find the detailed analysis report.")
                    except Exception as report_error:
                        print(f"\nError generating final report: {report_error}")
                        import traceback
                        traceback.print_exc()
                else:
                     print("\nSkipping final report generation: No successful fold completed or missing model/loader.")

            else:
                print("\nNo fold metrics available - training may have failed or all folds were skipped.")

            # Create summary plot with all model comparisons
            try:
                 # Check if there are transformer metrics and any baseline metrics to compare
                if fold_metrics and any(metrics for metrics in baseline_metrics.values()):
                    print("\nGenerating summary comparison plot...")
                    create_summary_comparison_plot(
                        fold_metrics,
                        baseline_metrics,
                        results_dir,
                        config # Pass final config
                    )
                    print("Summary comparison plot generated.")
                elif fold_metrics:
                    print("\nGenerating summary plot (Transformer only)...")
                    # If only transformer metrics are available, you might call a modified
                    # version of create_summary_comparison_plot or a dedicated function.
                    # For now, just note it wasn't created due to missing baselines.
                    create_summary_comparison_plot(fold_metrics, {}, results_dir, config) # Pass empty baseline dict
                    print("Summary plot generated (Transformer only).")
                else:
                     print("\nSkipping summary comparison plot: No metrics available.")
            except Exception as summary_plot_error:
                print(f"Error creating summary comparison plot: {summary_plot_error}")
        else:
            # Transfer learning is not enabled but run_only_transfer was requested
            print("\n=== Error: Transfer learning is not enabled but run_only_transfer_learning is set to True ===")
            print("Please enable transfer_learning or disable run_only_transfer_learning")

    except Exception as e:
        print(f"\n=== Main Execution Error: {str(e)} ===")
        import traceback
        traceback.print_exc()

    finally:
        # Stop GPU monitoring
        stop_gpu_memory_monitor(monitor_stop_flag)
        # Final cleanup
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

        print("\n=== Execution Complete ===")
        if hasattr(config, 'enable_transfer_learning') and config.enable_transfer_learning:
            print(f"Transfer learning completed for {getattr(config, 'target_dataset_name', 'target')} dataset")
        else:
            print("Standard training workflow completed")

if __name__ == "__main__":
    # Configure PyTorch to optimize for performance
    torch.backends.cudnn.benchmark = True  # Enable cudnn auto-tuner

    try:
        # Run main function with error handling
        main()
    except Exception as e:
        print(f"\n--- Error in main execution ---")
        print(f"Error Type: {type(e).__name__}")
        print(f"Error Details: {e}")
        import traceback
        traceback.print_exc()
        print("-----------------")

    # Final cleanup
    finally:
        gc.collect() # Ensure garbage collection runs
        if torch.cuda.is_available():
            try:
                torch.cuda.empty_cache()
                print("\n=== Final GPU Memory Usage ===")
                gpu_info = get_gpu_memory_info()
                if isinstance(gpu_info, dict) and 'error' not in gpu_info:
                    for gpu_id, info in gpu_info.items():
                        if isinstance(info, dict) and 'error' not in info:
                            print(f"{gpu_id.upper()}: Used {info['allocated_memory_GB']:.2f}/{info['total_memory_GB']:.2f} GB "\
                                  f"({info['utilization_pct']:.1f}%)")
                elif isinstance(gpu_info, dict) and 'error' in gpu_info:
                     print(f"Could not get GPU info: {gpu_info['error']}")
                else:
                     print("Could not retrieve final GPU info.")
                print("===============================")
            except Exception as gpu_info_err:
                 print(f"Error during final GPU memory check: {gpu_info_err}")
        print("\nExecution finished.")

<ipython-input-21-31b15b5b5c6c>:44: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
<ipython-input-21-31b15b5b5c6c>:94: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
<ipython-input-21-31b15b5b5c6c>:94: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


KeyboardInterrupt: 

In [ ]:
# Once your main code is done
#from google.colab import runtime
# Disconnect and delete the runtime
#runtime.unassign()